In [ ]:
# ============================================================
# CÉLULA 1 — SETUP E CONFIGURAÇÃO
# F1000 Corpus Builder 2.0
# ============================================================

from google.colab import drive

from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json
import os
import platform
import re
import sqlite3
import sys
import time
import xml.etree.ElementTree as ET

import requests


# ============================================================
# 1. IDENTIDADE DO BUILDER
# ============================================================

BUILDER_NAME = "F1000 Corpus Builder"
BUILDER_VERSION = "2.0.0"

TARGET_BASE_ARTICLES = 500


# ============================================================
# 2. ENDPOINTS F1000RESEARCH
# ============================================================

SEARCH_ENDPOINT = "https://f1000research.com/extapi/search"
ARTICLE_XML_ENDPOINT = "https://f1000research.com/extapi/article/xml"


# ============================================================
# 3. POLÍTICA HTTP
# ============================================================

# Mantemos aquisição deliberadamente conservadora.
HTTP_DELAY_SECONDS = 1.25
HTTP_TIMEOUT_SECONDS = 60
HTTP_MAX_RETRIES = 4

SEARCH_ROWS = 100

USER_AGENT = (
    f"{BUILDER_NAME}/{BUILDER_VERSION} "
    "(academic corpus construction)"
)


# ============================================================
# 4. GOOGLE DRIVE
# ============================================================

drive.mount("/content/drive")


# ============================================================
# 5. DIRETÓRIO RAIZ
# ============================================================

ROOT = Path("/content/drive/MyDrive/f1000_corpus_500")

RAW_DIR = ROOT / "raw"
RAW_CONTAINERS_DIR = RAW_DIR / "containers"
RAW_REVIEWED_VERSIONS_DIR = RAW_DIR / "reviewed_versions"

CORPUS_DIR = ROOT / "corpus"
CURATION_DIR = ROOT / "curation"

STATE_DIR = ROOT / "state"
LOGS_DIR = ROOT / "logs"

DB_PATH = STATE_DIR / "builder.sqlite"
LOG_PATH = LOGS_DIR / "builder.log"
MANIFEST_PATH = ROOT / "manifest.json"


# ============================================================
# 6. CRIAÇÃO DA ESTRUTURA
# ============================================================

DIRECTORIES = [
    ROOT,
    RAW_DIR,
    RAW_CONTAINERS_DIR,
    RAW_REVIEWED_VERSIONS_DIR,
    CORPUS_DIR,
    CURATION_DIR,
    STATE_DIR,
    LOGS_DIR,
]

for directory in DIRECTORIES:
    directory.mkdir(parents=True, exist_ok=True)


# ============================================================
# 7. HELPERS GERAIS
# ============================================================

def utc_now():
    return datetime.now(timezone.utc).isoformat()


def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def sha256_text(text: str) -> str:
    return hashlib.sha256(
        text.encode("utf-8")
    ).hexdigest()


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


def normalize_whitespace(text):
    if text is None:
        return None

    return re.sub(r"\s+", " ", str(text)).strip()


def normalize_doi(doi):
    """
    Normaliza DOI sem modificar sua identidade semântica.
    """

    if doi is None:
        return None

    doi = str(doi).strip()

    doi = re.sub(
        r"^https?://(?:dx\.)?doi\.org/",
        "",
        doi,
        flags=re.IGNORECASE,
    )

    doi = re.sub(
        r"^doi:\s*",
        "",
        doi,
        flags=re.IGNORECASE,
    )

    return doi.strip().lower()


def extract_version_from_doi(doi):
    """
    Suporta os dois padrões empiricamente observados:

    ...5742.1
    ...5742.v1
    """

    doi = normalize_doi(doi)

    if not doi:
        return None

    match = re.search(
        r"(?:\.v|\.)(\d+)$",
        doi,
        flags=re.IGNORECASE,
    )

    if not match:
        return None

    return int(match.group(1))


def derive_base_doi(doi):
    """
    Remove apenas o sufixo final de versão.

    Exemplos:
      10.12688/f1000research.5742.1
          -> 10.12688/f1000research.5742

      10.12688/f1000research.5742.v1
          -> 10.12688/f1000research.5742
    """

    doi = normalize_doi(doi)

    if not doi:
        return None

    return re.sub(
        r"(?:\.v|\.)(\d+)$",
        "",
        doi,
        flags=re.IGNORECASE,
    )


def safe_filename(value):
    """
    Converte identificadores em nomes seguros para arquivos.
    """

    value = str(value).strip().lower()

    value = re.sub(
        r"[^a-z0-9._-]+",
        "_",
        value,
    )

    return value.strip("_")


def make_knowledge_unit_id(base_doi):
    """
    ID determinístico da KnowledgeUnit.
    """

    normalized = normalize_doi(base_doi)

    digest = hashlib.sha256(
        normalized.encode("utf-8")
    ).hexdigest()[:20]

    return f"ku_{digest}"


def make_pair_id(review_doi):
    """
    ID determinístico da relação CanonicalPair.
    """

    normalized = normalize_doi(review_doi)

    digest = hashlib.sha256(
        normalized.encode("utf-8")
    ).hexdigest()[:20]

    return f"pair_{digest}"


# ============================================================
# 8. LOGGING SIMPLES
# ============================================================

def log(message):
    timestamp = utc_now()
    line = f"[{timestamp}] {message}"

    print(line)

    with LOG_PATH.open(
        "a",
        encoding="utf-8",
    ) as f:
        f.write(line + "\n")


# ============================================================
# 9. TESTES LOCAIS DE NORMALIZAÇÃO
# ============================================================

_test_cases = [
    (
        "10.12688/f1000research.5742.1",
        "10.12688/f1000research.5742",
        1,
    ),
    (
        "10.12688/f1000research.5742.v1",
        "10.12688/f1000research.5742",
        1,
    ),
    (
        "https://doi.org/10.12688/f1000research.5742.2",
        "10.12688/f1000research.5742",
        2,
    ),
]

for doi, expected_base, expected_version in _test_cases:

    actual_base = derive_base_doi(doi)
    actual_version = extract_version_from_doi(doi)

    assert actual_base == expected_base, (
        f"Falha derive_base_doi: "
        f"{doi} -> {actual_base}"
    )

    assert actual_version == expected_version, (
        f"Falha extract_version_from_doi: "
        f"{doi} -> {actual_version}"
    )


# ============================================================
# 10. VERIFICAÇÃO DO AMBIENTE
# ============================================================

assert ROOT.exists()
assert RAW_CONTAINERS_DIR.exists()
assert RAW_REVIEWED_VERSIONS_DIR.exists()
assert CORPUS_DIR.exists()
assert CURATION_DIR.exists()
assert STATE_DIR.exists()
assert LOGS_DIR.exists()

assert TARGET_BASE_ARTICLES == 500
assert SEARCH_ROWS <= 100

assert SEARCH_ENDPOINT.startswith("https://")
assert ARTICLE_XML_ENDPOINT.startswith("https://")


# ============================================================
# 11. RESUMO
# ============================================================

log(
    f"{BUILDER_NAME} {BUILDER_VERSION} inicializado."
)

print()
print("=" * 72)
print("F1000 CORPUS BUILDER 2.0 — SETUP")
print("=" * 72)

print(f"Python................: {sys.version.split()[0]}")
print(f"Platform..............: {platform.platform()}")
print(f"Requests..............: {requests.__version__}")
print(f"SQLite................: {sqlite3.sqlite_version}")

print()
print(f"Target base articles..: {TARGET_BASE_ARTICLES}")
print(f"Search rows/request...: {SEARCH_ROWS}")
print(f"HTTP delay............: {HTTP_DELAY_SECONDS} s")
print(f"HTTP retries..........: {HTTP_MAX_RETRIES}")
print(f"HTTP timeout..........: {HTTP_TIMEOUT_SECONDS} s")

print()
print(f"ROOT..................: {ROOT}")
print(f"Containers............: {RAW_CONTAINERS_DIR}")
print(f"Reviewed versions.....: {RAW_REVIEWED_VERSIONS_DIR}")
print(f"Corpus Markdown.......: {CORPUS_DIR}")
print(f"Curation Markdown.....: {CURATION_DIR}")
print(f"SQLite................: {DB_PATH}")
print(f"Log...................: {LOG_PATH}")
print(f"Manifest..............: {MANIFEST_PATH}")

print()
print("Endpoints:")
print(f"  Search...............: {SEARCH_ENDPOINT}")
print(f"  Article XML..........: {ARTICLE_XML_ENDPOINT}")

print()
print("Local normalization tests: OK")
print("Directory structure.......: OK")
print("HTTP requests performed...: 0")

print("=" * 72)
print("CÉLULA 1 CONCLUÍDA")
print("=" * 72)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[2026-09-17T22:38:26.197049+00:00] F1000 Corpus Builder 2.0.0 inicializado.

F1000 CORPUS BUILDER 2.0 — SETUP
Python................: 3.13.15
Platform..............: Linux-6.6.122+-x86_64-with-glibc2.39
Requests..............: 2.32.4
SQLite................: 3.45.1

Target base articles..: 500
Search rows/request...: 100
HTTP delay............: 1.25 s
HTTP retries..........: 4
HTTP timeout..........: 60 s

ROOT..................: /content/drive/MyDrive/f1000_corpus_500
Containers............: /content/drive/MyDrive/f1000_corpus_500/raw/containers
Reviewed versions.....: /content/drive/MyDrive/f1000_corpus_500/raw/reviewed_versions
Corpus Markdown.......: /content/drive/MyDrive/f1000_corpus_500/corpus
Curation Markdown.....: /content/drive/MyDrive/f1000_corpus_500/curation
SQLite................: /content/drive/MyDrive/f1000_corpus_500/state/builder.sqlite
Log.

In [ ]:
# ============================================================
# CÉLULA 2 — SQLITE / SCHEMA / MIGRAÇÃO RESTART-SAFE
# F1000 Corpus Builder 2.0
# ============================================================

# Responsabilidades:
#
#   1. Abrir o SQLite com política segura para Google Drive.
#   2. Criar o schema Builder 2.0 quando o banco for novo.
#   3. Detectar tabelas antigas/incompatíveis.
#   4. Migrar peer_review_states quando os campos de contagem
#      ainda estiverem definidos como NOT NULL.
#   5. Preservar BaseArticles, exclusões e demais dados.
#   6. Validar:
#        - colunas obrigatórias;
#        - PKs essenciais;
#        - nullability semântica de PeerReviewState;
#        - unicidade operacional;
#        - foreign keys;
#        - integridade SQLite.
#
# IMPORTANTE:
#
#   approved_count
#   reservations_count
#   not_approved_count
#
# devem aceitar NULL.
#
# Semântica:
#
#   NULL = informação não observada / não determinável
#   0    = zero explicitamente determinado
#
# ============================================================


# ============================================================
# 1. CONFIGURAÇÃO DO SCHEMA
# ============================================================

DB_SCHEMA_VERSION = "2.0"


# ============================================================
# 2. CONEXÃO SQLITE PADRÃO DO BUILDER
# ============================================================

def open_builder_db():
    """
    Abre o SQLite com configuração conservadora para arquivo
    persistido no Google Drive.

    Um único writer é assumido pelo pipeline.
    """

    connection = sqlite3.connect(
        DB_PATH,
        timeout=60.0,
    )

    connection.execute(
        "PRAGMA foreign_keys = ON;"
    )

    connection.execute(
        "PRAGMA busy_timeout = 60000;"
    )

    connection.execute(
        "PRAGMA journal_mode = DELETE;"
    )

    connection.execute(
        "PRAGMA synchronous = NORMAL;"
    )

    return connection


# ============================================================
# 3. HELPERS DE INTROSPECÇÃO
# ============================================================

def table_exists(
    conn,
    table_name,
):
    row = conn.execute(
        """
        SELECT 1
        FROM sqlite_master
        WHERE
            type = 'table'
            AND name = ?
        """,
        (
            table_name,
        ),
    ).fetchone()

    return row is not None


def get_table_info(
    conn,
    table_name,
):
    """
    Retorna PRAGMA table_info como lista de dicionários.
    """

    rows = conn.execute(
        f'PRAGMA table_info("{table_name}");'
    ).fetchall()

    return [
        {
            "cid": row[0],
            "name": row[1],
            "type": row[2],
            "notnull": bool(row[3]),
            "default": row[4],
            "pk": int(row[5]),
        }
        for row in rows
    ]


def get_column_map(
    conn,
    table_name,
):
    return {
        item["name"]: item
        for item in get_table_info(
            conn,
            table_name,
        )
    }


def table_count(
    conn,
    table_name,
):
    return conn.execute(
        f'SELECT COUNT(*) FROM "{table_name}";'
    ).fetchone()[0]


# ============================================================
# 4. SQL OFICIAL — run_info
# ============================================================

SQL_CREATE_RUN_INFO = """
CREATE TABLE IF NOT EXISTS run_info (

    key TEXT PRIMARY KEY,

    value TEXT NOT NULL,

    updated_at TEXT NOT NULL
);
"""


# ============================================================
# 5. SQL OFICIAL — base_articles
# ============================================================

SQL_CREATE_BASE_ARTICLES = """
CREATE TABLE IF NOT EXISTS base_articles (

    base_doi TEXT PRIMARY KEY,

    container_doi TEXT NOT NULL UNIQUE,

    container_version INTEGER NOT NULL,

    discovery_order INTEGER NOT NULL UNIQUE,

    discovered_at TEXT NOT NULL,

    discovery_status TEXT NOT NULL
        DEFAULT 'DISCOVERED'
        CHECK (
            discovery_status IN (
                'DISCOVERED',
                'ERROR'
            )
        ),

    container_status TEXT NOT NULL
        DEFAULT 'PENDING'
        CHECK (
            container_status IN (
                'PENDING',
                'READY',
                'ERROR'
            )
        ),

    container_xml_path TEXT,

    container_xml_sha256 TEXT,

    container_xml_bytes INTEGER,

    container_downloaded_at TEXT,

    review_structure_status TEXT
        CHECK (
            review_structure_status IS NULL
            OR
            review_structure_status IN (
                'STRUCTURED',
                'REVIEW_UNAVAILABLE',
                'NOT_PEER_REVIEWED',
                'NO_REVIEW_EVIDENCE',
                'ERROR'
            )
        ),

    error_message TEXT,

    created_at TEXT NOT NULL,

    updated_at TEXT NOT NULL
);
"""


# ============================================================
# 6. SQL OFICIAL — article_versions
# ============================================================

SQL_CREATE_ARTICLE_VERSIONS = """
CREATE TABLE IF NOT EXISTS article_versions (

    article_doi TEXT PRIMARY KEY,

    base_doi TEXT NOT NULL,

    article_version INTEGER NOT NULL,

    source_kind TEXT NOT NULL
        CHECK (
            source_kind IN (
                'CONTAINER',
                'REVIEWED_VERSION'
            )
        ),

    xml_path TEXT NOT NULL,

    xml_sha256 TEXT NOT NULL,

    xml_bytes INTEGER NOT NULL,

    article_date TEXT,

    title TEXT,

    abstract_text TEXT,

    body_text TEXT,

    abstract_word_count INTEGER,

    body_word_count INTEGER,

    article_word_count INTEGER,

    article_character_count INTEGER,

    section_count INTEGER,

    table_count INTEGER,

    figure_count INTEGER,

    reference_count INTEGER,

    parse_status TEXT NOT NULL
        DEFAULT 'PENDING'
        CHECK (
            parse_status IN (
                'PENDING',
                'PARSED',
                'ERROR'
            )
        ),

    error_message TEXT,

    collected_at TEXT,

    parsed_at TEXT,

    created_at TEXT NOT NULL,

    updated_at TEXT NOT NULL,

    FOREIGN KEY (
        base_doi
    )
    REFERENCES base_articles (
        base_doi
    )
    ON DELETE CASCADE
);
"""


# ============================================================
# 7. SQL OFICIAL — reviews
# ============================================================

SQL_CREATE_REVIEWS = """
CREATE TABLE IF NOT EXISTS reviews (

    review_doi TEXT PRIMARY KEY,

    base_doi TEXT NOT NULL,

    source_container_doi TEXT NOT NULL,

    report_id TEXT NOT NULL,

    reviewed_doi TEXT NOT NULL,

    reviewed_version INTEGER NOT NULL,

    review_date TEXT,

    recommendation_raw TEXT,

    recommendation_label TEXT
        CHECK (
            recommendation_label IS NULL
            OR
            recommendation_label IN (
                'Approved',
                'Approved with Reservations',
                'Not Approved'
            )
        ),

    review_text TEXT,

    review_text_sha256 TEXT,

    review_character_count INTEGER,

    review_word_count INTEGER,

    relation_to_container TEXT NOT NULL
        CHECK (
            relation_to_container IN (
                'CURRENT',
                'HISTORICAL'
            )
        ),

    status TEXT NOT NULL
        DEFAULT 'READY'
        CHECK (
            status IN (
                'READY',
                'ERROR'
            )
        ),

    error_message TEXT,

    extracted_at TEXT NOT NULL,

    FOREIGN KEY (
        base_doi
    )
    REFERENCES base_articles (
        base_doi
    )
    ON DELETE CASCADE,

    UNIQUE (
        source_container_doi,
        report_id
    )
);
"""


# ============================================================
# 8. SQL OFICIAL — author_responses
# ============================================================

SQL_CREATE_AUTHOR_RESPONSES = """
CREATE TABLE IF NOT EXISTS author_responses (

    response_id TEXT PRIMARY KEY,

    review_doi TEXT NOT NULL,

    base_doi TEXT NOT NULL,

    source_container_doi TEXT NOT NULL,

    response_order INTEGER NOT NULL,

    response_date TEXT,

    response_text TEXT,

    response_text_sha256 TEXT,

    response_character_count INTEGER,

    response_word_count INTEGER,

    status TEXT NOT NULL
        DEFAULT 'READY'
        CHECK (
            status IN (
                'READY',
                'ERROR'
            )
        ),

    error_message TEXT,

    extracted_at TEXT NOT NULL,

    FOREIGN KEY (
        review_doi
    )
    REFERENCES reviews (
        review_doi
    )
    ON DELETE CASCADE,

    FOREIGN KEY (
        base_doi
    )
    REFERENCES base_articles (
        base_doi
    )
    ON DELETE CASCADE,

    UNIQUE (
        review_doi,
        response_order
    )
);
"""


# ============================================================
# 9. SQL OFICIAL — peer_review_states
#
# CORREÇÃO IMPORTANTE:
#
# approved_count,
# reservations_count,
# not_approved_count
#
# SÃO NULLABLE.
# ============================================================

SQL_CREATE_PEER_REVIEW_STATES = """
CREATE TABLE IF NOT EXISTS peer_review_states (

    state_id TEXT PRIMARY KEY,

    base_doi TEXT NOT NULL,

    container_doi TEXT NOT NULL UNIQUE,

    article_version INTEGER NOT NULL,

    state_raw TEXT,

    state_raw_sha256 TEXT,

    approved_count INTEGER,

    reservations_count INTEGER,

    not_approved_count INTEGER,

    not_peer_reviewed INTEGER NOT NULL
        DEFAULT 0
        CHECK (
            not_peer_reviewed IN (
                0,
                1
            )
        ),

    status TEXT NOT NULL
        DEFAULT 'READY'
        CHECK (
            status IN (
                'READY',
                'ERROR'
            )
        ),

    error_message TEXT,

    extracted_at TEXT NOT NULL,

    FOREIGN KEY (
        base_doi
    )
    REFERENCES base_articles (
        base_doi
    )
    ON DELETE CASCADE
);
"""


# ============================================================
# 10. SQL OFICIAL — canonical_pairs
# ============================================================

SQL_CREATE_CANONICAL_PAIRS = """
CREATE TABLE IF NOT EXISTS canonical_pairs (

    pair_id TEXT PRIMARY KEY,

    base_doi TEXT NOT NULL,

    article_doi TEXT NOT NULL,

    article_version INTEGER NOT NULL,

    review_doi TEXT NOT NULL UNIQUE,

    recommendation_label TEXT
        CHECK (
            recommendation_label IS NULL
            OR
            recommendation_label IN (
                'Approved',
                'Approved with Reservations',
                'Not Approved'
            )
        ),

    relation_to_container TEXT NOT NULL
        CHECK (
            relation_to_container IN (
                'CURRENT',
                'HISTORICAL'
            )
        ),

    source_container_doi TEXT NOT NULL,

    status TEXT NOT NULL
        DEFAULT 'READY'
        CHECK (
            status IN (
                'READY',
                'ERROR'
            )
        ),

    error_message TEXT,

    created_at TEXT NOT NULL,

    updated_at TEXT NOT NULL,

    FOREIGN KEY (
        base_doi
    )
    REFERENCES base_articles (
        base_doi
    )
    ON DELETE CASCADE,

    FOREIGN KEY (
        article_doi
    )
    REFERENCES article_versions (
        article_doi
    )
    ON DELETE CASCADE,

    FOREIGN KEY (
        review_doi
    )
    REFERENCES reviews (
        review_doi
    )
    ON DELETE CASCADE
);
"""


# ============================================================
# 11. SQL OFICIAL — discovery_exclusions
# ============================================================

SQL_CREATE_DISCOVERY_EXCLUSIONS = """
CREATE TABLE IF NOT EXISTS discovery_exclusions (

    exclusion_id INTEGER PRIMARY KEY AUTOINCREMENT,

    base_doi TEXT NOT NULL,

    container_doi TEXT NOT NULL UNIQUE,

    container_version INTEGER,

    search_page INTEGER,

    search_position INTEGER,

    reason TEXT NOT NULL,

    http_status INTEGER,

    detail TEXT,

    excluded_at TEXT NOT NULL
);
"""


# ============================================================
# 12. CRIAÇÃO IDEMPOTENTE DAS TABELAS
# ============================================================

conn = open_builder_db()


schema_statements = [

    SQL_CREATE_RUN_INFO,

    SQL_CREATE_BASE_ARTICLES,

    SQL_CREATE_ARTICLE_VERSIONS,

    SQL_CREATE_REVIEWS,

    SQL_CREATE_AUTHOR_RESPONSES,

    SQL_CREATE_PEER_REVIEW_STATES,

    SQL_CREATE_CANONICAL_PAIRS,

    SQL_CREATE_DISCOVERY_EXCLUSIONS,
]


for statement in schema_statements:

    conn.execute(
        statement
    )


conn.commit()


# ============================================================
# 13. DETECTAR MIGRAÇÃO NECESSÁRIA
# ============================================================

peer_state_columns = get_column_map(
    conn,
    "peer_review_states",
)


required_peer_state_columns = {

    "state_id",
    "base_doi",
    "container_doi",
    "article_version",
    "state_raw",
    "state_raw_sha256",
    "approved_count",
    "reservations_count",
    "not_approved_count",
    "not_peer_reviewed",
    "status",
    "error_message",
    "extracted_at",
}


missing_peer_state_columns = (
    required_peer_state_columns
    -
    set(peer_state_columns)
)


if missing_peer_state_columns:

    conn.close()

    raise RuntimeError(
        "peer_review_states possui schema "
        "incompatível e faltam colunas que não podem "
        "ser migradas automaticamente com segurança: "
        f"{sorted(missing_peer_state_columns)}"
    )


count_columns = [

    "approved_count",
    "reservations_count",
    "not_approved_count",
]


nullable_problem_columns = [

    column_name

    for column_name in count_columns

    if peer_state_columns[
        column_name
    ]["notnull"]
]


peer_state_migration_needed = bool(
    nullable_problem_columns
)


peer_state_rows_before = table_count(
    conn,
    "peer_review_states",
)


peer_state_migrated = False


# ============================================================
# 14. MIGRAÇÃO TRANSACIONAL DE peer_review_states
# ============================================================

if peer_state_migration_needed:

    log(
        "Migração necessária em peer_review_states: "
        "campos de contagem ainda estão NOT NULL."
    )


    legacy_table = (
        "peer_review_states__legacy_2_0"
    )


    if table_exists(
        conn,
        legacy_table,
    ):

        conn.close()

        raise RuntimeError(
            f"A tabela temporária {legacy_table} "
            "já existe. Isso indica uma migração anterior "
            "incompleta. Nenhuma alteração adicional foi "
            "realizada."
        )


    try:

        # ----------------------------------------------------
        # Durante rebuild de tabela SQLite, desabilitamos FK
        # fora da transação.
        # ----------------------------------------------------

        conn.commit()

        conn.execute(
            "PRAGMA foreign_keys = OFF;"
        )

        conn.execute(
            "BEGIN IMMEDIATE;"
        )


        # ----------------------------------------------------
        # A. Renomear tabela antiga
        # ----------------------------------------------------

        conn.execute(
            f"""
            ALTER TABLE peer_review_states
            RENAME TO {legacy_table};
            """
        )


        # ----------------------------------------------------
        # B. Criar tabela oficial nova
        #
        # Removemos IF NOT EXISTS apenas nesta execução
        # para garantir que uma nova tabela seja criada.
        # ----------------------------------------------------

        create_new_peer_states_sql = (
            SQL_CREATE_PEER_REVIEW_STATES
            .replace(
                "CREATE TABLE IF NOT EXISTS",
                "CREATE TABLE",
                1,
            )
        )


        conn.execute(
            create_new_peer_states_sql
        )


        # ----------------------------------------------------
        # C. Copiar registros existentes sem alterar valores
        #
        # NULL continua NULL.
        # 0 continua 0.
        # Nenhuma inferência.
        # ----------------------------------------------------

        conn.execute(
            f"""
            INSERT INTO peer_review_states (

                state_id,
                base_doi,
                container_doi,
                article_version,
                state_raw,
                state_raw_sha256,
                approved_count,
                reservations_count,
                not_approved_count,
                not_peer_reviewed,
                status,
                error_message,
                extracted_at

            )
            SELECT

                state_id,
                base_doi,
                container_doi,
                article_version,
                state_raw,
                state_raw_sha256,
                approved_count,
                reservations_count,
                not_approved_count,
                not_peer_reviewed,
                status,
                error_message,
                extracted_at

            FROM {legacy_table};
            """
        )


        # ----------------------------------------------------
        # D. Remover tabela antiga
        # ----------------------------------------------------

        conn.execute(
            f"""
            DROP TABLE {legacy_table};
            """
        )


        conn.commit()

        conn.execute(
            "PRAGMA foreign_keys = ON;"
        )


        peer_state_migrated = True


    except Exception:

        try:
            conn.rollback()
        except Exception:
            pass

        try:
            conn.execute(
                "PRAGMA foreign_keys = ON;"
            )
        except Exception:
            pass

        conn.close()

        raise


# ============================================================
# 15. REVALIDAR NULLABILITY APÓS MIGRAÇÃO
# ============================================================

peer_state_columns = get_column_map(
    conn,
    "peer_review_states",
)


for column_name in count_columns:

    if peer_state_columns[
        column_name
    ]["notnull"]:

        conn.close()

        raise RuntimeError(
            "Migração não corrigiu a nullability de "
            f"{column_name}."
        )


peer_state_rows_after = table_count(
    conn,
    "peer_review_states",
)


if (
    peer_state_rows_after
    !=
    peer_state_rows_before
):

    conn.close()

    raise RuntimeError(
        "Migração alterou cardinalidade de "
        "peer_review_states: "
        f"before={peer_state_rows_before}, "
        f"after={peer_state_rows_after}."
    )


# ============================================================
# 16. VALIDAR COLUNAS DAS DEMAIS TABELAS
# ============================================================

expected_columns = {

    "run_info": {
        "key",
        "value",
        "updated_at",
    },

    "base_articles": {
        "base_doi",
        "container_doi",
        "container_version",
        "discovery_order",
        "discovered_at",
        "discovery_status",
        "container_status",
        "container_xml_path",
        "container_xml_sha256",
        "container_xml_bytes",
        "container_downloaded_at",
        "review_structure_status",
        "error_message",
        "created_at",
        "updated_at",
    },

    "article_versions": {
        "article_doi",
        "base_doi",
        "article_version",
        "source_kind",
        "xml_path",
        "xml_sha256",
        "xml_bytes",
        "article_date",
        "title",
        "abstract_text",
        "body_text",
        "abstract_word_count",
        "body_word_count",
        "article_word_count",
        "article_character_count",
        "section_count",
        "table_count",
        "figure_count",
        "reference_count",
        "parse_status",
        "error_message",
        "collected_at",
        "parsed_at",
        "created_at",
        "updated_at",
    },

    "reviews": {
        "review_doi",
        "base_doi",
        "source_container_doi",
        "report_id",
        "reviewed_doi",
        "reviewed_version",
        "review_date",
        "recommendation_raw",
        "recommendation_label",
        "review_text",
        "review_text_sha256",
        "review_character_count",
        "review_word_count",
        "relation_to_container",
        "status",
        "error_message",
        "extracted_at",
    },

    "author_responses": {
        "response_id",
        "review_doi",
        "base_doi",
        "source_container_doi",
        "response_order",
        "response_date",
        "response_text",
        "response_text_sha256",
        "response_character_count",
        "response_word_count",
        "status",
        "error_message",
        "extracted_at",
    },

    "peer_review_states": {
        "state_id",
        "base_doi",
        "container_doi",
        "article_version",
        "state_raw",
        "state_raw_sha256",
        "approved_count",
        "reservations_count",
        "not_approved_count",
        "not_peer_reviewed",
        "status",
        "error_message",
        "extracted_at",
    },

    "canonical_pairs": {
        "pair_id",
        "base_doi",
        "article_doi",
        "article_version",
        "review_doi",
        "recommendation_label",
        "relation_to_container",
        "source_container_doi",
        "status",
        "error_message",
        "created_at",
        "updated_at",
    },

    "discovery_exclusions": {
        "exclusion_id",
        "base_doi",
        "container_doi",
        "container_version",
        "search_page",
        "search_position",
        "reason",
        "http_status",
        "detail",
        "excluded_at",
    },
}


for table_name, required_columns in (
    expected_columns.items()
):

    actual_columns = set(
        get_column_map(
            conn,
            table_name,
        )
    )


    missing = (
        required_columns
        -
        actual_columns
    )


    if missing:

        conn.close()

        raise RuntimeError(
            f"Tabela {table_name} sem colunas "
            f"obrigatórias: {sorted(missing)}"
        )


# ============================================================
# 17. VALIDAR PRIMARY KEYS ESSENCIAIS
# ============================================================

expected_primary_keys = {

    "run_info":
        "key",

    "base_articles":
        "base_doi",

    "article_versions":
        "article_doi",

    "reviews":
        "review_doi",

    "author_responses":
        "response_id",

    "peer_review_states":
        "state_id",

    "canonical_pairs":
        "pair_id",

    "discovery_exclusions":
        "exclusion_id",
}


for table_name, pk_column in (
    expected_primary_keys.items()
):

    columns = get_column_map(
        conn,
        table_name,
    )


    if (
        pk_column not in columns
        or
        columns[pk_column]["pk"] != 1
    ):

        conn.close()

        raise RuntimeError(
            f"Primary key incompatível em "
            f"{table_name}: esperado {pk_column}."
        )


# ============================================================
# 18. ÍNDICES
# ============================================================

index_statements = [

    """
    CREATE INDEX IF NOT EXISTS
    idx_base_articles_discovery_order
    ON base_articles (
        discovery_order
    );
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_article_versions_base_doi
    ON article_versions (
        base_doi
    );
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_reviews_base_doi
    ON reviews (
        base_doi
    );
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_reviews_reviewed_doi
    ON reviews (
        reviewed_doi
    );
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_reviews_source_container
    ON reviews (
        source_container_doi
    );
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_author_responses_review
    ON author_responses (
        review_doi
    );
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_author_responses_base
    ON author_responses (
        base_doi
    );
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_peer_review_states_base
    ON peer_review_states (
        base_doi
    );
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_canonical_pairs_base
    ON canonical_pairs (
        base_doi
    );
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_canonical_pairs_article
    ON canonical_pairs (
        article_doi
    );
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_discovery_exclusions_base
    ON discovery_exclusions (
        base_doi
    );
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_discovery_exclusions_reason
    ON discovery_exclusions (
        reason
    );
    """,

    """
    CREATE INDEX IF NOT EXISTS
    idx_discovery_exclusions_search
    ON discovery_exclusions (
        search_page,
        search_position
    );
    """,
]


for statement in index_statements:

    conn.execute(
        statement
    )


conn.commit()


# ============================================================
# 19. run_info NORMATIVO
# ============================================================

now = utc_now()


normative_run_info = {

    "builder_name":
        BUILDER_NAME,

    "builder_version":
        BUILDER_VERSION,

    "db_schema_version":
        DB_SCHEMA_VERSION,

    "target_base_articles":
        str(
            TARGET_BASE_ARTICLES
        ),

    "pipeline_model":
        (
            "discovery->acquisition->"
            "exact_version_resolution->"
            "canonical_parsing->"
            "relational_assembly->"
            "knowledge_unit_assembly->"
            "markdown_materialization"
        ),

    "primary_analytical_unit":
        "KnowledgeUnit",

    "aggregation_key":
        "base_doi",

    "canonical_pair_role":
        "atomic_internal_relation",

    "curation_materialization":
        "markdown",

    "peer_review_state_count_semantics":
        (
            "NULL=not_observed_or_not_determinable;"
            "0=explicit_zero"
        ),
}


for key, value in (
    normative_run_info.items()
):

    conn.execute(
        """
        INSERT INTO run_info (
            key,
            value,
            updated_at
        )
        VALUES (?, ?, ?)

        ON CONFLICT(key)
        DO UPDATE SET
            value = excluded.value,
            updated_at = excluded.updated_at
        """,
        (
            key,
            value,
            now,
        ),
    )


conn.commit()


# ============================================================
# 20. CONTAGENS DO ESTADO PERSISTIDO
# ============================================================

tracked_tables = [

    "base_articles",
    "discovery_exclusions",
    "article_versions",
    "reviews",
    "author_responses",
    "peer_review_states",
    "canonical_pairs",
    "run_info",
]


table_counts = {

    table_name:
        table_count(
            conn,
            table_name,
        )

    for table_name in tracked_tables
}


# ============================================================
# 21. STATUS DOS BaseArticles
# ============================================================

base_status_rows = conn.execute(
    """
    SELECT
        container_status,
        COUNT(*)
    FROM base_articles
    GROUP BY container_status
    ORDER BY container_status
    """
).fetchall()


base_status_counts = {

    status: count

    for status, count
    in base_status_rows
}


# ============================================================
# 22. VALIDAR UNICIDADE OPERACIONAL
# ============================================================

duplicate_base_doi = conn.execute(
    """
    SELECT COUNT(*)
    FROM (
        SELECT
            base_doi
        FROM base_articles
        GROUP BY base_doi
        HAVING COUNT(*) > 1
    )
    """
).fetchone()[0]


duplicate_container_doi = conn.execute(
    """
    SELECT COUNT(*)
    FROM (
        SELECT
            container_doi
        FROM base_articles
        GROUP BY container_doi
        HAVING COUNT(*) > 1
    )
    """
).fetchone()[0]


duplicate_discovery_order = conn.execute(
    """
    SELECT COUNT(*)
    FROM (
        SELECT
            discovery_order
        FROM base_articles
        GROUP BY discovery_order
        HAVING COUNT(*) > 1
    )
    """
).fetchone()[0]


if (
    duplicate_base_doi
    or
    duplicate_container_doi
    or
    duplicate_discovery_order
):

    conn.close()

    raise RuntimeError(
        "Falha de unicidade em base_articles: "
        f"base_doi={duplicate_base_doi}, "
        f"container_doi={duplicate_container_doi}, "
        f"discovery_order={duplicate_discovery_order}."
    )


# ============================================================
# 23. DISCOVERY ORDER
# ============================================================

discovery_orders = [

    row[0]

    for row in conn.execute(
        """
        SELECT discovery_order
        FROM base_articles
        ORDER BY discovery_order
        """
    ).fetchall()
]


if discovery_orders:

    discovery_order_min = min(
        discovery_orders
    )

    discovery_order_max = max(
        discovery_orders
    )

    discovery_order_contiguous = (
        discovery_orders
        ==
        list(
            range(
                1,
                len(discovery_orders) + 1,
            )
        )
    )

else:

    discovery_order_min = None
    discovery_order_max = None
    discovery_order_contiguous = True


# ============================================================
# 24. ACCEPTED / EXCLUDED OVERLAP
# ============================================================

accepted_excluded_overlap = conn.execute(
    """
    SELECT COUNT(*)
    FROM base_articles AS b
    INNER JOIN discovery_exclusions AS e
        ON
            b.base_doi = e.base_doi
            OR
            b.container_doi = e.container_doi
    """
).fetchone()[0]


# ============================================================
# 25. INTEGRIDADE
# ============================================================

integrity = [

    row[0]

    for row in conn.execute(
        "PRAGMA integrity_check;"
    ).fetchall()
]


foreign_key_errors = conn.execute(
    "PRAGMA foreign_key_check;"
).fetchall()


if integrity != ["ok"]:

    conn.close()

    raise RuntimeError(
        "SQLite integrity_check falhou: "
        f"{integrity}"
    )


if foreign_key_errors:

    conn.close()

    raise RuntimeError(
        "SQLite foreign_key_check falhou: "
        f"{foreign_key_errors[:20]}"
    )


# ============================================================
# 26. PRAGMAS PARA RELATÓRIO
# ============================================================

journal_mode = conn.execute(
    "PRAGMA journal_mode;"
).fetchone()[0]


foreign_keys_enabled = conn.execute(
    "PRAGMA foreign_keys;"
).fetchone()[0]


busy_timeout = conn.execute(
    "PRAGMA busy_timeout;"
).fetchone()[0]


synchronous = conn.execute(
    "PRAGMA synchronous;"
).fetchone()[0]


# ============================================================
# 27. NULLABILITY FINAL
# ============================================================

final_peer_state_columns = (
    get_column_map(
        conn,
        "peer_review_states",
    )
)


approved_nullable = (
    not final_peer_state_columns[
        "approved_count"
    ]["notnull"]
)


reservations_nullable = (
    not final_peer_state_columns[
        "reservations_count"
    ]["notnull"]
)


not_approved_nullable = (
    not final_peer_state_columns[
        "not_approved_count"
    ]["notnull"]
)


# ============================================================
# 28. FECHAR CONEXÃO
# ============================================================

conn.close()


# ============================================================
# 29. LOG
# ============================================================

log(
    "SQLite schema 2.0 validado; "
    f"peer_review_states migration="
    f"{'MIGRATED' if peer_state_migrated else 'NOT_NEEDED'}."
)


# ============================================================
# 30. RELATÓRIO
# ============================================================

print()
print("=" * 72)
print(
    "F1000 CORPUS BUILDER 2.0 — "
    "SQLITE / SCHEMA MIGRATION"
)
print("=" * 72)

print(
    f"Database...............: "
    f"{DB_PATH}"
)

print(
    f"DB schema version......: "
    f"{DB_SCHEMA_VERSION}"
)

print(
    f"Journal mode...........: "
    f"{str(journal_mode).upper()}"
)

print(
    f"Foreign keys...........: "
    f"{'ON' if foreign_keys_enabled else 'OFF'}"
)

print(
    f"Busy timeout...........: "
    f"{busy_timeout} ms"
)

print(
    f"Synchronous............: "
    f"{synchronous}"
)


print()
print(
    "Schema migration:"
)

print(
    f"  peer_review_states....: "
    f"{'MIGRATED' if peer_state_migrated else 'NOT NEEDED'}"
)

print(
    f"  rows before...........: "
    f"{peer_state_rows_before}"
)

print(
    f"  rows after............: "
    f"{peer_state_rows_after}"
)

print(
    f"  approved_count nullable: "
    f"{'YES' if approved_nullable else 'NO'}"
)

print(
    f"  reservations nullable.: "
    f"{'YES' if reservations_nullable else 'NO'}"
)

print(
    f"  not_approved nullable..: "
    f"{'YES' if not_approved_nullable else 'NO'}"
)


print()
print(
    "Estado persistido:"
)

for table_name in tracked_tables:

    print(
        f"  {table_name:<24}: "
        f"{table_counts[table_name]}"
    )


print()
print(
    "BaseArticle acquisition state:"
)

print(
    f"  READY.................: "
    f"{base_status_counts.get('READY', 0)}"
)

print(
    f"  PENDING...............: "
    f"{base_status_counts.get('PENDING', 0)}"
)

print(
    f"  ERROR.................: "
    f"{base_status_counts.get('ERROR', 0)}"
)


print()

print(
    f"Discovery order contiguous: "
    f"{'YES' if discovery_order_contiguous else 'NO'}"
)


if discovery_orders:

    print(
        f"Discovery order range..: "
        f"{discovery_order_min}"
        f".."
        f"{discovery_order_max}"
    )

else:

    print(
        "Discovery order range..: EMPTY"
    )


print(
    f"Accepted/excluded overlap: "
    f"{accepted_excluded_overlap}"
)


print()
print(
    f"SQLite integrity.......: "
    f"{integrity}"
)

print(
    f"Foreign-key errors.....: "
    f"{len(foreign_key_errors)}"
)


print()
print(
    "Count semantics:"
)

print(
    "  NULL = not observed / not determinable"
)

print(
    "  0    = explicitly determined zero"
)


print()
print(
    "HTTP requests performed: 0"
)

print(
    "Destructive corpus operations: NONE"
)


print("=" * 72)
print(
    "CÉLULA 2 CONCLUÍDA"
)
print("=" * 72)

[2026-09-17T22:40:22.157852+00:00] Migração necessária em peer_review_states: campos de contagem ainda estão NOT NULL.
[2026-09-17T22:40:22.509300+00:00] SQLite schema 2.0 validado; peer_review_states migration=MIGRATED.

F1000 CORPUS BUILDER 2.0 — SQLITE / SCHEMA MIGRATION
Database...............: /content/drive/MyDrive/f1000_corpus_500/state/builder.sqlite
DB schema version......: 2.0
Journal mode...........: DELETE
Foreign keys...........: ON
Busy timeout...........: 60000 ms
Synchronous............: 1

Schema migration:
  peer_review_states....: MIGRATED
  rows before...........: 0
  rows after............: 0
  approved_count nullable: YES
  reservations nullable.: YES
  not_approved nullable..: YES

Estado persistido:
  base_articles           : 500
  discovery_exclusions    : 2
  article_versions        : 0
  reviews                 : 0
  author_responses        : 0
  peer_review_states      : 0
  canonical_pairs         : 0
  run_info                : 23

BaseArticle acquisition

In [ ]:
# ============================================================
# CÉLULA 3 — DISCOVERY IDEMPOTENTE E RESTART-SAFE
# F1000 Corpus Builder 2.0
# ============================================================

# Responsabilidade desta célula:
#
#   1. Em execução nova:
#      - consultar o SEARCH da F1000Research;
#      - descobrir os primeiros TARGET_BASE_ARTICLES
#        base_doi distintos;
#      - persistir esses candidatos em base_articles.
#
#   2. Em retomada:
#      - reconhecer candidatos já persistidos;
#      - reconhecer candidatos já excluídos;
#      - NÃO reconstruir os candidatos originais;
#      - NÃO repor exclusões;
#      - NÃO compactar discovery_order;
#      - NÃO baixar XMLs;
#      - NÃO modificar registros READY existentes.
#
# A Célula 4 será responsável por:
#
#   - validar aquisição;
#   - excluir containers inelegíveis;
#   - compactar a ordem dos aceitos;
#   - continuar o SEARCH;
#   - completar a amostra até exatamente 500 READY.
#
# Portanto:
#
#   DISCOVERY ≠ ELIGIBILITY
#
# Esta separação é intencional.


# ============================================================
# 1. PARÂMETROS DE DISCOVERY
# ============================================================

INITIAL_DISCOVERY_TARGET = TARGET_BASE_ARTICLES

if INITIAL_DISCOVERY_TARGET <= 0:
    raise RuntimeError(
        "TARGET_BASE_ARTICLES deve ser maior que zero."
    )


# ============================================================
# 2. HELPERS — SEARCH RESPONSE
# ============================================================

def extract_search_dois(response):
    """
    Extrai DOIs de uma resposta do endpoint SEARCH.

    A API da F1000Research foi observada retornando XML
    mesmo quando wt=json é solicitado.

    A função também possui suporte conservador a JSON,
    caso o endpoint altere o formato no futuro.

    Retorna:
        dois: list[str]
        response_format: str
    """

    content = response.content

    if not content:
        raise RuntimeError(
            "SEARCH retornou resposta vazia."
        )

    # --------------------------------------------------------
    # Primeiro: tentar XML
    # --------------------------------------------------------

    try:
        root = ET.fromstring(content)

        dois = []

        for element in root.iter():

            tag = element.tag

            if isinstance(tag, str):
                tag = tag.split("}")[-1].lower()
            else:
                continue

            if tag == "doi":

                value = normalize_doi(
                    element.text
                )

                if value:
                    dois.append(value)

        if dois:
            return dois, "XML"

    except ET.ParseError:
        pass

    # --------------------------------------------------------
    # Segundo: tentar JSON
    # --------------------------------------------------------

    try:
        payload = response.json()

    except Exception as exc:

        preview = (
            response.text[:500]
            if response.text
            else ""
        )

        raise RuntimeError(
            "Formato da resposta SEARCH não reconhecido. "
            f"HTTP={response.status_code}. "
            f"Preview={preview!r}"
        ) from exc

    dois = []


    def walk_json(value):

        if isinstance(value, dict):

            for key, item in value.items():

                if (
                    isinstance(key, str)
                    and key.lower() == "doi"
                ):

                    if isinstance(item, str):

                        normalized = normalize_doi(
                            item
                        )

                        if normalized:
                            dois.append(
                                normalized
                            )

                walk_json(item)

        elif isinstance(value, list):

            for item in value:
                walk_json(item)


    walk_json(payload)

    if not dois:

        raise RuntimeError(
            "SEARCH retornou JSON, mas nenhum DOI "
            "foi encontrado."
        )

    return dois, "JSON"


# ============================================================
# 3. HELPER — HTTP SEARCH
# ============================================================

def search_f1000_page(page_number):
    """
    Consulta uma página do SEARCH.

    Falhas transitórias NÃO são interpretadas como ausência
    de candidatos. Após esgotar retries, a execução para.

    Isso evita viés amostral causado por falha de rede.
    """

    if page_number < 1:
        raise ValueError(
            "page_number deve ser >= 1."
        )

    params = {
        "q": "*:*",
        "wt": "json",
        "rows": SEARCH_ROWS,
        "page": page_number,
    }

    last_error = None

    for attempt in range(
        1,
        HTTP_MAX_RETRIES + 1,
    ):

        try:

            response = requests.get(
                SEARCH_ENDPOINT,
                params=params,
                headers={
                    "User-Agent": USER_AGENT
                },
                timeout=HTTP_TIMEOUT_SECONDS,
            )

            # ------------------------------------------------
            # Sucesso
            # ------------------------------------------------

            if response.status_code == 200:

                dois, response_format = (
                    extract_search_dois(
                        response
                    )
                )

                return {
                    "dois": dois,
                    "format": response_format,
                    "status_code":
                        response.status_code,
                    "url":
                        response.url,
                }

            # ------------------------------------------------
            # HTTP transitório
            # ------------------------------------------------

            if response.status_code in {
                408,
                425,
                429,
                500,
                502,
                503,
                504,
            }:

                last_error = RuntimeError(
                    "SEARCH transient HTTP "
                    f"{response.status_code}"
                )

            else:

                # Para o SEARCH, qualquer outro HTTP é
                # tratado como falha metodológica da coleta,
                # não como página vazia.

                raise RuntimeError(
                    "SEARCH falhou com HTTP "
                    f"{response.status_code}. "
                    f"URL={response.url}"
                )

        except requests.RequestException as exc:

            last_error = exc


        if attempt < HTTP_MAX_RETRIES:

            wait_seconds = (
                HTTP_DELAY_SECONDS
                * attempt
            )

            log(
                "SEARCH retry "
                f"page={page_number}, "
                f"attempt={attempt}, "
                f"wait={wait_seconds:.2f}s, "
                f"error={last_error}"
            )

            time.sleep(
                wait_seconds
            )


    raise RuntimeError(
        "SEARCH falhou após "
        f"{HTTP_MAX_RETRIES} tentativas "
        f"na página {page_number}: "
        f"{last_error}"
    )


# ============================================================
# 4. ABRIR SQLITE
# ============================================================

conn = open_builder_db()


# ============================================================
# 5. VALIDAR ESTADO PRÉVIO
# ============================================================

existing_rows = conn.execute(
    """
    SELECT
        base_doi,
        container_doi,
        container_version,
        discovery_order,
        container_status
    FROM base_articles
    ORDER BY discovery_order
    """
).fetchall()


excluded_rows = conn.execute(
    """
    SELECT
        base_doi,
        container_doi,
        search_page,
        search_position,
        reason
    FROM discovery_exclusions
    ORDER BY exclusion_id
    """
).fetchall()


existing_base_dois = {
    row[0]
    for row in existing_rows
}


existing_container_dois = {
    row[1]
    for row in existing_rows
}


excluded_base_dois = {
    row[0]
    for row in excluded_rows
}


excluded_container_dois = {
    row[1]
    for row in excluded_rows
}


overlap_base = (
    existing_base_dois
    &
    excluded_base_dois
)


overlap_container = (
    existing_container_dois
    &
    excluded_container_dois
)


if overlap_base or overlap_container:

    conn.close()

    raise RuntimeError(
        "Estado inconsistente antes do discovery: "
        "há candidatos simultaneamente aceitos e "
        "registrados em discovery_exclusions. "
        f"base_overlap={len(overlap_base)}, "
        f"container_overlap={len(overlap_container)}."
    )


# ============================================================
# 6. DECIDIR ENTRE FRESH DISCOVERY E RESUME
# ============================================================

existing_count = len(
    existing_rows
)

excluded_count = len(
    excluded_rows
)


fresh_discovery = (
    existing_count == 0
    and
    excluded_count == 0
)


resume_discovery = (
    not fresh_discovery
)


# ============================================================
# 7. CONTADORES
# ============================================================

search_pages_requested = 0
raw_dois_processed = 0
new_base_dois = 0
duplicate_base_dois = 0
invalid_dois_skipped = 0

response_formats = set()

last_search_page = None


# ============================================================
# 8. EXECUÇÃO NOVA — INITIAL DISCOVERY
# ============================================================

print()
print("=" * 72)
print(
    "F1000 CORPUS BUILDER 2.0 — "
    "DISCOVERY / RESUME"
)
print("=" * 72)

print(
    f"Existing BaseArticles..: "
    f"{existing_count}"
)

print(
    f"Existing exclusions....: "
    f"{excluded_count}"
)

print(
    f"Mode...................: "
    f"{'FRESH' if fresh_discovery else 'RESUME'}"
)


if fresh_discovery:

    print()
    print(
        "DISCOVERY INICIAL EM ANDAMENTO"
    )
    print("=" * 72)

    discovered_base_dois = set()

    discovery_order = 0
    page_number = 1

    stop_discovery = False


    while (
        len(discovered_base_dois)
        <
        INITIAL_DISCOVERY_TARGET
    ):

        search_result = search_f1000_page(
            page_number
        )

        search_pages_requested += 1

        last_search_page = page_number

        response_formats.add(
            search_result["format"]
        )

        page_dois = search_result[
            "dois"
        ]


        if not page_dois:

            conn.close()

            raise RuntimeError(
                "SEARCH retornou página sem DOIs "
                f"antes de atingir o target. "
                f"Página={page_number}."
            )


        page_new = 0


        for search_position, raw_doi in enumerate(
            page_dois,
            start=1,
        ):

            raw_dois_processed += 1

            doi = normalize_doi(
                raw_doi
            )


            if not doi:

                invalid_dois_skipped += 1
                continue


            version = extract_version_from_doi(
                doi
            )


            base_doi = derive_base_doi(
                doi
            )


            if (
                version is None
                or
                not base_doi
                or
                base_doi == doi
            ):

                invalid_dois_skipped += 1
                continue


            if base_doi in discovered_base_dois:

                duplicate_base_dois += 1
                continue


            discovered_base_dois.add(
                base_doi
            )

            discovery_order += 1
            new_base_dois += 1
            page_new += 1

            now = utc_now()


            conn.execute(
                """
                INSERT INTO base_articles (

                    base_doi,
                    container_doi,
                    container_version,
                    discovery_order,
                    discovered_at,
                    discovery_status,
                    container_status,
                    container_xml_path,
                    container_xml_sha256,
                    container_xml_bytes,
                    container_downloaded_at,
                    review_structure_status,
                    error_message,
                    created_at,
                    updated_at

                )
                VALUES (
                    ?, ?, ?, ?, ?,
                    'DISCOVERED',
                    'PENDING',
                    NULL,
                    NULL,
                    NULL,
                    NULL,
                    NULL,
                    NULL,
                    ?,
                    ?
                )
                """,
                (
                    base_doi,
                    doi,
                    version,
                    discovery_order,
                    now,
                    now,
                    now,
                ),
            )


            if (
                len(discovered_base_dois)
                >=
                INITIAL_DISCOVERY_TARGET
            ):

                stop_discovery = True
                break


        conn.commit()


        print(
            f"Página {page_number:>3} | "
            f"DOIs recebidos: "
            f"{len(page_dois):>3} | "
            f"novos base_doi: "
            f"{page_new:>3} | "
            f"acumulado: "
            f"{len(discovered_base_dois)}"
            f"/"
            f"{INITIAL_DISCOVERY_TARGET}"
        )


        if stop_discovery:
            break


        page_number += 1

        time.sleep(
            HTTP_DELAY_SECONDS
        )


    log(
        "Discovery inicial concluído: "
        f"{len(discovered_base_dois)} "
        "base_doi persistidos."
    )


# ============================================================
# 9. RESUME — NÃO REDESCOBRIR / NÃO REPOR
# ============================================================

else:

    # --------------------------------------------------------
    # Em uma retomada, a existência de base_articles ou
    # discovery_exclusions prova que o discovery inicial já
    # começou.
    #
    # Não fazemos SEARCH aqui.
    #
    # Isso é especialmente importante no estado atual:
    #
    #   498 accepted
    #   + 2 excluded
    #   = 500 candidatos iniciais já documentados
    #
    # Reintroduzir os dois excluídos seria metodologicamente
    # incorreto.
    # --------------------------------------------------------

    initial_candidates_accounted_for = (
        existing_count
        +
        excluded_count
    )


    if (
        initial_candidates_accounted_for
        <
        INITIAL_DISCOVERY_TARGET
    ):

        # ----------------------------------------------------
        # Esse estado significaria que o discovery inicial foi
        # interrompido antes de documentar 500 candidatos.
        #
        # Não tentamos "adivinhar" a posição apenas olhando
        # para o SQLite.
        #
        # Em vez disso, a execução para para evitar alterar
        # silenciosamente a sequência amostral.
        #
        # Esse cenário não corresponde ao nosso estado atual.
        # ----------------------------------------------------

        conn.close()

        raise RuntimeError(
            "Discovery inicial aparentemente interrompido "
            "antes de atingir o target. "
            f"BaseArticles={existing_count}, "
            f"exclusions={excluded_count}, "
            f"accounted_for="
            f"{initial_candidates_accounted_for}, "
            f"target={INITIAL_DISCOVERY_TARGET}. "
            "A retomada automática desse estado exige "
            "proveniência adicional da página/posição de "
            "SEARCH e não será inferida."
        )


    print()
    print(
        "Discovery inicial já documentado."
    )

    print(
        "Nenhuma nova consulta SEARCH será feita "
        "nesta célula."
    )

    print(
        "Reposição de candidatos excluídos ficará "
        "exclusivamente a cargo da Célula 4."
    )


# ============================================================
# 10. RECARREGAR ESTADO FINAL DA CÉLULA
# ============================================================

final_rows = conn.execute(
    """
    SELECT
        base_doi,
        container_doi,
        container_version,
        discovery_order,
        container_status
    FROM base_articles
    ORDER BY discovery_order
    """
).fetchall()


final_excluded_rows = conn.execute(
    """
    SELECT
        base_doi,
        container_doi,
        search_page,
        search_position,
        reason
    FROM discovery_exclusions
    ORDER BY exclusion_id
    """
).fetchall()


final_count = len(
    final_rows
)

final_excluded_count = len(
    final_excluded_rows
)


final_base_dois = [
    row[0]
    for row in final_rows
]


final_container_dois = [
    row[1]
    for row in final_rows
]


final_orders = [
    row[3]
    for row in final_rows
]


# ============================================================
# 11. VALIDAÇÕES
# ============================================================

if (
    len(final_base_dois)
    !=
    len(set(final_base_dois))
):

    conn.close()

    raise RuntimeError(
        "Discovery inválido: base_doi duplicado."
    )


if (
    len(final_container_dois)
    !=
    len(set(final_container_dois))
):

    conn.close()

    raise RuntimeError(
        "Discovery inválido: container_doi duplicado."
    )


if (
    len(final_orders)
    !=
    len(set(final_orders))
):

    conn.close()

    raise RuntimeError(
        "Discovery inválido: discovery_order duplicado."
    )


final_excluded_base_dois = {
    row[0]
    for row in final_excluded_rows
}


final_excluded_container_dois = {
    row[1]
    for row in final_excluded_rows
}


if (
    set(final_base_dois)
    &
    final_excluded_base_dois
):

    conn.close()

    raise RuntimeError(
        "Há base_doi simultaneamente em "
        "base_articles e discovery_exclusions."
    )


if (
    set(final_container_dois)
    &
    final_excluded_container_dois
):

    conn.close()

    raise RuntimeError(
        "Há container_doi simultaneamente em "
        "base_articles e discovery_exclusions."
    )


# ============================================================
# 12. DISTRIBUIÇÃO DE VERSÕES
# ============================================================

version_distribution_rows = conn.execute(
    """
    SELECT
        container_version,
        COUNT(*)
    FROM base_articles
    GROUP BY container_version
    ORDER BY container_version
    """
).fetchall()


# ============================================================
# 13. STATUS DE AQUISIÇÃO
# ============================================================

status_rows = conn.execute(
    """
    SELECT
        container_status,
        COUNT(*)
    FROM base_articles
    GROUP BY container_status
    ORDER BY container_status
    """
).fetchall()


status_counts = {
    status: count
    for status, count
    in status_rows
}


# ============================================================
# 14. INTEGRIDADE SQLITE
# ============================================================

integrity = [
    row[0]
    for row in conn.execute(
        "PRAGMA integrity_check;"
    ).fetchall()
]


foreign_key_errors = conn.execute(
    "PRAGMA foreign_key_check;"
).fetchall()


if integrity != ["ok"]:

    conn.close()

    raise RuntimeError(
        "SQLite integrity_check falhou: "
        f"{integrity}"
    )


if foreign_key_errors:

    conn.close()

    raise RuntimeError(
        "SQLite foreign_key_check falhou: "
        f"{foreign_key_errors[:20]}"
    )


# ============================================================
# 15. REGISTRAR ESTADO OPERACIONAL
# ============================================================

now = utc_now()


run_info_updates = {

    "initial_discovery_target":
        str(
            INITIAL_DISCOVERY_TARGET
        ),

    "discovery_resume_policy":
        (
            "preserve_existing_candidates_"
            "and_exclusions"
        ),

    "discovery_last_mode":
        (
            "FRESH"
            if fresh_discovery
            else "RESUME"
        ),

    "discovery_base_articles_current":
        str(
            final_count
        ),

    "discovery_exclusions_current":
        str(
            final_excluded_count
        ),
}


if last_search_page is not None:

    run_info_updates[
        "initial_discovery_last_search_page"
    ] = str(
        last_search_page
    )


for key, value in (
    run_info_updates.items()
):

    conn.execute(
        """
        INSERT INTO run_info (
            key,
            value,
            updated_at
        )
        VALUES (?, ?, ?)

        ON CONFLICT(key)
        DO UPDATE SET
            value = excluded.value,
            updated_at = excluded.updated_at
        """,
        (
            key,
            value,
            now,
        ),
    )


conn.commit()


# ============================================================
# 16. FECHAR SQLITE
# ============================================================

conn.close()


# ============================================================
# 17. LOG
# ============================================================

log(
    "Discovery/Resume validado: "
    f"BaseArticles={final_count}, "
    f"exclusions={final_excluded_count}, "
    f"mode="
    f"{'FRESH' if fresh_discovery else 'RESUME'}."
)


# ============================================================
# 18. RELATÓRIO
# ============================================================

print()
print("=" * 72)
print(
    "F1000 CORPUS BUILDER 2.0 — "
    "DISCOVERY"
)
print("=" * 72)

print(
    f"Mode...................: "
    f"{'FRESH' if fresh_discovery else 'RESUME'}"
)

print(
    f"Initial target.........: "
    f"{INITIAL_DISCOVERY_TARGET}"
)

print(
    f"Base articles SQLite..: "
    f"{final_count}"
)

print(
    f"Discovery exclusions...: "
    f"{final_excluded_count}"
)

print(
    f"Candidates accounted...: "
    f"{final_count + final_excluded_count}"
)

print(
    f"Unique base DOI........: "
    f"{len(set(final_base_dois))}"
)

print(
    f"Unique container DOI...: "
    f"{len(set(final_container_dois))}"
)


print()
print(
    f"Search pages requested: "
    f"{search_pages_requested}"
)

print(
    f"Raw DOI processed.....: "
    f"{raw_dois_processed}"
)

print(
    f"New base DOI..........: "
    f"{new_base_dois}"
)

print(
    f"Duplicate base DOI....: "
    f"{duplicate_base_dois}"
)

print(
    f"Invalid DOI skipped...: "
    f"{invalid_dois_skipped}"
)


if response_formats:

    print(
        f"Response format(s)....: "
        f"{', '.join(sorted(response_formats))}"
    )

else:

    print(
        "Response format(s)....: "
        "N/A — resume without HTTP"
    )


print()
print(
    "Container version distribution:"
)

for version, count in (
    version_distribution_rows
):

    print(
        f"  V{version:<3}: {count}"
    )


print()
print(
    "Container acquisition state:"
)

for status in [
    "READY",
    "PENDING",
    "ERROR",
]:

    print(
        f"  {status:<22}: "
        f"{status_counts.get(status, 0)}"
    )


print()
print(
    "Discovery order:"
)

if final_orders:

    contiguous = (
        final_orders
        ==
        list(
            range(
                1,
                len(final_orders) + 1,
            )
        )
    )

    print(
        f"  Range.................: "
        f"{min(final_orders)}"
        f".."
        f"{max(final_orders)}"
    )

    print(
        f"  Contiguous............: "
        f"{'YES' if contiguous else 'NO'}"
    )

    if not contiguous:

        missing_orders = sorted(
            set(
                range(
                    min(final_orders),
                    max(final_orders) + 1,
                )
            )
            -
            set(final_orders)
        )

        print(
            f"  Missing order(s)......: "
            f"{missing_orders[:20]}"
        )

else:

    print(
        "  Range.................: EMPTY"
    )


print()
print(
    f"SQLite integrity.......: "
    f"{integrity}"
)

print(
    f"Foreign-key errors.....: "
    f"{len(foreign_key_errors)}"
)


print()
print(
    f"HTTP requests performed: "
    f"{search_pages_requested}"
)

print(
    "Container XML downloads: 0"
)


print()
print(
    "Responsibility boundary:"
)

print(
    "  C3 = discover/preserve candidates"
)

print(
    "  C4 = determine acquisition eligibility "
    "and close sample"
)


print("=" * 72)
print(
    "CÉLULA 3 CONCLUÍDA"
)
print("=" * 72)


F1000 CORPUS BUILDER 2.0 — DISCOVERY / RESUME
Existing BaseArticles..: 498
Existing exclusions....: 2
Mode...................: RESUME

Discovery inicial já documentado.
Nenhuma nova consulta SEARCH será feita nesta célula.
Reposição de candidatos excluídos ficará exclusivamente a cargo da Célula 4.
[2026-09-17T22:30:03.727054+00:00] Discovery/Resume validado: BaseArticles=498, exclusions=2, mode=RESUME.

F1000 CORPUS BUILDER 2.0 — DISCOVERY
Mode...................: RESUME
Initial target.........: 500
Base articles SQLite..: 498
Discovery exclusions...: 2
Candidates accounted...: 500
Unique base DOI........: 498
Unique container DOI...: 498

Search pages requested: 0
Raw DOI processed.....: 0
New base DOI..........: 0
Duplicate base DOI....: 0
Invalid DOI skipped...: 0
Response format(s)....: N/A — resume without HTTP

Container version distribution:
  V1  : 277
  V2  : 157
  V3  : 47
  V4  : 13
  V5  : 2
  V6  : 1
  V7  : 1

Container acquisition state:
  READY                 : 498
 

In [ ]:
# ============================================================
# CÉLULA 4 — AQUISIÇÃO, ELEGIBILIDADE E FECHAMENTO DA AMOSTRA
# F1000 Corpus Builder 2.0
# ============================================================

# Responsabilidades:
#
#   1. Reutilizar containers locais já válidos.
#   2. Validar registros existentes ainda não READY.
#   3. Registrar inelegibilidades determinísticas.
#   4. Remover candidatos inelegíveis de base_articles.
#   5. Compactar discovery_order dos candidatos aceitos.
#   6. Continuar o SEARCH até obter exatamente
#      TARGET_BASE_ARTICLES containers READY.
#   7. Fazer auditoria física + relacional final.
#
# PRINCÍPIOS:
#
#   - erro SQLite != erro XML;
#   - erro transitório HTTP NÃO exclui candidato;
#   - 404/410 no endpoint ARTICLE XML = inelegibilidade;
#   - conteúdo HTTP 200 que não seja um Article XML válido
#     = inelegibilidade técnica documentada;
#   - nenhuma exclusão é decisão de curadoria científica;
#   - execução é idempotente e restart-safe;
#   - somente um escritor SQLite;
#   - transações curtas;
#   - amostra final = exatamente 500 READY.
#
# A regra amostral operacional é:
#
#   "primeiros N base_doi distintos, na ordem observada pelo
#    endpoint SEARCH, para os quais o container DOI observado
#    possui XML de artigo válido e recuperável."
#
# ============================================================


# ============================================================
# 1. CONSTANTES
# ============================================================

ELIGIBLE_HTTP_STATUS = 200

TERMINAL_NOT_FOUND_STATUSES = {
    404,
    410,
}

TRANSIENT_HTTP_STATUSES = {
    408,
    425,
    429,
    500,
    502,
    503,
    504,
}


# ============================================================
# 2. XML HELPERS
# ============================================================

def local_tag(tag):
    """
    Remove namespace XML e normaliza tag para lowercase.
    """

    if not isinstance(tag, str):
        return ""

    return tag.split("}")[-1].lower()


def find_article_doi_in_xml(root):
    """
    Procura o DOI principal do artigo no front/article-meta.

    Evita usar indiscriminadamente qualquer <article-id>
    existente em sub-articles/reviewer reports.
    """

    front = None

    for child in list(root):

        if local_tag(child.tag) == "front":
            front = child
            break

    if front is None:
        return None


    article_meta = None

    for element in front.iter():

        if local_tag(element.tag) == "article-meta":
            article_meta = element
            break


    if article_meta is None:
        return None


    # Primeiro: article-id explicitamente DOI.

    for element in article_meta.iter():

        if local_tag(element.tag) != "article-id":
            continue

        pub_id_type = (
            element.attrib.get("pub-id-type")
            or
            element.attrib.get("publication-id-type")
            or
            ""
        ).strip().lower()

        if pub_id_type == "doi":

            value = normalize_doi(
                element.text
            )

            if value:
                return value


    # Fallback conservador: article-id cujo conteúdo pareça DOI.

    for element in article_meta.iter():

        if local_tag(element.tag) != "article-id":
            continue

        value = normalize_doi(
            element.text
        )

        if (
            value
            and
            value.startswith("10.")
            and
            "/" in value
        ):
            return value


    return None


def validate_article_xml_bytes(
    xml_bytes,
    expected_doi=None,
):
    """
    Valida um payload como XML de artigo F1000 utilizável.

    Retorna dicionário:
        valid
        reason
        detail
        internal_doi
        root_tag

    Não acessa SQLite.
    """

    if not xml_bytes:

        return {
            "valid": False,
            "reason": "EMPTY_XML",
            "detail": "Payload XML vazio.",
            "internal_doi": None,
            "root_tag": None,
        }


    try:

        root = ET.fromstring(
            xml_bytes
        )

    except ET.ParseError as exc:

        return {
            "valid": False,
            "reason": "INVALID_XML",
            "detail": str(exc),
            "internal_doi": None,
            "root_tag": None,
        }


    root_name = local_tag(
        root.tag
    )


    if root_name != "article":

        return {
            "valid": False,
            "reason": "NOT_ARTICLE_XML",
            "detail":
                f"Root XML inesperado: {root_name!r}.",
            "internal_doi": None,
            "root_tag": root_name,
        }


    internal_doi = find_article_doi_in_xml(
        root
    )


    if expected_doi:

        normalized_expected = normalize_doi(
            expected_doi
        )

        normalized_internal = normalize_doi(
            internal_doi
        )


        if not normalized_internal:

            return {
                "valid": False,
                "reason": "ARTICLE_DOI_MISSING",
                "detail":
                    "XML de artigo sem DOI principal "
                    "identificável.",
                "internal_doi": None,
                "root_tag": root_name,
            }


        if (
            normalized_internal
            !=
            normalized_expected
        ):

            return {
                "valid": False,
                "reason": "ARTICLE_DOI_MISMATCH",
                "detail":
                    "DOI interno não corresponde ao DOI "
                    "solicitado. "
                    f"expected={normalized_expected}, "
                    f"internal={normalized_internal}",
                "internal_doi":
                    normalized_internal,
                "root_tag":
                    root_name,
            }


    return {
        "valid": True,
        "reason": None,
        "detail": None,
        "internal_doi":
            normalize_doi(internal_doi),
        "root_tag": root_name,
    }


# ============================================================
# 3. CAMINHO DETERMINÍSTICO DO CONTAINER
# ============================================================

def container_xml_path(container_doi):
    """
    Caminho determinístico do XML bruto de container.
    """

    return (
        RAW_CONTAINERS_DIR
        /
        f"{safe_filename(container_doi)}.xml"
    )


# ============================================================
# 4. REUTILIZAÇÃO LOCAL
# ============================================================

def inspect_local_container(
    container_doi,
):
    """
    Examina arquivo local SEM tocar no SQLite.

    Retorna:
        exists
        valid
        path
        sha256
        bytes
        validation

    Erros de filesystem são propagados.
    """

    path = container_xml_path(
        container_doi
    )


    if not path.exists():

        return {
            "exists": False,
            "valid": False,
            "path": path,
            "sha256": None,
            "bytes": None,
            "validation": None,
        }


    xml_bytes = path.read_bytes()


    validation = (
        validate_article_xml_bytes(
            xml_bytes,
            expected_doi=container_doi,
        )
    )


    if not validation["valid"]:

        return {
            "exists": True,
            "valid": False,
            "path": path,
            "sha256": None,
            "bytes": len(xml_bytes),
            "validation": validation,
        }


    return {
        "exists": True,
        "valid": True,
        "path": path,
        "sha256":
            sha256_bytes(xml_bytes),
        "bytes":
            len(xml_bytes),
        "validation":
            validation,
    }


# ============================================================
# 5. HTTP — ARTICLE XML
# ============================================================

def acquire_article_xml(
    article_doi,
):
    """
    Adquire um XML de artigo.

    Retornos possíveis:

        {
            outcome: "READY",
            ...
        }

        {
            outcome: "INELIGIBLE",
            reason: ...,
            ...
        }

    Falhas transitórias após retries:
        -> RAISE RuntimeError

    Assim, indisponibilidade de rede não altera a composição
    da amostra.
    """

    params = {
        "doi": article_doi,
    }

    last_error = None


    for attempt in range(
        1,
        HTTP_MAX_RETRIES + 1,
    ):

        try:

            response = requests.get(
                ARTICLE_XML_ENDPOINT,
                params=params,
                headers={
                    "User-Agent": USER_AGENT
                },
                timeout=HTTP_TIMEOUT_SECONDS,
            )


            # ------------------------------------------------
            # 200 — validar conteúdo
            # ------------------------------------------------

            if (
                response.status_code
                ==
                ELIGIBLE_HTTP_STATUS
            ):

                xml_bytes = response.content

                validation = (
                    validate_article_xml_bytes(
                        xml_bytes,
                        expected_doi=article_doi,
                    )
                )


                if validation["valid"]:

                    return {
                        "outcome": "READY",
                        "http_status":
                            response.status_code,
                        "xml_bytes":
                            xml_bytes,
                        "validation":
                            validation,
                        "attempts":
                            attempt,
                        "url":
                            response.url,
                    }


                # HTTP 200 mas não é o recurso XML esperado.
                # Isso é uma inelegibilidade técnica
                # reproduzível do endpoint de aquisição.

                return {
                    "outcome": "INELIGIBLE",
                    "http_status":
                        response.status_code,
                    "reason":
                        validation["reason"],
                    "detail":
                        validation["detail"],
                    "validation":
                        validation,
                    "attempts":
                        attempt,
                    "url":
                        response.url,
                }


            # ------------------------------------------------
            # 404 / 410 — ausência determinística
            # ------------------------------------------------

            if (
                response.status_code
                in
                TERMINAL_NOT_FOUND_STATUSES
            ):

                return {
                    "outcome": "INELIGIBLE",
                    "http_status":
                        response.status_code,
                    "reason":
                        "XML_NOT_FOUND",
                    "detail":
                        (
                            "ARTICLE XML endpoint returned "
                            f"HTTP {response.status_code}."
                        ),
                    "validation":
                        None,
                    "attempts":
                        attempt,
                    "url":
                        response.url,
                }


            # ------------------------------------------------
            # HTTP transitório
            # ------------------------------------------------

            if (
                response.status_code
                in
                TRANSIENT_HTTP_STATUSES
            ):

                last_error = RuntimeError(
                    "ARTICLE XML transient HTTP "
                    f"{response.status_code}"
                )


            else:

                # Outros códigos não são interpretados como
                # inelegibilidade sem evidência.

                raise RuntimeError(
                    "ARTICLE XML endpoint retornou HTTP "
                    f"{response.status_code} para "
                    f"{article_doi}. "
                    f"URL={response.url}"
                )


        except requests.RequestException as exc:

            last_error = exc


        if attempt < HTTP_MAX_RETRIES:

            wait_seconds = (
                HTTP_DELAY_SECONDS
                * attempt
            )

            log(
                "ARTICLE XML retry "
                f"doi={article_doi}, "
                f"attempt={attempt}, "
                f"wait={wait_seconds:.2f}s, "
                f"error={last_error}"
            )

            time.sleep(
                wait_seconds
            )


    # --------------------------------------------------------
    # IMPORTANTE:
    # falha transitória persistente INTERROMPE a coleta.
    # Não excluímos o candidato.
    # --------------------------------------------------------

    raise RuntimeError(
        "Aquisição interrompida após falhas "
        "transitórias persistentes para "
        f"{article_doi}. "
        "O candidato NÃO foi excluído. "
        f"Último erro: {last_error}"
    )


# ============================================================
# 6. PERSISTIR XML
# ============================================================

def persist_container_xml(
    container_doi,
    xml_bytes,
):
    """
    Persiste XML bruto de forma determinística.

    O conteúdo já deve ter sido validado antes desta função.
    """

    path = container_xml_path(
        container_doi
    )

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    # Escrita atômica local: primeiro arquivo temporário,
    # depois replace.

    temp_path = path.with_suffix(
        path.suffix + ".tmp"
    )


    temp_path.write_bytes(
        xml_bytes
    )


    temp_path.replace(
        path
    )


    return {
        "path": path,
        "sha256":
            sha256_bytes(xml_bytes),
        "bytes":
            len(xml_bytes),
    }


# ============================================================
# 7. ATUALIZAR BaseArticle COMO READY
# ============================================================

def mark_base_article_ready(
    conn,
    base_doi,
    container_doi,
    file_info,
):
    """
    Atualiza apenas o estado SQLite.

    Erros SQLite são propagados.
    """

    now = utc_now()


    conn.execute(
        """
        UPDATE base_articles
        SET
            container_status = 'READY',
            container_xml_path = ?,
            container_xml_sha256 = ?,
            container_xml_bytes = ?,
            container_downloaded_at =
                COALESCE(
                    container_downloaded_at,
                    ?
                ),
            error_message = NULL,
            updated_at = ?
        WHERE
            base_doi = ?
            AND
            container_doi = ?
        """,
        (
            str(
                file_info["path"]
            ),
            file_info["sha256"],
            file_info["bytes"],
            now,
            now,
            base_doi,
            container_doi,
        ),
    )


    if conn.total_changes < 1:
        # total_changes é cumulativo, portanto a checagem
        # forte será feita pelo SELECT abaixo.
        pass


    row = conn.execute(
        """
        SELECT
            container_status,
            container_xml_path,
            container_xml_sha256,
            container_xml_bytes
        FROM base_articles
        WHERE base_doi = ?
        """,
        (
            base_doi,
        ),
    ).fetchone()


    if row is None:

        raise RuntimeError(
            "BaseArticle desapareceu durante "
            f"mark READY: {base_doi}"
        )


    if row[0] != "READY":

        raise RuntimeError(
            "Falha ao marcar BaseArticle READY: "
            f"{base_doi}"
        )


# ============================================================
# 8. REGISTRAR EXCLUSÃO
# ============================================================

def record_discovery_exclusion(
    conn,
    *,
    base_doi,
    container_doi,
    container_version,
    search_page,
    search_position,
    reason,
    http_status,
    detail,
):
    """
    Registra inelegibilidade técnica.

    Idempotente por container_doi.
    """

    now = utc_now()


    conn.execute(
        """
        INSERT INTO discovery_exclusions (

            base_doi,
            container_doi,
            container_version,
            search_page,
            search_position,
            reason,
            http_status,
            detail,
            excluded_at

        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)

        ON CONFLICT(container_doi)
        DO UPDATE SET

            base_doi =
                excluded.base_doi,

            container_version =
                COALESCE(
                    discovery_exclusions.container_version,
                    excluded.container_version
                ),

            search_page =
                COALESCE(
                    discovery_exclusions.search_page,
                    excluded.search_page
                ),

            search_position =
                COALESCE(
                    discovery_exclusions.search_position,
                    excluded.search_position
                ),

            reason =
                excluded.reason,

            http_status =
                excluded.http_status,

            detail =
                excluded.detail
        """,
        (
            base_doi,
            container_doi,
            container_version,
            search_page,
            search_position,
            reason,
            http_status,
            detail,
            now,
        ),
    )


# ============================================================
# 9. COMPACTAR discovery_order
# ============================================================

def compact_discovery_order(
    conn,
):
    """
    Normaliza discovery_order para 1..N preservando a ordem
    relativa atual.

    Usa valores negativos temporários para evitar colisões
    com UNIQUE(discovery_order).
    """

    rows = conn.execute(
        """
        SELECT
            base_doi,
            discovery_order
        FROM base_articles
        ORDER BY discovery_order, base_doi
        """
    ).fetchall()


    expected = list(
        range(
            1,
            len(rows) + 1,
        )
    )


    current = [
        row[1]
        for row in rows
    ]


    if current == expected:

        return False


    # --------------------------------------------------------
    # Fase temporária negativa
    # --------------------------------------------------------

    for new_order, (
        base_doi,
        old_order,
    ) in enumerate(
        rows,
        start=1,
    ):

        conn.execute(
            """
            UPDATE base_articles
            SET
                discovery_order = ?
            WHERE base_doi = ?
            """,
            (
                -new_order,
                base_doi,
            ),
        )


    # --------------------------------------------------------
    # Fase definitiva positiva
    # --------------------------------------------------------

    for new_order, (
        base_doi,
        old_order,
    ) in enumerate(
        rows,
        start=1,
    ):

        conn.execute(
            """
            UPDATE base_articles
            SET
                discovery_order = ?,
                updated_at = ?
            WHERE base_doi = ?
            """,
            (
                new_order,
                utc_now(),
                base_doi,
            ),
        )


    return True


# ============================================================
# 10. SEARCH ROBUSTO PARA SUPLEMENTAÇÃO
# ============================================================

def search_page_for_acquisition(
    page_number,
):
    """
    SEARCH utilizado para suplementação.

    Reutiliza a semântica da Célula 3:
    falha transitória interrompe, nunca pula página.
    """

    return search_f1000_page(
        page_number
    )


# ============================================================
# 11. DESCOBRIR POSIÇÃO ORIGINAL DOS CANDIDATOS EXISTENTES
# ============================================================

# Para a execução atual, as duas exclusões já possuem
# search_page/search_position.
#
# Para registros existentes, discovery_order é suficiente
# para a amostra aceita.
#
# Na suplementação, registraremos explicitamente página e
# posição somente se um novo candidato for excluído.


# ============================================================
# 12. ABRIR SQLITE
# ============================================================

conn = open_builder_db()


# ============================================================
# 13. VALIDAR PRÉ-CONDIÇÕES
# ============================================================

initial_rows = conn.execute(
    """
    SELECT
        base_doi,
        container_doi,
        container_version,
        discovery_order,
        container_status,
        container_xml_path,
        container_xml_sha256,
        container_xml_bytes
    FROM base_articles
    ORDER BY discovery_order
    """
).fetchall()


initial_exclusions = conn.execute(
    """
    SELECT
        base_doi,
        container_doi,
        search_page,
        search_position,
        reason
    FROM discovery_exclusions
    ORDER BY exclusion_id
    """
).fetchall()


initial_count = len(
    initial_rows
)


if initial_count > TARGET_BASE_ARTICLES:

    conn.close()

    raise RuntimeError(
        "Há mais BaseArticles do que o target antes "
        "da aquisição. "
        f"count={initial_count}, "
        f"target={TARGET_BASE_ARTICLES}."
    )


if (
    initial_count
    +
    len(initial_exclusions)
    <
    TARGET_BASE_ARTICLES
):

    conn.close()

    raise RuntimeError(
        "Discovery inicial incompleto. "
        "Execute a Célula 3 antes da Célula 4."
    )


print()
print("=" * 72)
print(
    "AQUISIÇÃO E FECHAMENTO DA AMOSTRA"
)
print("=" * 72)

print(
    f"BaseArticles no início.: "
    f"{initial_count}"
)

print(
    f"Exclusões já registradas: "
    f"{len(initial_exclusions)}"
)

print(
    f"Target.................: "
    f"{TARGET_BASE_ARTICLES}"
)


# ============================================================
# 14. CONTADORES
# ============================================================

reused_this_run = 0
downloaded_this_run = 0
excluded_this_run = 0
additional_candidates_considered = 0
search_pages_requested = 0
physical_errors = []

initial_records_processed = 0


# ============================================================
# 15. VALIDAR / REUTILIZAR BaseArticles EXISTENTES
# ============================================================

# IMPORTANTE:
#
# O filesystem é validado primeiro.
# Só depois tocamos no SQLite.
#
# Um erro SQLite não pode cair dentro de um bloco que
# interprete a falha como XML inválido.


existing_rows = conn.execute(
    """
    SELECT
        base_doi,
        container_doi,
        container_version,
        discovery_order,
        container_status
    FROM base_articles
    ORDER BY discovery_order
    """
).fetchall()


for index, (
    base_doi,
    container_doi,
    container_version,
    discovery_order,
    container_status,
) in enumerate(
    existing_rows,
    start=1,
):

    initial_records_processed += 1


    # --------------------------------------------------------
    # A. Tentar reutilização local
    # --------------------------------------------------------

    local_info = inspect_local_container(
        container_doi
    )


    if local_info["valid"]:

        file_info = {
            "path":
                local_info["path"],
            "sha256":
                local_info["sha256"],
            "bytes":
                local_info["bytes"],
        }


        mark_base_article_ready(
            conn,
            base_doi,
            container_doi,
            file_info,
        )

        conn.commit()

        reused_this_run += 1


    else:

        # ----------------------------------------------------
        # B. Arquivo ausente ou inválido:
        #    consultar endpoint.
        #
        # Se houver erro transitório, acquire_article_xml
        # lança exceção e paramos sem excluir o candidato.
        # ----------------------------------------------------

        acquisition = acquire_article_xml(
            container_doi
        )


        if acquisition["outcome"] == "READY":

            persisted = persist_container_xml(
                container_doi,
                acquisition["xml_bytes"],
            )


            mark_base_article_ready(
                conn,
                base_doi,
                container_doi,
                persisted,
            )

            conn.commit()

            downloaded_this_run += 1


        elif (
            acquisition["outcome"]
            ==
            "INELIGIBLE"
        ):

            # ------------------------------------------------
            # Preservar a posição original se ela puder ser
            # inferida do discovery inicial.
            #
            # Com SEARCH_ROWS=100:
            #   order 1..100   -> page 1
            #   order 101..200 -> page 2
            #
            # Isso é válido para o discovery inicial observado,
            # no qual cada posição resultou em base_doi único.
            #
            # Para suplementação usamos a página/posição real.
            # ------------------------------------------------

            search_page = (
                (
                    discovery_order - 1
                )
                //
                SEARCH_ROWS
            ) + 1


            search_position = (
                (
                    discovery_order - 1
                )
                %
                SEARCH_ROWS
            ) + 1


            record_discovery_exclusion(
                conn,
                base_doi=base_doi,
                container_doi=container_doi,
                container_version=
                    container_version,
                search_page=
                    search_page,
                search_position=
                    search_position,
                reason=
                    acquisition["reason"],
                http_status=
                    acquisition["http_status"],
                detail=
                    acquisition["detail"],
            )


            # Remove da amostra aceita.
            # Tabelas posteriores ainda estão vazias neste
            # estágio, mas ON DELETE CASCADE preserva
            # consistência em retomadas futuras.

            conn.execute(
                """
                DELETE FROM base_articles
                WHERE base_doi = ?
                """,
                (
                    base_doi,
                ),
            )


            conn.commit()

            excluded_this_run += 1


            print(
                f"  EXCLUDE "
                f"#{discovery_order} "
                f"{container_doi} | "
                f"{acquisition['reason']} | "
                f"HTTP="
                f"{acquisition['http_status']}"
            )


        else:

            conn.close()

            raise RuntimeError(
                "Outcome de aquisição desconhecido: "
                f"{acquisition}"
            )


    # --------------------------------------------------------
    # Progresso
    # --------------------------------------------------------

    if (
        index == 1
        or
        index % 25 == 0
        or
        index == len(existing_rows)
    ):

        print(
            f"[{index:>3}/{len(existing_rows)}] "
            f"reused={reused_this_run} | "
            f"downloaded={downloaded_this_run} | "
            f"excluded={excluded_this_run}"
        )


# ============================================================
# 16. COMPACTAR IMEDIATAMENTE APÓS EXCLUSÕES
# ============================================================

try:

    conn.execute(
        "BEGIN IMMEDIATE;"
    )

    compacted = compact_discovery_order(
        conn
    )

    conn.commit()

except Exception:

    conn.rollback()
    conn.close()
    raise


if compacted:

    log(
        "discovery_order compactado antes "
        "da suplementação."
    )


# ============================================================
# 17. VALIDAR READY APÓS BASE EXISTENTE
# ============================================================

current_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM base_articles
    """
).fetchone()[0]


ready_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM base_articles
    WHERE container_status = 'READY'
    """
).fetchone()[0]


non_ready_rows = conn.execute(
    """
    SELECT
        base_doi,
        container_doi,
        container_status
    FROM base_articles
    WHERE container_status <> 'READY'
    ORDER BY discovery_order
    """
).fetchall()


if non_ready_rows:

    conn.close()

    raise RuntimeError(
        "Após validação dos candidatos existentes, "
        "restaram BaseArticles não READY: "
        f"{non_ready_rows[:20]}"
    )


if current_count != ready_count:

    conn.close()

    raise RuntimeError(
        "Inconsistência: quantidade de BaseArticles "
        "difere da quantidade READY."
    )


print()
print(
    "Após validação/compactação:"
)

print(
    f"  READY.................: "
    f"{ready_count}"
)

print(
    f"  replacements needed...: "
    f"{TARGET_BASE_ARTICLES - ready_count}"
)


# ============================================================
# 18. CONJUNTO GLOBAL DE CANDIDATOS JÁ CONSIDERADOS
# ============================================================

accepted_base_dois = {
    row[0]
    for row in conn.execute(
        """
        SELECT base_doi
        FROM base_articles
        """
    ).fetchall()
}


accepted_container_dois = {
    row[0]
    for row in conn.execute(
        """
        SELECT container_doi
        FROM base_articles
        """
    ).fetchall()
}


excluded_db_rows = conn.execute(
    """
    SELECT
        base_doi,
        container_doi
    FROM discovery_exclusions
    """
).fetchall()


excluded_base_dois = {
    row[0]
    for row in excluded_db_rows
}


excluded_container_dois = {
    row[1]
    for row in excluded_db_rows
}


considered_base_dois = (
    accepted_base_dois
    |
    excluded_base_dois
)


considered_container_dois = (
    accepted_container_dois
    |
    excluded_container_dois
)


# ============================================================
# 19. SUPLEMENTAÇÃO DA AMOSTRA
# ============================================================

# Para não depender de "a página 6 é necessariamente a
# próxima", fazemos um RESCAN determinístico desde a página 1.
#
# Candidatos já aceitos ou excluídos são ignorados.
#
# Isso torna a retomada robusta mesmo se, em outra execução,
# o discovery inicial tivesse duplicatas entre páginas.


page_number = 1

consecutive_pages_without_new_candidate = 0

MAX_EMPTY_NEW_CANDIDATE_PAGES = 20


while ready_count < TARGET_BASE_ARTICLES:

    search_result = (
        search_page_for_acquisition(
            page_number
        )
    )

    search_pages_requested += 1


    page_dois = search_result[
        "dois"
    ]


    if not page_dois:

        conn.close()

        raise RuntimeError(
            "SEARCH retornou página vazia antes "
            "de completar a amostra. "
            f"page={page_number}"
        )


    print()
    print(
        f"Search page {page_number}: "
        f"{len(page_dois)} candidatos"
    )


    page_had_new_candidate = False


    for search_position, raw_doi in enumerate(
        page_dois,
        start=1,
    ):

        if ready_count >= TARGET_BASE_ARTICLES:
            break


        candidate_doi = normalize_doi(
            raw_doi
        )


        if not candidate_doi:
            continue


        candidate_version = (
            extract_version_from_doi(
                candidate_doi
            )
        )


        candidate_base_doi = (
            derive_base_doi(
                candidate_doi
            )
        )


        if (
            candidate_version is None
            or
            not candidate_base_doi
            or
            candidate_base_doi
            ==
            candidate_doi
        ):

            continue


        # ----------------------------------------------------
        # Base DOI já considerado:
        # não pode voltar à amostra.
        # ----------------------------------------------------

        if (
            candidate_base_doi
            in
            considered_base_dois
        ):

            continue


        page_had_new_candidate = True

        additional_candidates_considered += 1


        # ----------------------------------------------------
        # Adquirir antes de inserir no SQLite.
        #
        # Assim um candidato só entra em base_articles quando
        # já sabemos que seu container é elegível.
        # ----------------------------------------------------

        acquisition = acquire_article_xml(
            candidate_doi
        )


        if (
            acquisition["outcome"]
            ==
            "INELIGIBLE"
        ):

            record_discovery_exclusion(
                conn,
                base_doi=
                    candidate_base_doi,
                container_doi=
                    candidate_doi,
                container_version=
                    candidate_version,
                search_page=
                    page_number,
                search_position=
                    search_position,
                reason=
                    acquisition["reason"],
                http_status=
                    acquisition["http_status"],
                detail=
                    acquisition["detail"],
            )

            conn.commit()

            excluded_this_run += 1

            considered_base_dois.add(
                candidate_base_doi
            )

            considered_container_dois.add(
                candidate_doi
            )


            print(
                f"  EXCLUDE supplemental "
                f"page={page_number} "
                f"pos={search_position} "
                f"{candidate_doi} | "
                f"{acquisition['reason']} | "
                f"HTTP="
                f"{acquisition['http_status']}"
            )


            time.sleep(
                HTTP_DELAY_SECONDS
            )

            continue


        if (
            acquisition["outcome"]
            !=
            "READY"
        ):

            conn.close()

            raise RuntimeError(
                "Outcome inesperado na suplementação: "
                f"{acquisition}"
            )


        # ----------------------------------------------------
        # Persistir XML
        # ----------------------------------------------------

        persisted = persist_container_xml(
            candidate_doi,
            acquisition["xml_bytes"],
        )


        # ----------------------------------------------------
        # Ordem é sempre N+1 APÓS compactação.
        # ----------------------------------------------------

        new_discovery_order = (
            current_count
            + 1
        )

        now = utc_now()


        try:

            conn.execute(
                "BEGIN IMMEDIATE;"
            )


            conn.execute(
                """
                INSERT INTO base_articles (

                    base_doi,
                    container_doi,
                    container_version,
                    discovery_order,
                    discovered_at,
                    discovery_status,
                    container_status,
                    container_xml_path,
                    container_xml_sha256,
                    container_xml_bytes,
                    container_downloaded_at,
                    review_structure_status,
                    error_message,
                    created_at,
                    updated_at

                )
                VALUES (
                    ?, ?, ?, ?, ?,
                    'DISCOVERED',
                    'READY',
                    ?, ?, ?, ?,
                    NULL,
                    NULL,
                    ?, ?
                )
                """,
                (
                    candidate_base_doi,
                    candidate_doi,
                    candidate_version,
                    new_discovery_order,
                    now,
                    str(
                        persisted["path"]
                    ),
                    persisted["sha256"],
                    persisted["bytes"],
                    now,
                    now,
                    now,
                ),
            )


            conn.commit()


        except Exception:

            conn.rollback()
            conn.close()
            raise


        current_count += 1
        ready_count += 1

        downloaded_this_run += 1


        considered_base_dois.add(
            candidate_base_doi
        )

        considered_container_dois.add(
            candidate_doi
        )


        print(
            f"  ACCEPT "
            f"#{new_discovery_order} "
            f"{candidate_doi}"
        )


        time.sleep(
            HTTP_DELAY_SECONDS
        )


    if page_had_new_candidate:

        consecutive_pages_without_new_candidate = 0

    else:

        consecutive_pages_without_new_candidate += 1


    if (
        consecutive_pages_without_new_candidate
        >=
        MAX_EMPTY_NEW_CANDIDATE_PAGES
    ):

        conn.close()

        raise RuntimeError(
            "Não foram encontrados novos base_doi "
            "em "
            f"{MAX_EMPTY_NEW_CANDIDATE_PAGES} "
            "páginas consecutivas do SEARCH."
        )


    page_number += 1


    if ready_count < TARGET_BASE_ARTICLES:

        time.sleep(
            HTTP_DELAY_SECONDS
        )


# ============================================================
# 20. COMPACTAÇÃO DEFENSIVA FINAL
# ============================================================

try:

    conn.execute(
        "BEGIN IMMEDIATE;"
    )

    compact_discovery_order(
        conn
    )

    conn.commit()

except Exception:

    conn.rollback()
    conn.close()
    raise


# ============================================================
# 21. AUDITORIA RELACIONAL FINAL
# ============================================================

final_rows = conn.execute(
    """
    SELECT
        base_doi,
        container_doi,
        container_version,
        discovery_order,
        container_status,
        container_xml_path,
        container_xml_sha256,
        container_xml_bytes
    FROM base_articles
    ORDER BY discovery_order
    """
).fetchall()


final_count = len(
    final_rows
)


final_base_dois = [
    row[0]
    for row in final_rows
]


final_container_dois = [
    row[1]
    for row in final_rows
]


final_orders = [
    row[3]
    for row in final_rows
]


final_statuses = [
    row[4]
    for row in final_rows
]


if (
    final_count
    !=
    TARGET_BASE_ARTICLES
):

    conn.close()

    raise RuntimeError(
        "Amostra final com cardinalidade incorreta: "
        f"{final_count} != "
        f"{TARGET_BASE_ARTICLES}"
    )


if (
    len(set(final_base_dois))
    !=
    TARGET_BASE_ARTICLES
):

    conn.close()

    raise RuntimeError(
        "Amostra final não possui exatamente "
        f"{TARGET_BASE_ARTICLES} base_doi únicos."
    )


if (
    len(set(final_container_dois))
    !=
    TARGET_BASE_ARTICLES
):

    conn.close()

    raise RuntimeError(
        "Amostra final não possui exatamente "
        f"{TARGET_BASE_ARTICLES} container_doi únicos."
    )


expected_orders = list(
    range(
        1,
        TARGET_BASE_ARTICLES + 1,
    )
)


if final_orders != expected_orders:

    conn.close()

    raise RuntimeError(
        "discovery_order final não é 1..N."
    )


if any(
    status != "READY"
    for status in final_statuses
):

    conn.close()

    raise RuntimeError(
        "Amostra final contém BaseArticle não READY."
    )


# ============================================================
# 22. AUDITORIA FÍSICA DOS 500 XMLs REFERENCIADOS
# ============================================================

total_xml_bytes = 0


for (
    base_doi,
    container_doi,
    container_version,
    discovery_order,
    container_status,
    xml_path_text,
    expected_sha256,
    expected_bytes,
) in final_rows:

    if not xml_path_text:

        physical_errors.append(
            (
                container_doi,
                "MISSING_XML_PATH",
            )
        )

        continue


    xml_path = Path(
        xml_path_text
    )


    if not xml_path.exists():

        physical_errors.append(
            (
                container_doi,
                "FILE_NOT_FOUND",
            )
        )

        continue


    actual_bytes = (
        xml_path.stat().st_size
    )


    actual_sha256 = sha256_file(
        xml_path
    )


    if (
        expected_bytes is not None
        and
        actual_bytes != expected_bytes
    ):

        physical_errors.append(
            (
                container_doi,
                (
                    "BYTE_COUNT_MISMATCH "
                    f"db={expected_bytes} "
                    f"file={actual_bytes}"
                ),
            )
        )


    if (
        expected_sha256
        and
        actual_sha256 != expected_sha256
    ):

        physical_errors.append(
            (
                container_doi,
                "SHA256_MISMATCH",
            )
        )


    xml_bytes = xml_path.read_bytes()


    validation = (
        validate_article_xml_bytes(
            xml_bytes,
            expected_doi=
                container_doi,
        )
    )


    if not validation["valid"]:

        physical_errors.append(
            (
                container_doi,
                (
                    f"{validation['reason']}: "
                    f"{validation['detail']}"
                ),
            )
        )


    total_xml_bytes += actual_bytes


if physical_errors:

    conn.close()

    raise RuntimeError(
        "Auditoria física encontrou erros: "
        f"{physical_errors[:20]}"
    )


# ============================================================
# 23. AUDITORIA ACCEPTED × EXCLUDED
# ============================================================

final_exclusion_rows = conn.execute(
    """
    SELECT
        base_doi,
        container_doi,
        reason,
        http_status
    FROM discovery_exclusions
    ORDER BY exclusion_id
    """
).fetchall()


final_excluded_base_dois = {
    row[0]
    for row in final_exclusion_rows
}


final_excluded_container_dois = {
    row[1]
    for row in final_exclusion_rows
}


base_overlap = (
    set(final_base_dois)
    &
    final_excluded_base_dois
)


container_overlap = (
    set(final_container_dois)
    &
    final_excluded_container_dois
)


if base_overlap or container_overlap:

    conn.close()

    raise RuntimeError(
        "Amostra final sobrepõe discovery_exclusions. "
        f"base_overlap={base_overlap}, "
        f"container_overlap={container_overlap}"
    )


# ============================================================
# 24. DISTRIBUIÇÃO DE VERSÕES
# ============================================================

version_distribution = conn.execute(
    """
    SELECT
        container_version,
        COUNT(*)
    FROM base_articles
    GROUP BY container_version
    ORDER BY container_version
    """
).fetchall()


# ============================================================
# 25. SQLITE INTEGRITY
# ============================================================

integrity = [
    row[0]
    for row in conn.execute(
        "PRAGMA integrity_check;"
    ).fetchall()
]


foreign_key_errors = conn.execute(
    "PRAGMA foreign_key_check;"
).fetchall()


if integrity != ["ok"]:

    conn.close()

    raise RuntimeError(
        "SQLite integrity_check falhou: "
        f"{integrity}"
    )


if foreign_key_errors:

    conn.close()

    raise RuntimeError(
        "SQLite foreign_key_check falhou: "
        f"{foreign_key_errors[:20]}"
    )


# ============================================================
# 26. REGISTRAR RESULTADO EM run_info
# ============================================================

now = utc_now()


run_info_updates = {

    "acquisition_status":
        "PASS",

    "eligible_base_articles":
        str(
            final_count
        ),

    "discovery_exclusions_total":
        str(
            len(final_exclusion_rows)
        ),

    "acquisition_sampling_rule":
        (
            "first_distinct_base_doi_in_search_order_"
            "with_retrievable_valid_observed_container_xml"
        ),

    "acquisition_transient_failure_policy":
        "stop_without_exclusion",

    "acquisition_completed_at":
        now,
}


for key, value in (
    run_info_updates.items()
):

    conn.execute(
        """
        INSERT INTO run_info (
            key,
            value,
            updated_at
        )
        VALUES (?, ?, ?)

        ON CONFLICT(key)
        DO UPDATE SET
            value = excluded.value,
            updated_at = excluded.updated_at
        """,
        (
            key,
            value,
            now,
        ),
    )


conn.commit()


# ============================================================
# 27. FECHAR SQLITE
# ============================================================

conn.close()


# ============================================================
# 28. LOG
# ============================================================

log(
    "Aquisição concluída: "
    f"{final_count} BaseArticles READY, "
    f"{len(final_exclusion_rows)} exclusões."
)


# ============================================================
# 29. RELATÓRIO FINAL
# ============================================================

print()
print("=" * 72)
print(
    "F1000 CORPUS BUILDER 2.0 — "
    "ACQUISITION"
)
print("=" * 72)

print(
    f"Final sample size......: "
    f"{final_count}"
)

print(
    f"Unique base DOI........: "
    f"{len(set(final_base_dois))}"
)

print(
    f"Unique container DOI...: "
    f"{len(set(final_container_dois))}"
)

print(
    f"All containers READY...: "
    f"{all(s == 'READY' for s in final_statuses)}"
)


print()
print(
    "Container version distribution:"
)

for version, count in (
    version_distribution
):

    print(
        f"  V{version:<3}: {count}"
    )


print()
print(
    f"Reused this run........: "
    f"{reused_this_run}"
)

print(
    f"Downloaded this run....: "
    f"{downloaded_this_run}"
)

print(
    f"Excluded this run......: "
    f"{excluded_this_run}"
)

print(
    f"Total exclusions.......: "
    f"{len(final_exclusion_rows)}"
)

print(
    f"Additional candidates..: "
    f"{additional_candidates_considered}"
)

print(
    f"Search pages requested.: "
    f"{search_pages_requested}"
)


print()
print(
    f"Referenced XML bytes...: "
    f"{total_xml_bytes:,}"
)

print(
    f"Referenced XML MiB.....: "
    f"{total_xml_bytes / (1024 ** 2):.2f}"
)

print(
    f"Physical XML errors....: "
    f"{len(physical_errors)}"
)


print()
print(
    f"Accepted/excluded base overlap.....: "
    f"{len(base_overlap)}"
)

print(
    f"Accepted/excluded container overlap: "
    f"{len(container_overlap)}"
)


print()
print(
    f"SQLite integrity.......: "
    f"{integrity}"
)

print(
    f"Foreign-key errors.....: "
    f"{len(foreign_key_errors)}"
)


print()
print(
    "Transient failure policy: "
    "STOP WITHOUT EXCLUSION"
)

print(
    "Sampling target.........: "
    f"{TARGET_BASE_ARTICLES} eligible base articles"
)

print(
    "Final discovery order...: "
    f"1..{TARGET_BASE_ARTICLES}"
)

print()
print(
    "STATUS.................: PASS"
)

print("=" * 72)
print(
    "CÉLULA 4 CONCLUÍDA"
)
print("=" * 72)


AQUISIÇÃO E FECHAMENTO DA AMOSTRA
BaseArticles no início.: 498
Exclusões já registradas: 2
Target.................: 500
[  1/498] reused=1 | downloaded=0 | excluded=0
[ 25/498] reused=25 | downloaded=0 | excluded=0
[ 50/498] reused=50 | downloaded=0 | excluded=0
[ 75/498] reused=75 | downloaded=0 | excluded=0
[100/498] reused=100 | downloaded=0 | excluded=0
[125/498] reused=125 | downloaded=0 | excluded=0
[150/498] reused=150 | downloaded=0 | excluded=0
[175/498] reused=175 | downloaded=0 | excluded=0
[200/498] reused=200 | downloaded=0 | excluded=0
[225/498] reused=225 | downloaded=0 | excluded=0
[250/498] reused=250 | downloaded=0 | excluded=0
[275/498] reused=275 | downloaded=0 | excluded=0
[300/498] reused=300 | downloaded=0 | excluded=0
[325/498] reused=325 | downloaded=0 | excluded=0
[350/498] reused=350 | downloaded=0 | excluded=0
[375/498] reused=375 | downloaded=0 | excluded=0
[400/498] reused=400 | downloaded=0 | excluded=0
[425/498] reused=425 | downloaded=0 | excluded=0
[4

In [ ]:
# ============================================================
# CÉLULA 5 — EXTRAÇÃO ESTRUTURAL DOS CONTAINERS
# F1000 Corpus Builder 2.0
# ============================================================
#
# RESPONSABILIDADES
#
# 1. Ler os 500 Container XML já adquiridos.
# 2. Validar localmente cada XML.
# 3. Identificar reviewer reports estruturados.
# 4. Extrair:
#      - Review
#      - reviewed_doi exato via related-article/xlink:href
#      - recommendation
#      - review text
#      - AuthorResponses
#      - PeerReviewState observado
# 5. Classificar cada container:
#      - STRUCTURED
#      - REVIEW_UNAVAILABLE
#      - NOT_PEER_REVIEWED
#      - NO_REVIEW_EVIDENCE
# 6. Persistir entidades no SQLite.
# 7. Identificar exact reviewed_doi necessários para C6.
#
# NÃO FAZ:
#
# - downloads;
# - parsing canônico dos artigos;
# - CanonicalPairs;
# - KnowledgeUnits;
# - Markdown;
# - curadoria.
#
# Esta célula NÃO depende de C3 ou C4 em memória.
#
# ============================================================


# ============================================================
# 1. IMPORTS LOCAIS
# ============================================================

import re
import hashlib
import sqlite3
import xml.etree.ElementTree as ET

from pathlib import Path


CONTAINER_EXTRACTOR_VERSION = "2.0.2"

XLINK_HREF = (
    "{http://www.w3.org/1999/xlink}href"
)


# ============================================================
# 2. HELPERS XML AUTOSSUFICIENTES
# ============================================================

def local_tag(tag):

    if tag is None:
        return ""

    return str(tag).split("}")[-1].lower()


def element_text(element):

    if element is None:
        return ""

    parts = []

    for text in element.itertext():

        cleaned = normalize_whitespace(text)

        if cleaned:
            parts.append(cleaned)

    return normalize_whitespace(
        " ".join(parts)
    )


def find_article_doi_in_xml(root):
    """
    DOI do artigo principal.
    """

    front = None

    for child in list(root):

        if local_tag(child.tag) == "front":
            front = child
            break

    if front is None:
        return None


    article_meta = None

    for child in list(front):

        if local_tag(child.tag) == "article-meta":
            article_meta = child
            break

    if article_meta is None:
        return None


    for element in article_meta.iter():

        if local_tag(element.tag) != "article-id":
            continue

        pub_id_type = (
            element.attrib.get("pub-id-type")
            or
            element.attrib.get("publication-id-type")
            or
            ""
        ).strip().lower()

        if pub_id_type == "doi":

            doi = normalize_doi(
                element_text(element)
            )

            if doi:
                return doi


    for element in article_meta.iter():

        if local_tag(element.tag) != "article-id":
            continue

        doi = normalize_doi(
            element_text(element)
        )

        if (
            doi
            and
            doi.startswith("10.")
            and
            "/" in doi
        ):
            return doi

    return None


def validate_article_xml_bytes(
    xml_bytes,
    expected_doi=None,
):

    if not xml_bytes:

        return {
            "valid": False,
            "reason": "EMPTY_XML",
            "detail": "XML file contains zero bytes.",
            "internal_doi": None,
            "root_tag": None,
        }


    try:

        root = ET.fromstring(
            xml_bytes
        )

    except ET.ParseError as exc:

        return {
            "valid": False,
            "reason": "XML_PARSE_ERROR",
            "detail": str(exc),
            "internal_doi": None,
            "root_tag": None,
        }


    root_tag = local_tag(
        root.tag
    )


    if root_tag != "article":

        return {
            "valid": False,
            "reason": "INVALID_ROOT",
            "detail": (
                "Expected XML root <article>, "
                f"found <{root_tag}>."
            ),
            "internal_doi": None,
            "root_tag": root_tag,
        }


    internal_doi = find_article_doi_in_xml(
        root
    )


    if not internal_doi:

        return {
            "valid": False,
            "reason": "MAIN_ARTICLE_DOI_MISSING",
            "detail": (
                "Main article DOI not found."
            ),
            "internal_doi": None,
            "root_tag": root_tag,
        }


    if expected_doi:

        expected = normalize_doi(
            expected_doi
        )

        observed = normalize_doi(
            internal_doi
        )

        if observed != expected:

            return {
                "valid": False,
                "reason": "DOI_MISMATCH",
                "detail": (
                    f"Expected {expected}; "
                    f"observed {observed}."
                ),
                "internal_doi": observed,
                "root_tag": root_tag,
            }


    return {
        "valid": True,
        "reason": None,
        "detail": None,
        "internal_doi": normalize_doi(
            internal_doi
        ),
        "root_tag": root_tag,
    }


# ============================================================
# 3. DATA JATS
# ============================================================

def extract_date_from_element(date_element):

    if date_element is None:
        return None

    year = None
    month = None
    day = None


    for child in list(date_element):

        tag = local_tag(
            child.tag
        )

        value = normalize_whitespace(
            element_text(child)
        )

        if not value:
            continue

        if tag == "year":
            year = value

        elif tag == "month":
            month = value

        elif tag == "day":
            day = value


    if year:

        if month:

            try:
                month = f"{int(month):02d}"
            except Exception:
                pass

        if day:

            try:
                day = f"{int(day):02d}"
            except Exception:
                pass


        if month and day:
            return f"{year}-{month}-{day}"

        if month:
            return f"{year}-{month}"

        return year


    raw = element_text(
        date_element
    )

    return raw or None


# ============================================================
# 4. SUB-ARTICLE DOI / ID
# ============================================================

def extract_subarticle_doi(sub_article):

    front_stub = None

    for child in list(sub_article):

        if local_tag(child.tag) == "front-stub":
            front_stub = child
            break


    search_root = (
        front_stub
        if front_stub is not None
        else sub_article
    )


    for element in search_root.iter():

        if local_tag(element.tag) != "article-id":
            continue

        pub_id_type = (
            element.attrib.get("pub-id-type")
            or
            element.attrib.get("publication-id-type")
            or
            ""
        ).strip().lower()


        if pub_id_type == "doi":

            doi = normalize_doi(
                element_text(element)
            )

            if doi:
                return doi


    for element in search_root.iter():

        if local_tag(element.tag) != "article-id":
            continue

        doi = normalize_doi(
            element_text(element)
        )

        if (
            doi
            and
            doi.startswith("10.")
            and
            "/" in doi
        ):
            return doi


    return None


def extract_subarticle_id(sub_article):

    for attribute in [
        "id",
        "{http://www.w3.org/XML/1998/namespace}id",
    ]:

        value = normalize_whitespace(
            sub_article.attrib.get(attribute)
        )

        if value:
            return value


    doi = extract_subarticle_doi(
        sub_article
    )

    if doi:
        return doi


    return None


# ============================================================
# 5. REVIEWER REPORTS DIRETOS
# ============================================================

def get_direct_reviewer_reports(article_root):

    reports = []


    for child in list(article_root):

        if local_tag(child.tag) != "sub-article":
            continue


        article_type = (
            child.attrib.get("article-type")
            or
            ""
        ).strip().lower()


        normalized_type = (
            article_type
            .replace("_", "-")
            .replace(" ", "-")
        )


        if normalized_type in {
            "reviewer-report",
            "referee-report",
            "peer-review",
        }:

            reports.append(
                child
            )


    return reports


# ============================================================
# 6. CUSTOM META
# ============================================================

def extract_custom_meta_map(element):

    result = {}


    for custom_meta in element.iter():

        if local_tag(custom_meta.tag) != "custom-meta":
            continue


        meta_name = None
        meta_value = None


        for child in list(custom_meta):

            tag = local_tag(
                child.tag
            )


            if tag == "meta-name":

                meta_name = normalize_whitespace(
                    element_text(child)
                )

            elif tag == "meta-value":

                meta_value = normalize_whitespace(
                    element_text(child)
                )


        if meta_name:

            key = meta_name.lower()

            result.setdefault(
                key,
                []
            )

            if meta_value:
                result[key].append(
                    meta_value
                )


    return result


def first_meta_value(
    meta_map,
    candidate_names,
):

    candidates = {
        str(name).lower()
        for name in candidate_names
    }


    for key, values in meta_map.items():

        if key not in candidates:
            continue

        for value in values:

            if value:
                return value


    return None


# ============================================================
# 7. RECOMMENDATION
# ============================================================

def normalize_recommendation(raw_value):

    raw = normalize_whitespace(
        raw_value
    )

    if not raw:
        return None


    normalized = (
        raw.lower()
        .strip()
        .replace("_", "-")
        .replace(" ", "-")
    )


    mapping = {

        "approve":
            "Approved",

        "approved":
            "Approved",

        "approve-with-reservations":
            "Approved with Reservations",

        "approved-with-reservations":
            "Approved with Reservations",

        "approve-with-reservation":
            "Approved with Reservations",

        "approved-with-reservation":
            "Approved with Reservations",

        "reject":
            "Not Approved",

        "rejected":
            "Not Approved",

        "not-approved":
            "Not Approved",
    }


    return mapping.get(
        normalized
    )


# ============================================================
# 8. reviewed_doi — CORRIGIDO EMPIRICAMENTE
# ============================================================

def extract_reviewed_doi(review_element):
    """
    Extrai a versão EXATA do artigo revisado.

    Estrutura empiricamente observada nos XMLs F1000:

        <related-article
            ext-link-type="doi"
            related-article-type="peer-reviewed-article"
            xlink:href="10.12688/f1000research....N"
        />

    O xlink:href é a fonte primária.

    NÃO usa container_doi como fallback.
    NÃO infere pelo título "Reviewer response for version N".
    """

    front_stub = None


    for child in list(review_element):

        if local_tag(child.tag) == "front-stub":

            front_stub = child
            break


    search_root = (
        front_stub
        if front_stub is not None
        else review_element
    )


    # --------------------------------------------------------
    # A. related-article semanticamente explícito
    # --------------------------------------------------------

    for element in search_root.iter():

        if local_tag(element.tag) != "related-article":
            continue


        related_type = (
            element.attrib.get(
                "related-article-type"
            )
            or
            ""
        ).strip().lower()


        ext_link_type = (
            element.attrib.get(
                "ext-link-type"
            )
            or
            ""
        ).strip().lower()


        if (
            related_type
            !=
            "peer-reviewed-article"
        ):
            continue


        href = (
            element.attrib.get(
                XLINK_HREF
            )
            or
            element.attrib.get(
                "href"
            )
        )


        doi = normalize_doi(
            href
        )


        if (
            doi
            and
            doi.startswith("10.")
            and
            "/" in doi
        ):

            return doi


        # Caso raro: related-article semanticamente correto
        # mas DOI no conteúdo textual.

        if ext_link_type == "doi":

            doi = normalize_doi(
                element_text(element)
            )

            if (
                doi
                and
                doi.startswith("10.")
                and
                "/" in doi
            ):
                return doi


    # --------------------------------------------------------
    # B. Fallback estrutural:
    # related-article com ext-link-type=doi e DOI F1000
    #
    # Continua sendo relação explícita; não é inferência
    # pelo container.
    # --------------------------------------------------------

    for element in search_root.iter():

        if local_tag(element.tag) != "related-article":
            continue


        ext_link_type = (
            element.attrib.get(
                "ext-link-type"
            )
            or
            ""
        ).strip().lower()


        if ext_link_type != "doi":
            continue


        href = (
            element.attrib.get(
                XLINK_HREF
            )
            or
            element.attrib.get(
                "href"
            )
        )


        doi = normalize_doi(
            href
        )


        if (
            doi
            and
            doi.startswith(
                "10.12688/f1000research."
            )
        ):

            return doi


    return None


# ============================================================
# 9. REVIEW DATE
# ============================================================

def extract_review_date(review_element):

    front_stub = None


    for child in list(review_element):

        if local_tag(child.tag) == "front-stub":
            front_stub = child
            break


    if front_stub is None:
        return None


    candidates = []


    for element in front_stub.iter():

        tag = local_tag(
            element.tag
        )


        if tag not in {
            "date",
            "pub-date",
            "history-date",
        }:
            continue


        date_type = (
            element.attrib.get("date-type")
            or
            element.attrib.get("pub-type")
            or
            ""
        ).strip().lower()


        candidates.append(
            (
                date_type,
                element,
            )
        )


    preferred_types = {
        "review-received",
        "reviewed",
        "received",
        "epub",
        "pub",
        "publication",
    }


    for date_type, element in candidates:

        if date_type in preferred_types:

            value = extract_date_from_element(
                element
            )

            if value:
                return value


    for _, element in candidates:

        value = extract_date_from_element(
            element
        )

        if value:
            return value


    return None


# ============================================================
# 10. REVIEW BODY
# ============================================================

def extract_review_body_text(review_element):

    for child in list(review_element):

        if local_tag(child.tag) == "body":

            return element_text(
                child
            )


    return ""


# ============================================================
# 11. AUTHOR RESPONSES
# ============================================================

def is_author_response_subarticle(sub_article):

    article_type = (
        sub_article.attrib.get("article-type")
        or
        ""
    ).strip().lower()


    normalized = (
        article_type
        .replace("_", "-")
        .replace(" ", "-")
    )


    if normalized in {
        "author-response",
        "authors-response",
        "author-reply",
        "response",
    }:
        return True


    if (
        "author" in normalized
        and
        (
            "response" in normalized
            or
            "reply" in normalized
        )
    ):
        return True


    return False


def get_author_responses(review_element):

    responses = []


    for element in review_element.iter():

        if element is review_element:
            continue

        if local_tag(element.tag) != "sub-article":
            continue

        if is_author_response_subarticle(
            element
        ):
            responses.append(
                element
            )


    return responses


def extract_author_response_date(response_element):

    front_stub = None


    for child in list(response_element):

        if local_tag(child.tag) == "front-stub":
            front_stub = child
            break


    if front_stub is None:
        return None


    for element in front_stub.iter():

        if local_tag(element.tag) not in {
            "date",
            "pub-date",
            "history-date",
        }:
            continue


        value = extract_date_from_element(
            element
        )

        if value:
            return value


    return None


def extract_author_response_body(response_element):

    for child in list(response_element):

        if local_tag(child.tag) == "body":

            return element_text(
                child
            )


    return ""


# ============================================================
# 12. PEER REVIEW STATE
# ============================================================

def extract_peer_review_state_raw(article_root):

    candidates = []


    for element in article_root.iter():

        tag = local_tag(
            element.tag
        )


        if tag not in {
            "fn-group",
            "fn",
            "custom-meta",
            "meta-value",
        }:
            continue


        content_type = (
            element.attrib.get("content-type")
            or
            ""
        ).strip().lower()


        if content_type in {
            "pub-status",
            "publication-status",
            "peer-review-status",
        }:

            text = element_text(
                element
            )

            if text:
                candidates.append(
                    text
                )


    if not candidates:

        front = None


        for child in list(article_root):

            if local_tag(child.tag) == "front":
                front = child
                break


        if front is not None:

            for element in front.iter():

                text = element_text(
                    element
                )

                if not text:
                    continue


                lower = text.lower()


                if (
                    "peer review" in lower
                    and
                    (
                        "approved" in lower
                        or
                        "not peer reviewed" in lower
                    )
                    and
                    len(text) <= 1000
                ):

                    candidates.append(
                        text
                    )


    if not candidates:
        return None


    candidates = sorted(
        set(candidates),
        key=lambda value: (
            len(value),
            value,
        ),
    )


    return candidates[0]


def interpret_peer_review_state(state_raw):

    if not state_raw:

        return {
            "not_peer_reviewed": 0,
            "approved_count": None,
            "reservations_count": None,
            "not_approved_count": None,
        }


    lower = state_raw.lower()


    if "not peer reviewed" in lower:

        return {
            "not_peer_reviewed": 1,
            "approved_count": 0,
            "reservations_count": 0,
            "not_approved_count": 0,
        }


    def find_count(patterns):

        for pattern in patterns:

            match = re.search(
                pattern,
                state_raw,
                flags=re.IGNORECASE,
            )

            if match:

                try:
                    return int(
                        match.group(1)
                    )

                except Exception:
                    return None


        return None


    reservations_count = find_count(
        [
            r"(\d+)\s+approved\s+with\s+reservations",
            r"approved\s+with\s+reservations\s*[:=]?\s*(\d+)",
        ]
    )


    not_approved_count = find_count(
        [
            r"(\d+)\s+not\s+approved",
            r"not\s+approved\s*[:=]?\s*(\d+)",
        ]
    )


    approved_count = find_count(
        [
            r"(\d+)\s+approved(?!\s+with\s+reservations)",
            r"approved(?!\s+with\s+reservations)\s*[:=]?\s*(\d+)",
        ]
    )


    return {
        "not_peer_reviewed": 0,
        "approved_count": approved_count,
        "reservations_count": reservations_count,
        "not_approved_count": not_approved_count,
    }


def classify_review_structure(
    reports,
    state_raw,
    state_interpretation,
):

    if (
        state_interpretation[
            "not_peer_reviewed"
        ]
        == 1
    ):
        return "NOT_PEER_REVIEWED"


    if reports:
        return "STRUCTURED"


    if state_raw:
        return "REVIEW_UNAVAILABLE"


    return "NO_REVIEW_EVIDENCE"


# ============================================================
# 13. IDs
# ============================================================

def make_fallback_review_id(
    container_doi,
    report_id,
    report_order,
):

    seed = (
        f"{container_doi}|"
        f"{report_id or ''}|"
        f"{report_order}"
    )


    digest = hashlib.sha256(
        seed.encode("utf-8")
    ).hexdigest()[:24]


    return (
        "internal-review:"
        f"{digest}"
    )


def make_response_id(
    review_doi,
    response_order,
    response_text,
):

    seed = (
        f"{review_doi}|"
        f"{response_order}|"
        f"{sha256_text(response_text)}"
    )


    digest = hashlib.sha256(
        seed.encode("utf-8")
    ).hexdigest()[:24]


    return (
        "response_"
        f"{digest}"
    )


def make_state_id(container_doi):

    normalized = normalize_doi(
        container_doi
    )


    digest = hashlib.sha256(
        normalized.encode("utf-8")
    ).hexdigest()[:24]


    return (
        "state_"
        f"{digest}"
    )


# ============================================================
# 14. SQLITE / PRÉ-CONDIÇÕES
# ============================================================

conn = open_builder_db()


container_rows = conn.execute(
    """
    SELECT
        base_doi,
        container_doi,
        container_version,
        discovery_order,
        container_status,
        container_xml_path,
        container_xml_sha256
    FROM base_articles
    ORDER BY discovery_order
    """
).fetchall()


if len(container_rows) != TARGET_BASE_ARTICLES:

    conn.close()

    raise RuntimeError(
        "C5 exige exatamente "
        f"{TARGET_BASE_ARTICLES} BaseArticles; "
        f"encontrados={len(container_rows)}."
    )


not_ready = [
    row
    for row in container_rows
    if row[4] != "READY"
]


if not_ready:

    conn.close()

    raise RuntimeError(
        "Há BaseArticles não READY: "
        f"{not_ready[:10]}"
    )


canonical_pair_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM canonical_pairs
    """
).fetchone()[0]


if canonical_pair_count != 0:

    conn.close()

    raise RuntimeError(
        "canonical_pairs já contém dados. "
        "C5 não reconstruirá Reviews silenciosamente."
    )


# ============================================================
# 15. LIMPEZA CONTROLADA
# ============================================================

try:

    conn.execute(
        "BEGIN IMMEDIATE;"
    )

    conn.execute(
        "DELETE FROM author_responses;"
    )

    conn.execute(
        "DELETE FROM reviews;"
    )

    conn.execute(
        "DELETE FROM peer_review_states;"
    )

    conn.execute(
        """
        UPDATE base_articles
        SET
            review_structure_status = NULL,
            error_message = NULL,
            updated_at = ?
        """,
        (
            utc_now(),
        ),
    )

    conn.commit()


except Exception:

    conn.rollback()
    conn.close()
    raise


# ============================================================
# 16. CONTADORES
# ============================================================

containers_processed = 0

structured_count = 0
review_unavailable_count = 0
not_peer_reviewed_count = 0
no_review_evidence_count = 0

reports_detected = 0

reviews_inserted = 0
responses_inserted = 0
states_inserted = 0

reviews_with_real_doi = 0
reviews_with_internal_id = 0

reports_missing_reviewed_doi = []
reports_invalid_reviewed_version = []

reviews_missing_recommendation = []
reviews_empty_body = []
responses_empty_body = []

reviewed_doi_set = set()

current_reviews = 0
historical_reviews = 0


# ============================================================
# 17. PROCESSAMENTO
# ============================================================

print()
print("=" * 72)
print(
    "EXTRAÇÃO ESTRUTURAL DOS CONTAINERS"
)
print("=" * 72)

print(
    f"Containers............: "
    f"{len(container_rows)}"
)

print(
    f"Extractor version.....: "
    f"{CONTAINER_EXTRACTOR_VERSION}"
)

print()


for index, (
    base_doi,
    container_doi,
    container_version,
    discovery_order,
    container_status,
    xml_path_text,
    expected_xml_sha256,
) in enumerate(
    container_rows,
    start=1,
):

    xml_path = Path(
        xml_path_text
    )


    # --------------------------------------------------------
    # A. XML
    # --------------------------------------------------------

    if not xml_path.exists():

        conn.close()

        raise RuntimeError(
            "Container XML ausente: "
            f"{container_doi}"
        )


    actual_sha256 = sha256_file(
        xml_path
    )


    if (
        expected_xml_sha256
        and
        actual_sha256
        !=
        expected_xml_sha256
    ):

        conn.close()

        raise RuntimeError(
            "Hash divergente: "
            f"{container_doi}"
        )


    xml_bytes = xml_path.read_bytes()


    validation = validate_article_xml_bytes(
        xml_bytes,
        expected_doi=container_doi,
    )


    if not validation["valid"]:

        conn.close()

        raise RuntimeError(
            "Container XML inválido: "
            f"{container_doi} | "
            f"{validation['reason']} | "
            f"{validation['detail']}"
        )


    root = ET.fromstring(
        xml_bytes
    )


    # --------------------------------------------------------
    # B. Reports
    # --------------------------------------------------------

    reports = get_direct_reviewer_reports(
        root
    )


    reports_detected += len(
        reports
    )


    # --------------------------------------------------------
    # C. PeerReviewState
    # --------------------------------------------------------

    state_raw = extract_peer_review_state_raw(
        root
    )


    state_interpretation = interpret_peer_review_state(
        state_raw
    )


    structure_status = classify_review_structure(
        reports=reports,
        state_raw=state_raw,
        state_interpretation=state_interpretation,
    )


    if structure_status == "STRUCTURED":
        structured_count += 1

    elif structure_status == "REVIEW_UNAVAILABLE":
        review_unavailable_count += 1

    elif structure_status == "NOT_PEER_REVIEWED":
        not_peer_reviewed_count += 1

    elif structure_status == "NO_REVIEW_EVIDENCE":
        no_review_evidence_count += 1


    state_id = make_state_id(
        container_doi
    )


    conn.execute(
        """
        INSERT INTO peer_review_states (

            state_id,
            base_doi,
            container_doi,
            article_version,
            state_raw,
            state_raw_sha256,
            approved_count,
            reservations_count,
            not_approved_count,
            not_peer_reviewed,
            status,
            error_message,
            extracted_at

        )
        VALUES (
            ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
            'READY',
            NULL,
            ?
        )
        """,
        (
            state_id,
            base_doi,
            container_doi,
            container_version,
            state_raw,
            (
                sha256_text(state_raw)
                if state_raw
                else None
            ),
            state_interpretation[
                "approved_count"
            ],
            state_interpretation[
                "reservations_count"
            ],
            state_interpretation[
                "not_approved_count"
            ],
            state_interpretation[
                "not_peer_reviewed"
            ],
            utc_now(),
        ),
    )


    states_inserted += 1


    # --------------------------------------------------------
    # D. Reviews
    # --------------------------------------------------------

    for report_order, report in enumerate(
        reports,
        start=1,
    ):

        report_id = extract_subarticle_id(
            report
        )


        real_review_doi = extract_subarticle_doi(
            report
        )


        if real_review_doi:

            review_doi = real_review_doi
            reviews_with_real_doi += 1

        else:

            review_doi = make_fallback_review_id(
                container_doi,
                report_id,
                report_order,
            )

            reviews_with_internal_id += 1


        # ----------------------------------------------------
        # reviewed_doi explícito
        # ----------------------------------------------------

        reviewed_doi = extract_reviewed_doi(
            report
        )


        if not reviewed_doi:

            reports_missing_reviewed_doi.append(
                {
                    "container_doi":
                        container_doi,

                    "review_id":
                        review_doi,

                    "report_id":
                        report_id,

                    "report_order":
                        report_order,
                }
            )

            continue


        reviewed_version = extract_version_from_doi(
            reviewed_doi
        )


        if reviewed_version is None:

            reports_invalid_reviewed_version.append(
                {
                    "container_doi":
                        container_doi,

                    "review_id":
                        review_doi,

                    "reviewed_doi":
                        reviewed_doi,
                }
            )

            continue


        reviewed_base_doi = derive_base_doi(
            reviewed_doi
        )


        if reviewed_base_doi != base_doi:

            conn.close()

            raise RuntimeError(
                "reviewed_doi pertence a outro base_doi: "
                f"container={container_doi}, "
                f"review={review_doi}, "
                f"reviewed_doi={reviewed_doi}, "
                f"expected_base={base_doi}, "
                f"observed_base={reviewed_base_doi}"
            )


        reviewed_doi_set.add(
            normalize_doi(
                reviewed_doi
            )
        )


        # ----------------------------------------------------
        # Recommendation
        # ----------------------------------------------------

        meta_map = extract_custom_meta_map(
            report
        )


        recommendation_raw = first_meta_value(
            meta_map,
            {
                "recommendation",
                "recommendation-status",
                "recommendation_status",
                "review-status",
                "review_status",
            },
        )


        recommendation_label = normalize_recommendation(
            recommendation_raw
        )


        if recommendation_label is None:

            reviews_missing_recommendation.append(
                {
                    "container_doi":
                        container_doi,

                    "review_id":
                        review_doi,

                    "raw":
                        recommendation_raw,
                }
            )


        # ----------------------------------------------------
        # Review text/date
        # ----------------------------------------------------

        review_date = extract_review_date(
            report
        )


        review_text = extract_review_body_text(
            report
        )


        if not review_text:

            reviews_empty_body.append(
                {
                    "container_doi":
                        container_doi,

                    "review_id":
                        review_doi,
                }
            )


        # ----------------------------------------------------
        # CURRENT / HISTORICAL
        # ----------------------------------------------------

        if (
            normalize_doi(reviewed_doi)
            ==
            normalize_doi(container_doi)
        ):

            relation_to_container = "CURRENT"
            current_reviews += 1

        else:

            relation_to_container = "HISTORICAL"
            historical_reviews += 1


        # ----------------------------------------------------
        # Persistir Review
        # ----------------------------------------------------

        conn.execute(
            """
            INSERT INTO reviews (

                review_doi,
                base_doi,
                source_container_doi,
                report_id,
                reviewed_doi,
                reviewed_version,
                review_date,
                recommendation_raw,
                recommendation_label,
                review_text,
                review_text_sha256,
                review_character_count,
                review_word_count,
                relation_to_container,
                status,
                error_message,
                extracted_at

            )
            VALUES (
                ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
                'READY',
                NULL,
                ?
            )
            """,
            (
                review_doi,
                base_doi,
                container_doi,
                (
                    report_id
                    or
                    f"report-{report_order}"
                ),
                reviewed_doi,
                reviewed_version,
                review_date,
                recommendation_raw,
                recommendation_label,
                review_text,
                (
                    sha256_text(review_text)
                    if review_text
                    else None
                ),
                len(review_text),
                len(review_text.split()),
                relation_to_container,
                utc_now(),
            ),
        )


        reviews_inserted += 1


        # ----------------------------------------------------
        # E. AuthorResponses
        # ----------------------------------------------------

        responses = get_author_responses(
            report
        )


        for response_order, response in enumerate(
            responses,
            start=1,
        ):

            response_date = (
                extract_author_response_date(
                    response
                )
            )


            response_text = (
                extract_author_response_body(
                    response
                )
            )


            if not response_text:

                responses_empty_body.append(
                    {
                        "review_id":
                            review_doi,

                        "response_order":
                            response_order,
                    }
                )


            response_id = make_response_id(
                review_doi,
                response_order,
                response_text,
            )


            conn.execute(
                """
                INSERT INTO author_responses (

                    response_id,
                    review_doi,
                    base_doi,
                    source_container_doi,
                    response_order,
                    response_date,
                    response_text,
                    response_text_sha256,
                    response_character_count,
                    response_word_count,
                    status,
                    error_message,
                    extracted_at

                )
                VALUES (
                    ?, ?, ?, ?, ?, ?, ?, ?, ?, ?,
                    'READY',
                    NULL,
                    ?
                )
                """,
                (
                    response_id,
                    review_doi,
                    base_doi,
                    container_doi,
                    response_order,
                    response_date,
                    response_text,
                    (
                        sha256_text(response_text)
                        if response_text
                        else None
                    ),
                    len(response_text),
                    len(response_text.split()),
                    utc_now(),
                ),
            )


            responses_inserted += 1


    # --------------------------------------------------------
    # F. Classificação do BaseArticle
    # --------------------------------------------------------

    conn.execute(
        """
        UPDATE base_articles
        SET
            review_structure_status = ?,
            error_message = NULL,
            updated_at = ?
        WHERE base_doi = ?
        """,
        (
            structure_status,
            utc_now(),
            base_doi,
        ),
    )


    conn.commit()

    containers_processed += 1


    if (
        index == 1
        or
        index % 25 == 0
        or
        index == len(container_rows)
    ):

        print(
            f"[{index:>3}/{len(container_rows)}] "
            f"structured={structured_count} | "
            f"reports={reports_detected} | "
            f"reviews={reviews_inserted} | "
            f"responses={responses_inserted}"
        )


# ============================================================
# 18. CONTAGENS SQLITE
# ============================================================

db_review_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM reviews
    """
).fetchone()[0]


db_response_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM author_responses
    """
).fetchone()[0]


db_state_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM peer_review_states
    """
).fetchone()[0]


classified_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM base_articles
    WHERE review_structure_status IS NOT NULL
    """
).fetchone()[0]


# ============================================================
# 19. VALIDAÇÕES
# ============================================================

if db_review_count != reviews_inserted:

    conn.close()

    raise RuntimeError(
        "Contagem de Reviews divergiu: "
        f"memory={reviews_inserted}, "
        f"db={db_review_count}"
    )


if db_response_count != responses_inserted:

    conn.close()

    raise RuntimeError(
        "Contagem de AuthorResponses divergiu: "
        f"memory={responses_inserted}, "
        f"db={db_response_count}"
    )


if db_state_count != TARGET_BASE_ARTICLES:

    conn.close()

    raise RuntimeError(
        "PeerReviewStates != containers: "
        f"{db_state_count} != "
        f"{TARGET_BASE_ARTICLES}"
    )


if classified_count != TARGET_BASE_ARTICLES:

    conn.close()

    raise RuntimeError(
        "Nem todos os containers foram classificados."
    )


# ============================================================
# 20. EXACT REVIEWED DOI
# ============================================================

container_doi_set = {
    normalize_doi(
        row[1]
    )
    for row in container_rows
}


reviewed_dois_sorted = sorted(
    reviewed_doi_set
)


reviewed_dois_already_containers = [
    doi
    for doi in reviewed_dois_sorted
    if doi in container_doi_set
]


reviewed_dois_missing_exact_xml = [
    doi
    for doi in reviewed_dois_sorted
    if doi not in container_doi_set
]


# ============================================================
# 21. DISTRIBUIÇÕES
# ============================================================

structure_distribution = conn.execute(
    """
    SELECT
        review_structure_status,
        COUNT(*)
    FROM base_articles
    GROUP BY review_structure_status
    ORDER BY review_structure_status
    """
).fetchall()


relation_distribution = conn.execute(
    """
    SELECT
        relation_to_container,
        COUNT(*)
    FROM reviews
    GROUP BY relation_to_container
    ORDER BY relation_to_container
    """
).fetchall()


recommendation_distribution = conn.execute(
    """
    SELECT
        COALESCE(
            recommendation_label,
            'MISSING_OR_UNKNOWN'
        ),
        COUNT(*)
    FROM reviews
    GROUP BY recommendation_label
    ORDER BY recommendation_label
    """
).fetchall()


reviews_without_text_db = conn.execute(
    """
    SELECT COUNT(*)
    FROM reviews
    WHERE
        review_text IS NULL
        OR TRIM(review_text) = ''
    """
).fetchone()[0]


reviews_without_recommendation_db = conn.execute(
    """
    SELECT COUNT(*)
    FROM reviews
    WHERE recommendation_label IS NULL
    """
).fetchone()[0]


duplicate_report_keys = conn.execute(
    """
    SELECT COUNT(*)
    FROM (
        SELECT
            source_container_doi,
            report_id,
            COUNT(*) AS n
        FROM reviews
        GROUP BY
            source_container_doi,
            report_id
        HAVING COUNT(*) > 1
    )
    """
).fetchone()[0]


# ============================================================
# 22. SQLITE INTEGRITY
# ============================================================

integrity = [
    row[0]
    for row in conn.execute(
        "PRAGMA integrity_check;"
    ).fetchall()
]


foreign_key_errors = conn.execute(
    "PRAGMA foreign_key_check;"
).fetchall()


if integrity != ["ok"]:

    conn.close()

    raise RuntimeError(
        "SQLite integrity_check falhou: "
        f"{integrity}"
    )


if foreign_key_errors:

    conn.close()

    raise RuntimeError(
        "SQLite foreign_key_check falhou: "
        f"{foreign_key_errors[:20]}"
    )


if duplicate_report_keys != 0:

    conn.close()

    raise RuntimeError(
        "Duplicate report keys: "
        f"{duplicate_report_keys}"
    )


# ============================================================
# 23. RUN INFO
# ============================================================

now = utc_now()


run_info_updates = {

    "container_extractor_version":
        CONTAINER_EXTRACTOR_VERSION,

    "container_extraction_status":
        "PASS",

    "reviewed_doi_source":
        (
            "reviewer-report/front-stub/"
            "related-article[@related-article-type="
            "'peer-reviewed-article']/@xlink:href"
        ),

    "containers_extracted":
        str(containers_processed),

    "structured_containers":
        str(structured_count),

    "review_unavailable_containers":
        str(review_unavailable_count),

    "not_peer_reviewed_containers":
        str(not_peer_reviewed_count),

    "no_review_evidence_containers":
        str(no_review_evidence_count),

    "structured_reports_detected":
        str(reports_detected),

    "reviews_extracted":
        str(db_review_count),

    "author_responses_extracted":
        str(db_response_count),

    "peer_review_states_extracted":
        str(db_state_count),

    "exact_reviewed_doi_count":
        str(
            len(reviewed_dois_sorted)
        ),

    "exact_reviewed_doi_missing_xml_count":
        str(
            len(
                reviewed_dois_missing_exact_xml
            )
        ),

    "container_extraction_completed_at":
        now,
}


for key, value in run_info_updates.items():

    conn.execute(
        """
        INSERT INTO run_info (
            key,
            value,
            updated_at
        )
        VALUES (?, ?, ?)

        ON CONFLICT(key)
        DO UPDATE SET
            value = excluded.value,
            updated_at = excluded.updated_at
        """,
        (
            key,
            value,
            now,
        ),
    )


conn.commit()


# ============================================================
# 24. FECHAR
# ============================================================

conn.close()


# ============================================================
# 25. LOG
# ============================================================

log(
    "Extração estrutural concluída: "
    f"containers={containers_processed}, "
    f"reports={reports_detected}, "
    f"reviews={db_review_count}, "
    f"responses={db_response_count}, "
    f"states={db_state_count}, "
    f"exact_reviewed_doi="
    f"{len(reviewed_dois_sorted)}, "
    f"missing_exact_xml="
    f"{len(reviewed_dois_missing_exact_xml)}."
)


# ============================================================
# 26. RELATÓRIO
# ============================================================

print()
print("=" * 72)
print(
    "F1000 CORPUS BUILDER 2.0 — "
    "CONTAINER EXTRACTION"
)
print("=" * 72)


print(
    f"Containers processed...: "
    f"{containers_processed}"
)

print(
    f"PeerReviewStates........: "
    f"{db_state_count}"
)

print(
    f"Structured reports......: "
    f"{reports_detected}"
)

print(
    f"Canonical Reviews.......: "
    f"{db_review_count}"
)

print(
    f"AuthorResponses.........: "
    f"{db_response_count}"
)


print()
print("Review structure classification:")

for status, count in structure_distribution:

    print(
        f"  {status:<22}: "
        f"{count}"
    )


print()
print("Review relation to container:")

if relation_distribution:

    for relation, count in relation_distribution:

        print(
            f"  {relation:<22}: "
            f"{count}"
        )

else:

    print(
        "  No canonical reviews."
    )


print()
print("Recommendation distribution:")

if recommendation_distribution:

    for recommendation, count in recommendation_distribution:

        print(
            f"  {recommendation:<30}: "
            f"{count}"
        )

else:

    print(
        "  No canonical reviews."
    )


print()
print("Review identifiers:")

print(
    f"  Real review DOI.......: "
    f"{reviews_with_real_doi}"
)

print(
    f"  Internal fallback ID..: "
    f"{reviews_with_internal_id}"
)


print()
print("Extraction anomalies:")

print(
    f"  Reports missing reviewed_doi: "
    f"{len(reports_missing_reviewed_doi)}"
)

print(
    f"  Invalid reviewed version....: "
    f"{len(reports_invalid_reviewed_version)}"
)

print(
    f"  Missing recommendation......: "
    f"{reviews_without_recommendation_db}"
)

print(
    f"  Empty review body...........: "
    f"{reviews_without_text_db}"
)

print(
    f"  Empty response body.........: "
    f"{len(responses_empty_body)}"
)


print()
print("Exact reviewed versions:")

print(
    f"  Unique reviewed_doi...: "
    f"{len(reviewed_dois_sorted)}"
)

print(
    f"  Already containers....: "
    f"{len(reviewed_dois_already_containers)}"
)

print(
    f"  Missing exact XML.....: "
    f"{len(reviewed_dois_missing_exact_xml)}"
)


if reviewed_dois_missing_exact_xml:

    print()
    print("First missing exact versions:")

    for doi in reviewed_dois_missing_exact_xml[:20]:

        print(
            f"  {doi}"
        )


if reports_missing_reviewed_doi:

    print()
    print("First reports without reviewed_doi:")

    for item in reports_missing_reviewed_doi[:10]:

        print(
            f"  {item}"
        )


if reviews_missing_recommendation:

    print()
    print(
        "First reviews without normalized recommendation:"
    )

    for item in reviews_missing_recommendation[:10]:

        print(
            f"  {item}"
        )


print()
print(
    f"SQLite integrity.......: "
    f"{integrity}"
)

print(
    f"Foreign-key errors.....: "
    f"{len(foreign_key_errors)}"
)

print(
    f"Duplicate report keys..: "
    f"{duplicate_report_keys}"
)


print()
print(
    "HTTP requests performed: 0"
)

print(
    "Historical XML downloads: 0"
)


print()
print(
    "STATUS.................: PASS"
)

print("=" * 72)
print(
    "CÉLULA 5 CONCLUÍDA"
)
print("=" * 72)


EXTRAÇÃO ESTRUTURAL DOS CONTAINERS
Containers............: 500
Extractor version.....: 2.0.2

[  1/500] structured=1 | reports=2 | reviews=2 | responses=0
[ 25/500] structured=21 | reports=57 | reviews=57 | responses=8
[ 50/500] structured=42 | reports=103 | reviews=103 | responses=19
[ 75/500] structured=64 | reports=182 | reviews=182 | responses=57
[100/500] structured=88 | reports=284 | reviews=284 | responses=105
[125/500] structured=108 | reports=338 | reviews=338 | responses=126
[150/500] structured=129 | reports=399 | reviews=399 | responses=154
[175/500] structured=145 | reports=450 | reviews=450 | responses=172
[200/500] structured=165 | reports=519 | reviews=519 | responses=205
[225/500] structured=188 | reports=582 | reviews=582 | responses=233
[250/500] structured=210 | reports=643 | reviews=643 | responses=260
[275/500] structured=232 | reports=688 | reviews=688 | responses=267
[300/500] structured=254 | reports=767 | reviews=767 | responses=302
[325/500] structured=275 |

In [ ]:
# ============================================================
# CÉLULA 6 — AQUISIÇÃO DAS EXACT REVIEWED VERSIONS
# F1000 Corpus Builder 2.0
# ============================================================
#
# RESPONSABILIDADE
#
# Garantir disponibilidade local de TODOS os ArticleVersion XML
# referenciados explicitamente pelos 1.131 Reviews.
#
# Fontes possíveis:
#
#   1. CONTAINER
#      O reviewed_doi já é um dos 500 Container XML.
#
#   2. REVIEWED_VERSION
#      O reviewed_doi não é um container da amostra e precisa
#      ser adquirido pelo endpoint Article XML.
#
# ESTA CÉLULA:
#
#   - identifica todos os reviewed_doi distintos;
#   - reutiliza Container XML quando DOI exato coincide;
#   - reutiliza reviewed_versions já baixadas;
#   - baixa somente versões ausentes;
#   - valida DOI interno;
#   - grava hash e tamanho;
#   - registra ArticleVersion em estado PENDING.
#
# NÃO FAZ:
#
#   - parsing textual;
#   - CanonicalPairs;
#   - KnowledgeUnits;
#   - Markdown;
#   - curadoria.
#
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import time
import hashlib
import sqlite3
import xml.etree.ElementTree as ET

from pathlib import Path

import requests


EXACT_VERSION_ACQUISITION_VERSION = "2.0.0"


# ============================================================
# 2. HELPERS XML AUTOSSUFICIENTES
# ============================================================

def local_tag(tag):

    if tag is None:
        return ""

    return str(tag).split("}")[-1].lower()


def element_text(element):

    if element is None:
        return ""

    parts = []

    for text in element.itertext():

        cleaned = normalize_whitespace(
            text
        )

        if cleaned:
            parts.append(
                cleaned
            )

    return normalize_whitespace(
        " ".join(parts)
    )


def find_article_doi_in_xml(root):

    front = None


    for child in list(root):

        if local_tag(child.tag) == "front":

            front = child
            break


    if front is None:
        return None


    article_meta = None


    for child in list(front):

        if local_tag(child.tag) == "article-meta":

            article_meta = child
            break


    if article_meta is None:
        return None


    for element in article_meta.iter():

        if local_tag(element.tag) != "article-id":
            continue


        pub_id_type = (
            element.attrib.get(
                "pub-id-type"
            )
            or
            element.attrib.get(
                "publication-id-type"
            )
            or
            ""
        ).strip().lower()


        if pub_id_type == "doi":

            doi = normalize_doi(
                element_text(element)
            )

            if doi:
                return doi


    for element in article_meta.iter():

        if local_tag(element.tag) != "article-id":
            continue


        doi = normalize_doi(
            element_text(element)
        )


        if (
            doi
            and
            doi.startswith("10.")
            and
            "/" in doi
        ):

            return doi


    return None


def validate_article_xml_bytes(
    xml_bytes,
    expected_doi=None,
):

    if not xml_bytes:

        return {
            "valid": False,
            "reason": "EMPTY_XML",
            "detail": "XML contains zero bytes.",
            "internal_doi": None,
        }


    try:

        root = ET.fromstring(
            xml_bytes
        )

    except ET.ParseError as exc:

        return {
            "valid": False,
            "reason": "XML_PARSE_ERROR",
            "detail": str(exc),
            "internal_doi": None,
        }


    if local_tag(root.tag) != "article":

        return {
            "valid": False,
            "reason": "INVALID_ROOT",
            "detail": (
                "Expected <article>; "
                f"found <{local_tag(root.tag)}>."
            ),
            "internal_doi": None,
        }


    internal_doi = find_article_doi_in_xml(
        root
    )


    if not internal_doi:

        return {
            "valid": False,
            "reason": "ARTICLE_DOI_MISSING",
            "detail": (
                "Main article DOI not found."
            ),
            "internal_doi": None,
        }


    internal_doi = normalize_doi(
        internal_doi
    )


    if expected_doi:

        expected_doi = normalize_doi(
            expected_doi
        )


        if internal_doi != expected_doi:

            return {
                "valid": False,
                "reason": "DOI_MISMATCH",
                "detail": (
                    f"expected={expected_doi}; "
                    f"observed={internal_doi}"
                ),
                "internal_doi": internal_doi,
            }


    return {
        "valid": True,
        "reason": None,
        "detail": None,
        "internal_doi": internal_doi,
    }


# ============================================================
# 3. HTTP
# ============================================================

session = requests.Session()


session.headers.update(
    {
        "User-Agent": USER_AGENT,
        "Accept": (
            "application/xml,"
            "text/xml,"
            "application/octet-stream,"
            "*/*"
        ),
    }
)


http_requests_this_run = 0


def fetch_article_xml(
    doi,
):
    """
    Download conservador e sequencial.

    Política:
      - retry em falhas transitórias;
      - 404 é falha persistente;
      - nenhum DOI alternativo é inferido.
    """

    global http_requests_this_run


    normalized_doi = normalize_doi(
        doi
    )


    last_error = None


    for attempt in range(
        1,
        HTTP_MAX_RETRIES + 1,
    ):

        if http_requests_this_run > 0:

            time.sleep(
                HTTP_DELAY_SECONDS
            )


        try:

            response = session.get(
                ARTICLE_XML_ENDPOINT,
                params={
                    "doi": normalized_doi,
                },
                timeout=HTTP_TIMEOUT_SECONDS,
            )


            http_requests_this_run += 1


        except requests.RequestException as exc:

            last_error = (
                "REQUEST_EXCEPTION",
                None,
                str(exc),
            )


            if attempt < HTTP_MAX_RETRIES:
                continue


            return {
                "ok": False,
                "transient": True,
                "reason": last_error[0],
                "http_status": None,
                "detail": last_error[2],
                "content": None,
            }


        status = response.status_code


        # ----------------------------------------------------
        # SUCCESS
        # ----------------------------------------------------

        if status == 200:

            content = response.content


            validation = validate_article_xml_bytes(
                content,
                expected_doi=normalized_doi,
            )


            if validation["valid"]:

                return {
                    "ok": True,
                    "transient": False,
                    "reason": None,
                    "http_status": status,
                    "detail": None,
                    "content": content,
                }


            return {
                "ok": False,
                "transient": False,
                "reason": validation[
                    "reason"
                ],
                "http_status": status,
                "detail": validation[
                    "detail"
                ],
                "content": None,
            }


        # ----------------------------------------------------
        # PERSISTENT NOT FOUND
        # ----------------------------------------------------

        if status == 404:

            return {
                "ok": False,
                "transient": False,
                "reason": "HTTP_404",
                "http_status": status,
                "detail": (
                    "Exact reviewed version "
                    "not found at Article XML endpoint."
                ),
                "content": None,
            }


        # ----------------------------------------------------
        # TRANSIENT HTTP
        # ----------------------------------------------------

        if (
            status == 408
            or
            status == 425
            or
            status == 429
            or
            500 <= status <= 599
        ):

            last_error = (
                f"HTTP_{status}",
                status,
                response.text[:500],
            )


            if attempt < HTTP_MAX_RETRIES:
                continue


            return {
                "ok": False,
                "transient": True,
                "reason": last_error[0],
                "http_status": status,
                "detail": last_error[2],
                "content": None,
            }


        # ----------------------------------------------------
        # OTHER HTTP ERROR
        # ----------------------------------------------------

        return {
            "ok": False,
            "transient": False,
            "reason": f"HTTP_{status}",
            "http_status": status,
            "detail": response.text[:500],
            "content": None,
        }


    return {
        "ok": False,
        "transient": True,
        "reason": "UNKNOWN_HTTP_FAILURE",
        "http_status": None,
        "detail": str(last_error),
        "content": None,
    }


# ============================================================
# 4. SQLITE / PRÉ-CONDIÇÕES
# ============================================================

conn = open_builder_db()


base_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM base_articles
    """
).fetchone()[0]


ready_base_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM base_articles
    WHERE container_status = 'READY'
    """
).fetchone()[0]


review_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM reviews
    WHERE status = 'READY'
    """
).fetchone()[0]


if (
    base_count
    !=
    TARGET_BASE_ARTICLES
):

    conn.close()

    raise RuntimeError(
        "BaseArticle cardinality inválida: "
        f"{base_count}"
    )


if (
    ready_base_count
    !=
    TARGET_BASE_ARTICLES
):

    conn.close()

    raise RuntimeError(
        "Nem todos os BaseArticles estão READY."
    )


if review_count == 0:

    conn.close()

    raise RuntimeError(
        "Nenhuma Review disponível. "
        "C5 deve ser concluída antes da C6."
    )


# ============================================================
# 5. MAPA DOS CONTAINERS
# ============================================================

container_rows = conn.execute(
    """
    SELECT
        base_doi,
        container_doi,
        container_version,
        container_xml_path,
        container_xml_sha256,
        container_xml_bytes,
        container_downloaded_at
    FROM base_articles
    WHERE container_status = 'READY'
    """
).fetchall()


container_map = {

    normalize_doi(
        row[1]
    ): {
        "base_doi":
            row[0],

        "container_doi":
            normalize_doi(
                row[1]
            ),

        "container_version":
            row[2],

        "xml_path":
            row[3],

        "xml_sha256":
            row[4],

        "xml_bytes":
            row[5],

        "downloaded_at":
            row[6],
    }

    for row in container_rows
}


# ============================================================
# 6. TODOS OS EXACT reviewed_doi
# ============================================================

reviewed_rows = conn.execute(
    """
    SELECT DISTINCT
        base_doi,
        reviewed_doi,
        reviewed_version
    FROM reviews
    WHERE
        status = 'READY'
        AND reviewed_doi IS NOT NULL
    ORDER BY
        base_doi,
        reviewed_version,
        reviewed_doi
    """
).fetchall()


if not reviewed_rows:

    conn.close()

    raise RuntimeError(
        "Nenhum reviewed_doi encontrado."
    )


# ============================================================
# 7. VALIDAR RELAÇÃO DOI / VERSION
# ============================================================

for (
    base_doi,
    reviewed_doi,
    reviewed_version,
) in reviewed_rows:

    normalized_reviewed_doi = normalize_doi(
        reviewed_doi
    )


    derived_base = derive_base_doi(
        normalized_reviewed_doi
    )


    derived_version = extract_version_from_doi(
        normalized_reviewed_doi
    )


    if derived_base != base_doi:

        conn.close()

        raise RuntimeError(
            "Base DOI inconsistente: "
            f"{reviewed_doi}"
        )


    if derived_version != reviewed_version:

        conn.close()

        raise RuntimeError(
            "Version inconsistente: "
            f"{reviewed_doi} | "
            f"stored={reviewed_version} | "
            f"derived={derived_version}"
        )


# ============================================================
# 8. CLASSIFICAR FONTES
# ============================================================

container_reviewed_rows = []

external_reviewed_rows = []


for row in reviewed_rows:

    (
        base_doi,
        reviewed_doi,
        reviewed_version,
    ) = row


    normalized_reviewed_doi = normalize_doi(
        reviewed_doi
    )


    if normalized_reviewed_doi in container_map:

        container_reviewed_rows.append(
            row
        )

    else:

        external_reviewed_rows.append(
            row
        )


# ============================================================
# 9. RELATÓRIO INICIAL
# ============================================================

print()
print("=" * 72)
print(
    "AQUISIÇÃO DAS EXACT REVIEWED VERSIONS"
)
print("=" * 72)

print(
    f"Canonical Reviews......: "
    f"{review_count}"
)

print(
    f"Unique reviewed_doi....: "
    f"{len(reviewed_rows)}"
)

print(
    f"Available as containers: "
    f"{len(container_reviewed_rows)}"
)

print(
    f"Need reviewed XML......: "
    f"{len(external_reviewed_rows)}"
)

print(
    f"Acquisition version....: "
    f"{EXACT_VERSION_ACQUISITION_VERSION}"
)

print()


# ============================================================
# 10. REGISTRAR VERSÕES QUE JÁ SÃO CONTAINERS
# ============================================================

container_versions_registered = 0


for (
    base_doi,
    reviewed_doi,
    reviewed_version,
) in container_reviewed_rows:

    doi = normalize_doi(
        reviewed_doi
    )


    source = container_map[
        doi
    ]


    xml_path = Path(
        source["xml_path"]
    )


    if not xml_path.exists():

        conn.close()

        raise RuntimeError(
            "Container XML referenciado "
            f"não existe: {doi}"
        )


    actual_hash = sha256_file(
        xml_path
    )


    if (
        source["xml_sha256"]
        and
        actual_hash
        !=
        source["xml_sha256"]
    ):

        conn.close()

        raise RuntimeError(
            "Hash do Container XML divergiu: "
            f"{doi}"
        )


    xml_bytes = xml_path.read_bytes()


    validation = validate_article_xml_bytes(
        xml_bytes,
        expected_doi=doi,
    )


    if not validation["valid"]:

        conn.close()

        raise RuntimeError(
            "Container XML inválido: "
            f"{doi} | "
            f"{validation['reason']}"
        )


    now = utc_now()


    conn.execute(
        """
        INSERT INTO article_versions (

            article_doi,
            base_doi,
            article_version,
            source_kind,
            xml_path,
            xml_sha256,
            xml_bytes,
            parse_status,
            error_message,
            collected_at,
            created_at,
            updated_at

        )
        VALUES (
            ?, ?, ?, 'CONTAINER',
            ?, ?, ?,
            'PENDING',
            NULL,
            ?,
            ?,
            ?
        )

        ON CONFLICT(article_doi)
        DO UPDATE SET

            base_doi =
                excluded.base_doi,

            article_version =
                excluded.article_version,

            source_kind =
                excluded.source_kind,

            xml_path =
                excluded.xml_path,

            xml_sha256 =
                excluded.xml_sha256,

            xml_bytes =
                excluded.xml_bytes,

            error_message =
                NULL,

            collected_at =
                COALESCE(
                    article_versions.collected_at,
                    excluded.collected_at
                ),

            updated_at =
                excluded.updated_at
        """,
        (
            doi,
            base_doi,
            reviewed_version,
            str(xml_path),
            actual_hash,
            len(xml_bytes),
            (
                source[
                    "downloaded_at"
                ]
                or
                now
            ),
            now,
            now,
        ),
    )


    container_versions_registered += 1


conn.commit()


# ============================================================
# 11. ADQUIRIR / REUTILIZAR REVIEWED VERSIONS
# ============================================================

reviewed_downloaded = 0
reviewed_reused = 0

persistent_failures = []
transient_failures = []


for index, (
    base_doi,
    reviewed_doi,
    reviewed_version,
) in enumerate(
    external_reviewed_rows,
    start=1,
):

    doi = normalize_doi(
        reviewed_doi
    )


    filename = (
        safe_filename(doi)
        + ".xml"
    )


    target_path = (
        RAW_REVIEWED_VERSIONS_DIR
        /
        filename
    )


    reused = False


    # --------------------------------------------------------
    # A. REUSE LOCAL FILE
    # --------------------------------------------------------

    if target_path.exists():

        existing_bytes = (
            target_path.read_bytes()
        )


        validation = validate_article_xml_bytes(
            existing_bytes,
            expected_doi=doi,
        )


        if validation["valid"]:

            xml_bytes = existing_bytes

            reused = True

            reviewed_reused += 1


        else:

            # Arquivo local inválido não será aceito.
            # Remove apenas este cache derivado.
            target_path.unlink()


    # --------------------------------------------------------
    # B. DOWNLOAD
    # --------------------------------------------------------

    if not reused:

        result = fetch_article_xml(
            doi
        )


        if not result["ok"]:

            failure = {
                "doi":
                    doi,

                "base_doi":
                    base_doi,

                "version":
                    reviewed_version,

                "reason":
                    result["reason"],

                "http_status":
                    result[
                        "http_status"
                    ],

                "detail":
                    result["detail"],
            }


            if result["transient"]:

                transient_failures.append(
                    failure
                )

                conn.close()

                raise RuntimeError(
                    "Falha HTTP transitória durante "
                    "aquisição de exact reviewed version. "
                    "Execução interrompida sem inferência "
                    "ou exclusão. "
                    f"{failure}"
                )


            persistent_failures.append(
                failure
            )


            print(
                f"[{index:>3}/"
                f"{len(external_reviewed_rows)}] "
                f"FAILED "
                f"{doi} | "
                f"{result['reason']}"
            )


            continue


        xml_bytes = result[
            "content"
        ]


        target_path.write_bytes(
            xml_bytes
        )


        reviewed_downloaded += 1


    # --------------------------------------------------------
    # C. HASH / VALIDATION
    # --------------------------------------------------------

    validation = validate_article_xml_bytes(
        xml_bytes,
        expected_doi=doi,
    )


    if not validation["valid"]:

        conn.close()

        raise RuntimeError(
            "Exact reviewed XML inválido "
            "após aquisição: "
            f"{doi} | "
            f"{validation}"
        )


    xml_hash = sha256_bytes(
        xml_bytes
    )


    now = utc_now()


    # --------------------------------------------------------
    # D. REGISTRAR ArticleVersion
    # --------------------------------------------------------

    conn.execute(
        """
        INSERT INTO article_versions (

            article_doi,
            base_doi,
            article_version,
            source_kind,
            xml_path,
            xml_sha256,
            xml_bytes,
            parse_status,
            error_message,
            collected_at,
            created_at,
            updated_at

        )
        VALUES (
            ?, ?, ?, 'REVIEWED_VERSION',
            ?, ?, ?,
            'PENDING',
            NULL,
            ?,
            ?,
            ?
        )

        ON CONFLICT(article_doi)
        DO UPDATE SET

            base_doi =
                excluded.base_doi,

            article_version =
                excluded.article_version,

            source_kind =
                excluded.source_kind,

            xml_path =
                excluded.xml_path,

            xml_sha256 =
                excluded.xml_sha256,

            xml_bytes =
                excluded.xml_bytes,

            error_message =
                NULL,

            collected_at =
                COALESCE(
                    article_versions.collected_at,
                    excluded.collected_at
                ),

            updated_at =
                excluded.updated_at
        """,
        (
            doi,
            base_doi,
            reviewed_version,
            str(target_path),
            xml_hash,
            len(xml_bytes),
            now,
            now,
            now,
        ),
    )


    conn.commit()


    # --------------------------------------------------------
    # E. PROGRESS
    # --------------------------------------------------------

    if (
        index == 1
        or
        index % 25 == 0
        or
        index
        ==
        len(external_reviewed_rows)
    ):

        print(
            f"[{index:>3}/"
            f"{len(external_reviewed_rows)}] "
            f"reused={reviewed_reused} | "
            f"downloaded={reviewed_downloaded} | "
            f"failed={len(persistent_failures)}"
        )


# ============================================================
# 12. VALIDAR COBERTURA FINAL
# ============================================================

registered_rows = conn.execute(
    """
    SELECT
        article_doi,
        base_doi,
        article_version,
        source_kind,
        xml_path,
        xml_sha256,
        xml_bytes,
        parse_status
    FROM article_versions
    ORDER BY
        base_doi,
        article_version,
        article_doi
    """
).fetchall()


registered_map = {

    normalize_doi(
        row[0]
    ): row

    for row in registered_rows
}


missing_registry = []

physical_errors = []


for (
    base_doi,
    reviewed_doi,
    reviewed_version,
) in reviewed_rows:

    doi = normalize_doi(
        reviewed_doi
    )


    if doi not in registered_map:

        missing_registry.append(
            doi
        )

        continue


    row = registered_map[
        doi
    ]


    (
        article_doi,
        stored_base_doi,
        stored_version,
        source_kind,
        xml_path_text,
        stored_hash,
        stored_bytes,
        parse_status,
    ) = row


    if stored_base_doi != base_doi:

        physical_errors.append(
            (
                doi,
                "BASE_DOI_MISMATCH",
            )
        )

        continue


    if stored_version != reviewed_version:

        physical_errors.append(
            (
                doi,
                "VERSION_MISMATCH",
            )
        )

        continue


    xml_path = Path(
        xml_path_text
    )


    if not xml_path.exists():

        physical_errors.append(
            (
                doi,
                "FILE_MISSING",
            )
        )

        continue


    xml_bytes = xml_path.read_bytes()


    if len(xml_bytes) != stored_bytes:

        physical_errors.append(
            (
                doi,
                "BYTE_COUNT_MISMATCH",
            )
        )

        continue


    actual_hash = sha256_bytes(
        xml_bytes
    )


    if actual_hash != stored_hash:

        physical_errors.append(
            (
                doi,
                "HASH_MISMATCH",
            )
        )

        continue


    validation = validate_article_xml_bytes(
        xml_bytes,
        expected_doi=doi,
    )


    if not validation["valid"]:

        physical_errors.append(
            (
                doi,
                validation["reason"],
            )
        )


# ============================================================
# 13. COBERTURA REVIEW → ARTICLE VERSION
# ============================================================

review_relation_missing = conn.execute(
    """
    SELECT COUNT(*)
    FROM reviews AS r

    LEFT JOIN article_versions AS a
        ON a.article_doi = r.reviewed_doi

    WHERE
        r.status = 'READY'
        AND a.article_doi IS NULL
    """
).fetchone()[0]


# ============================================================
# 14. DISTRIBUIÇÃO SOURCE KIND
# ============================================================

source_distribution = conn.execute(
    """
    SELECT
        source_kind,
        COUNT(*)
    FROM article_versions
    GROUP BY source_kind
    ORDER BY source_kind
    """
).fetchall()


parse_distribution = conn.execute(
    """
    SELECT
        parse_status,
        COUNT(*)
    FROM article_versions
    GROUP BY parse_status
    ORDER BY parse_status
    """
).fetchall()


# ============================================================
# 15. SQLITE INTEGRITY
# ============================================================

integrity = [
    row[0]
    for row in conn.execute(
        "PRAGMA integrity_check;"
    ).fetchall()
]


foreign_key_errors = conn.execute(
    "PRAGMA foreign_key_check;"
).fetchall()


if integrity != ["ok"]:

    conn.close()

    raise RuntimeError(
        "SQLite integrity_check falhou: "
        f"{integrity}"
    )


if foreign_key_errors:

    conn.close()

    raise RuntimeError(
        "SQLite foreign_key_check falhou: "
        f"{foreign_key_errors[:20]}"
    )


# ============================================================
# 16. DETERMINAR STATUS
# ============================================================

acquisition_pass = (

    len(persistent_failures) == 0

    and

    len(missing_registry) == 0

    and

    len(physical_errors) == 0

    and

    review_relation_missing == 0

    and

    len(registered_rows)
    ==
    len(reviewed_rows)
)


# ============================================================
# 17. RUN INFO
# ============================================================

now = utc_now()


run_info_updates = {

    "exact_version_acquisition_version":
        EXACT_VERSION_ACQUISITION_VERSION,

    "exact_version_acquisition_status":
        (
            "PASS"
            if acquisition_pass
            else "INCOMPLETE"
        ),

    "exact_reviewed_versions_required":
        str(
            len(reviewed_rows)
        ),

    "exact_reviewed_versions_from_containers":
        str(
            len(container_reviewed_rows)
        ),

    "exact_reviewed_versions_external":
        str(
            len(external_reviewed_rows)
        ),

    "exact_reviewed_versions_downloaded_this_run":
        str(
            reviewed_downloaded
        ),

    "exact_reviewed_versions_reused_this_run":
        str(
            reviewed_reused
        ),

    "exact_reviewed_versions_persistent_failures":
        str(
            len(persistent_failures)
        ),

    "exact_version_acquisition_completed_at":
        now,
}


for key, value in run_info_updates.items():

    conn.execute(
        """
        INSERT INTO run_info (
            key,
            value,
            updated_at
        )
        VALUES (?, ?, ?)

        ON CONFLICT(key)
        DO UPDATE SET
            value = excluded.value,
            updated_at = excluded.updated_at
        """,
        (
            key,
            value,
            now,
        ),
    )


conn.commit()


# ============================================================
# 18. FECHAR SQLITE
# ============================================================

conn.close()


# ============================================================
# 19. LOG
# ============================================================

log(
    "Exact reviewed version acquisition: "
    f"required={len(reviewed_rows)}, "
    f"containers={len(container_reviewed_rows)}, "
    f"external={len(external_reviewed_rows)}, "
    f"downloaded={reviewed_downloaded}, "
    f"reused={reviewed_reused}, "
    f"persistent_failures="
    f"{len(persistent_failures)}, "
    f"status="
    f"{'PASS' if acquisition_pass else 'INCOMPLETE'}."
)


# ============================================================
# 20. RELATÓRIO FINAL
# ============================================================

print()
print("=" * 72)
print(
    "F1000 CORPUS BUILDER 2.0 — "
    "EXACT VERSION ACQUISITION"
)
print("=" * 72)


print(
    f"Canonical Reviews.......: "
    f"{review_count}"
)

print(
    f"Required ArticleVersions: "
    f"{len(reviewed_rows)}"
)

print(
    f"From containers.........: "
    f"{len(container_reviewed_rows)}"
)

print(
    f"External exact versions.: "
    f"{len(external_reviewed_rows)}"
)


print()
print("This run:")

print(
    f"  Reused external XML...: "
    f"{reviewed_reused}"
)

print(
    f"  Downloaded XML........: "
    f"{reviewed_downloaded}"
)

print(
    f"  Persistent failures...: "
    f"{len(persistent_failures)}"
)

print(
    f"  HTTP requests.........: "
    f"{http_requests_this_run}"
)


print()
print("ArticleVersion registry:")

print(
    f"  Registered............: "
    f"{len(registered_rows)}"
)

for source_kind, count in source_distribution:

    print(
        f"  {source_kind:<20}: "
        f"{count}"
    )


print()
print("Parse state:")

for status, count in parse_distribution:

    print(
        f"  {status:<20}: "
        f"{count}"
    )


print()
print("Coverage validation:")

print(
    f"  Missing registry......: "
    f"{len(missing_registry)}"
)

print(
    f"  Physical XML errors...: "
    f"{len(physical_errors)}"
)

print(
    f"  Reviews without article: "
    f"{review_relation_missing}"
)


if persistent_failures:

    print()
    print(
        "Persistent acquisition failures:"
    )

    for item in persistent_failures[:30]:

        print(
            f"  {item}"
        )


if missing_registry:

    print()
    print(
        "First missing registry DOI:"
    )

    for doi in missing_registry[:30]:

        print(
            f"  {doi}"
        )


if physical_errors:

    print()
    print(
        "First physical errors:"
    )

    for item in physical_errors[:30]:

        print(
            f"  {item}"
        )


print()
print(
    f"SQLite integrity.......: "
    f"{integrity}"
)

print(
    f"Foreign-key errors.....: "
    f"{len(foreign_key_errors)}"
)


print()
print(
    "STATUS.................: "
    f"{'PASS' if acquisition_pass else 'INCOMPLETE'}"
)

print("=" * 72)
print(
    "CÉLULA 6 CONCLUÍDA"
)
print("=" * 72)


AQUISIÇÃO DAS EXACT REVIEWED VERSIONS
Canonical Reviews......: 1131
Unique reviewed_doi....: 639
Available as containers: 345
Need reviewed XML......: 294
Acquisition version....: 2.0.0

[  1/294] reused=0 | downloaded=1 | failed=0
[ 25/294] reused=0 | downloaded=25 | failed=0
[ 50/294] reused=0 | downloaded=50 | failed=0
[ 75/294] reused=0 | downloaded=75 | failed=0
[100/294] reused=0 | downloaded=100 | failed=0
[125/294] reused=0 | downloaded=125 | failed=0
[150/294] reused=0 | downloaded=150 | failed=0
[175/294] reused=0 | downloaded=175 | failed=0
[200/294] reused=0 | downloaded=200 | failed=0
[225/294] reused=0 | downloaded=225 | failed=0
[250/294] reused=0 | downloaded=250 | failed=0
[275/294] reused=0 | downloaded=275 | failed=0
[294/294] reused=0 | downloaded=294 | failed=0
[2026-09-17T23:03:29.215247+00:00] Exact reviewed version acquisition: required=639, containers=345, external=294, downloaded=294, reused=0, persistent_failures=0, status=PASS.

F1000 CORPUS BUILDER 2.0 — E

In [ ]:
# ============================================================
# CÉLULA 7 — PARSING CANÔNICO DAS ARTICLEVERSIONS
# F1000 Corpus Builder 2.0
# ============================================================
#
# RESPONSABILIDADE
#
# Transformar os 639 XMLs JATS já adquiridos em
# ArticleVersions canônicas.
#
# EXTRAI:
#
#   - article_date
#   - title
#   - abstract_text
#   - body_text em Markdown estruturado
#   - tabelas
#   - legendas de figuras
#   - referências
#   - contagens estruturais
#   - contagens de palavras/caracteres
#
# REGRA DE HEADINGS
#
#   H1-H4 ficam reservados à futura Knowledge Unit.
#
#   Estrutura originada do artigo:
#
#       seção nível 1 -> H5
#       seção nível 2 -> H6
#       seção nível 3+ -> bold heading
#
# NÃO FAZ:
#
#   - HTTP;
#   - CanonicalPairs;
#   - KnowledgeUnits;
#   - corpus/*.md;
#   - curation/*.md;
#   - curadoria.
#
# A célula é restart-safe:
#
#   - reparsa deterministicamente as 639 versões;
#   - preserva XML/proveniência;
#   - substitui somente os campos derivados do parsing.
#
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import re
import html
import sqlite3
import xml.etree.ElementTree as ET

from pathlib import Path
from collections import Counter


ARTICLE_PARSER_VERSION = "2.0.0"


# ============================================================
# 2. HELPERS BÁSICOS
# ============================================================

def local_tag(tag):

    if tag is None:
        return ""

    return str(tag).split("}")[-1].lower()


def clean_inline_text(value):

    if value is None:
        return ""

    return re.sub(
        r"\s+",
        " ",
        str(value),
    ).strip()


def element_text(element):

    if element is None:
        return ""

    parts = []

    for text in element.itertext():

        cleaned = clean_inline_text(
            text
        )

        if cleaned:
            parts.append(
                cleaned
            )

    return clean_inline_text(
        " ".join(parts)
    )


def normalize_markdown(markdown_text):
    """
    Normalização conservadora.

    - remove espaços finais;
    - limita blocos vazios;
    - preserva headings/listas/tabelas.
    """

    if not markdown_text:
        return ""


    lines = [
        line.rstrip()
        for line in str(markdown_text).splitlines()
    ]


    output = []
    blank = False


    for line in lines:

        if line.strip():

            output.append(
                line
            )

            blank = False

        else:

            if not blank:

                output.append(
                    ""
                )

            blank = True


    while output and not output[0].strip():
        output.pop(0)

    while output and not output[-1].strip():
        output.pop()


    return "\n".join(
        output
    )


def markdown_word_count(text):

    if not text:
        return 0

    # Remove alguns marcadores Markdown antes da contagem.

    cleaned = re.sub(
        r"[#*_>`|]",
        " ",
        text,
    )

    cleaned = re.sub(
        r"\s+",
        " ",
        cleaned,
    ).strip()


    if not cleaned:
        return 0


    return len(
        cleaned.split()
    )


def escape_markdown_table_cell(value):

    value = clean_inline_text(
        value
    )

    value = value.replace(
        "|",
        "\\|",
    )

    return value


# ============================================================
# 3. XML PRINCIPAL
# ============================================================

def find_direct_child(
    element,
    tag_name,
):

    if element is None:
        return None


    for child in list(element):

        if local_tag(child.tag) == tag_name:
            return child


    return None


def find_front(root):

    return find_direct_child(
        root,
        "front",
    )


def find_body(root):

    return find_direct_child(
        root,
        "body",
    )


def find_back(root):

    return find_direct_child(
        root,
        "back",
    )


def find_article_meta(root):

    front = find_front(
        root
    )

    if front is None:
        return None


    return find_direct_child(
        front,
        "article-meta",
    )


# ============================================================
# 4. DOI
# ============================================================

def extract_main_article_doi(root):

    article_meta = find_article_meta(
        root
    )

    if article_meta is None:
        return None


    for element in article_meta.iter():

        if local_tag(element.tag) != "article-id":
            continue


        pub_id_type = (
            element.attrib.get(
                "pub-id-type"
            )
            or
            element.attrib.get(
                "publication-id-type"
            )
            or
            ""
        ).strip().lower()


        if pub_id_type == "doi":

            doi = normalize_doi(
                element_text(element)
            )

            if doi:
                return doi


    return None


# ============================================================
# 5. TITLE
# ============================================================

def extract_article_title(root):

    article_meta = find_article_meta(
        root
    )

    if article_meta is None:
        return ""


    title_group = find_direct_child(
        article_meta,
        "title-group",
    )


    if title_group is None:

        return ""


    article_title = find_direct_child(
        title_group,
        "article-title",
    )


    if article_title is None:
        return ""


    return element_text(
        article_title
    )


# ============================================================
# 6. ARTICLE DATE
# ============================================================

def parse_date_element(
    date_element,
):

    if date_element is None:
        return None


    year = None
    month = None
    day = None


    for child in list(date_element):

        tag = local_tag(
            child.tag
        )


        value = clean_inline_text(
            element_text(child)
        )


        if not value:
            continue


        if tag == "year":
            year = value

        elif tag == "month":
            month = value

        elif tag == "day":
            day = value


    if not year:

        return None


    if month:

        try:

            month = (
                f"{int(month):02d}"
            )

        except Exception:
            pass


    if day:

        try:

            day = (
                f"{int(day):02d}"
            )

        except Exception:
            pass


    if month and day:

        return (
            f"{year}-{month}-{day}"
        )


    if month:

        return (
            f"{year}-{month}"
        )


    return year


def extract_article_date(root):
    """
    Preferência:
      1. pub-date epub
      2. pub-date collection
      3. qualquer pub-date
      4. history/date
    """

    article_meta = find_article_meta(
        root
    )

    if article_meta is None:
        return None


    pub_dates = []


    for element in article_meta.iter():

        if local_tag(element.tag) != "pub-date":
            continue


        pub_type = (
            element.attrib.get(
                "pub-type"
            )
            or
            element.attrib.get(
                "publication-format"
            )
            or
            ""
        ).strip().lower()


        pub_dates.append(
            (
                pub_type,
                element,
            )
        )


    preferred = [
        "epub",
        "electronic",
        "collection",
    ]


    for preferred_type in preferred:

        for pub_type, element in pub_dates:

            if pub_type == preferred_type:

                value = parse_date_element(
                    element
                )

                if value:
                    return value


    for _, element in pub_dates:

        value = parse_date_element(
            element
        )

        if value:
            return value


    for element in article_meta.iter():

        if local_tag(element.tag) != "date":
            continue


        value = parse_date_element(
            element
        )

        if value:
            return value


    return None


# ============================================================
# 7. INLINE JATS → MARKDOWN
# ============================================================

def inline_to_markdown(element):
    """
    Renderizador inline conservador.

    Preserva principalmente:
      bold
      italic
      superscript
      subscript
      ext-link
      xref text

    Não tenta reproduzir layout editorial.
    """

    if element is None:
        return ""


    parts = []


    if element.text:

        parts.append(
            element.text
        )


    for child in list(element):

        tag = local_tag(
            child.tag
        )


        inner = inline_to_markdown(
            child
        )


        if tag in {
            "bold",
            "strong",
        }:

            rendered = (
                f"**{inner.strip()}**"
                if inner.strip()
                else ""
            )


        elif tag in {
            "italic",
            "i",
        }:

            rendered = (
                f"*{inner.strip()}*"
                if inner.strip()
                else ""
            )


        elif tag == "sup":

            rendered = (
                f"^{inner.strip()}"
                if inner.strip()
                else ""
            )


        elif tag == "sub":

            rendered = (
                f"_{inner.strip()}"
                if inner.strip()
                else ""
            )


        elif tag in {
            "ext-link",
            "uri",
        }:

            href = (
                child.attrib.get(
                    "{http://www.w3.org/1999/xlink}href"
                )
                or
                child.attrib.get(
                    "href"
                )
            )


            label = clean_inline_text(
                inner
            )


            if href and label:

                rendered = (
                    f"[{label}]({href})"
                )

            elif href:

                rendered = (
                    f"<{href}>"
                )

            else:

                rendered = label


        else:

            rendered = inner


        if rendered:

            parts.append(
                rendered
            )


        if child.tail:

            parts.append(
                child.tail
            )


    result = "".join(
        parts
    )


    result = re.sub(
        r"[ \t\r\f\v]+",
        " ",
        result,
    )


    return result.strip()


# ============================================================
# 8. PARÁGRAFOS
# ============================================================

def render_paragraph(element):

    text = inline_to_markdown(
        element
    )


    return clean_inline_text(
        text
    )


# ============================================================
# 9. LISTAS
# ============================================================

def render_list(
    list_element,
    indent=0,
):

    lines = []


    list_type = (
        list_element.attrib.get(
            "list-type"
        )
        or
        ""
    ).strip().lower()


    ordered = (
        list_type
        in {
            "order",
            "ordered",
            "number",
            "numeric",
        }
    )


    item_number = 1


    for child in list(
        list_element
    ):

        if local_tag(child.tag) != "list-item":
            continue


        marker = (
            f"{item_number}."
            if ordered
            else "-"
        )


        paragraphs = []

        nested_lists = []


        for item_child in list(
            child
        ):

            tag = local_tag(
                item_child.tag
            )


            if tag == "p":

                text = render_paragraph(
                    item_child
                )

                if text:
                    paragraphs.append(
                        text
                    )


            elif tag == "list":

                nested_lists.append(
                    item_child
                )


            else:

                text = element_text(
                    item_child
                )

                if text:
                    paragraphs.append(
                        text
                    )


        item_text = clean_inline_text(
            " ".join(paragraphs)
        )


        prefix = (
            " " * indent
        )


        lines.append(
            f"{prefix}{marker} {item_text}".rstrip()
        )


        for nested in nested_lists:

            nested_text = render_list(
                nested,
                indent=indent + 2,
            )


            if nested_text:

                lines.append(
                    nested_text
                )


        item_number += 1


    return "\n".join(
        lines
    )


# ============================================================
# 10. QUOTES / CODE
# ============================================================

def render_quote(element):

    text = element_text(
        element
    )


    if not text:
        return ""


    return "\n".join(
        f"> {line}"
        for line in text.splitlines()
        if line.strip()
    )


def render_code(element):

    text = "".join(
        element.itertext()
    ).strip()


    if not text:
        return ""


    return (
        "```\n"
        f"{text}\n"
        "```"
    )


# ============================================================
# 11. TABELAS
# ============================================================

def find_table_element(
    table_wrap,
):

    for element in table_wrap.iter():

        if local_tag(element.tag) == "table":
            return element


    return None


def extract_table_rows(
    table_element,
):

    rows = []


    for tr in table_element.iter():

        if local_tag(tr.tag) != "tr":
            continue


        cells = []


        for child in list(tr):

            tag = local_tag(
                child.tag
            )


            if tag not in {
                "td",
                "th",
            }:
                continue


            cells.append(
                {
                    "tag":
                        tag,

                    "text":
                        element_text(
                            child
                        ),
                }
            )


        if cells:
            rows.append(
                cells
            )


    return rows


def render_table_wrap(
    table_wrap,
):

    blocks = []


    label = None
    caption = None


    for child in list(
        table_wrap
    ):

        tag = local_tag(
            child.tag
        )


        if tag == "label":

            label = element_text(
                child
            )


        elif tag == "caption":

            caption = element_text(
                child
            )


    heading_parts = [
        value
        for value in [
            label,
            caption,
        ]
        if value
    ]


    if heading_parts:

        blocks.append(
            "**"
            +
            " — ".join(
                heading_parts
            )
            +
            "**"
        )


    table_element = find_table_element(
        table_wrap
    )


    if table_element is None:

        fallback = element_text(
            table_wrap
        )


        if fallback:

            blocks.append(
                fallback
            )


        return normalize_markdown(
            "\n\n".join(blocks)
        )


    rows = extract_table_rows(
        table_element
    )


    if not rows:

        fallback = element_text(
            table_element
        )


        if fallback:

            blocks.append(
                fallback
            )


        return normalize_markdown(
            "\n\n".join(blocks)
        )


    max_columns = max(
        len(row)
        for row in rows
    )


    normalized_rows = []


    for row in rows:

        values = [
            escape_markdown_table_cell(
                cell["text"]
            )
            for cell in row
        ]


        while len(values) < max_columns:

            values.append(
                ""
            )


        normalized_rows.append(
            values
        )


    # --------------------------------------------------------
    # Header:
    # se primeira linha tem TH, usa-a.
    # Caso contrário cria cabeçalho neutro.
    # --------------------------------------------------------

    first_row_has_header = any(
        cell["tag"] == "th"
        for cell in rows[0]
    )


    if first_row_has_header:

        header = normalized_rows[0]

        data_rows = normalized_rows[1:]

    else:

        header = [
            f"Column {i}"
            for i in range(
                1,
                max_columns + 1,
            )
        ]

        data_rows = normalized_rows


    markdown_lines = []


    markdown_lines.append(
        "| "
        +
        " | ".join(header)
        +
        " |"
    )


    markdown_lines.append(
        "| "
        +
        " | ".join(
            ["---"] * max_columns
        )
        +
        " |"
    )


    for row in data_rows:

        markdown_lines.append(
            "| "
            +
            " | ".join(row)
            +
            " |"
        )


    blocks.append(
        "\n".join(
            markdown_lines
        )
    )


    return normalize_markdown(
        "\n\n".join(blocks)
    )


# ============================================================
# 12. FIGURAS
# ============================================================

def render_figure(
    fig_element,
):

    label = None
    caption = None


    for child in list(
        fig_element
    ):

        tag = local_tag(
            child.tag
        )


        if tag == "label":

            label = element_text(
                child
            )


        elif tag == "caption":

            caption = element_text(
                child
            )


    blocks = [
        "[IMAGE OMITTED]"
    ]


    description = " — ".join(
        value
        for value in [
            label,
            caption,
        ]
        if value
    )


    if description:

        blocks.append(
            f"**{description}**"
        )


    return "\n\n".join(
        blocks
    )


# ============================================================
# 13. SUPPLEMENTARY MATERIAL
# ============================================================

def render_supplementary_material(
    element,
):

    label = None
    caption = None


    for child in list(
        element
    ):

        tag = local_tag(
            child.tag
        )


        if tag == "label":

            label = element_text(
                child
            )


        elif tag == "caption":

            caption = element_text(
                child
            )


    description = " — ".join(
        value
        for value in [
            label,
            caption,
        ]
        if value
    )


    if description:

        return (
            "[SUPPLEMENTARY MATERIAL OMITTED]\n\n"
            f"**{description}**"
        )


    return (
        "[SUPPLEMENTARY MATERIAL OMITTED]"
    )


# ============================================================
# 14. ELEMENTO GENÉRICO DE CONTEÚDO
# ============================================================

def render_content_element(
    element,
    section_depth=1,
):
    """
    Renderiza elemento JATS de conteúdo.

    Não processa <sec> aqui; <sec> possui função própria.
    """

    tag = local_tag(
        element.tag
    )


    if tag == "p":

        return render_paragraph(
            element
        )


    if tag == "list":

        return render_list(
            element
        )


    if tag == "table-wrap":

        return render_table_wrap(
            element
        )


    if tag == "fig":

        return render_figure(
            element
        )


    if tag == "supplementary-material":

        return render_supplementary_material(
            element
        )


    if tag in {
        "disp-quote",
        "speech",
    }:

        return render_quote(
            element
        )


    if tag in {
        "code",
        "preformat",
    }:

        return render_code(
            element
        )


    if tag in {
        "boxed-text",
        "statement",
        "disp-formula",
    }:

        text = element_text(
            element
        )

        return text


    # Elementos puramente estruturais/metadados que não
    # devem virar bloco independente.

    if tag in {
        "title",
        "label",
        "caption",
        "permissions",
    }:

        return ""


    # --------------------------------------------------------
    # Fallback conservador.
    #
    # Só usa texto quando não há filhos estruturais que serão
    # processados separadamente.
    # --------------------------------------------------------

    structural_children = {

        "sec",
        "p",
        "list",
        "table-wrap",
        "fig",
        "supplementary-material",
        "disp-quote",
        "boxed-text",
        "code",
        "preformat",
    }


    if any(
        local_tag(child.tag)
        in structural_children
        for child in list(element)
    ):

        blocks = []


        for child in list(element):

            child_tag = local_tag(
                child.tag
            )


            if child_tag == "sec":

                rendered = render_section(
                    child,
                    depth=section_depth,
                )

            else:

                rendered = render_content_element(
                    child,
                    section_depth=section_depth,
                )


            if rendered:

                blocks.append(
                    rendered
                )


        return normalize_markdown(
            "\n\n".join(blocks)
        )


    return element_text(
        element
    )


# ============================================================
# 15. SEÇÕES
# ============================================================

def render_section(
    section_element,
    depth=1,
):

    blocks = []


    title = None


    for child in list(
        section_element
    ):

        if local_tag(child.tag) == "title":

            title = element_text(
                child
            )

            break


    if title:

        if depth == 1:

            blocks.append(
                f"##### {title}"
            )


        elif depth == 2:

            blocks.append(
                f"###### {title}"
            )


        else:

            # Markdown padrão só define H1-H6.
            # Profundidade adicional fica semanticamente
            # marcada em bold.

            blocks.append(
                f"**{title}**"
            )


    for child in list(
        section_element
    ):

        tag = local_tag(
            child.tag
        )


        if tag == "title":
            continue


        if tag == "sec":

            rendered = render_section(
                child,
                depth=depth + 1,
            )


        else:

            rendered = render_content_element(
                child,
                section_depth=depth + 1,
            )


        if rendered:

            blocks.append(
                rendered
            )


    return normalize_markdown(
        "\n\n".join(blocks)
    )


# ============================================================
# 16. ABSTRACT
# ============================================================

def extract_abstract_markdown(
    root,
):

    article_meta = find_article_meta(
        root
    )


    if article_meta is None:
        return ""


    abstracts = []


    for child in list(
        article_meta
    ):

        if local_tag(child.tag) == "abstract":

            abstracts.append(
                child
            )


    if not abstracts:
        return ""


    blocks = []


    for abstract in abstracts:

        abstract_type = (
            abstract.attrib.get(
                "abstract-type"
            )
            or
            ""
        ).strip()


        # Evita duplicar graphical abstract como texto
        # principal.

        if (
            abstract_type.lower()
            in {
                "graphical",
                "graphical-abstract",
            }
        ):

            continue


        abstract_title = None


        for child in list(
            abstract
        ):

            if local_tag(child.tag) == "title":

                abstract_title = element_text(
                    child
                )

                break


        if abstract_title:

            blocks.append(
                f"**{abstract_title}**"
            )


        for child in list(
            abstract
        ):

            tag = local_tag(
                child.tag
            )


            if tag == "title":
                continue


            if tag == "sec":

                section_title = None


                for sec_child in list(
                    child
                ):

                    if (
                        local_tag(sec_child.tag)
                        ==
                        "title"
                    ):

                        section_title = element_text(
                            sec_child
                        )

                        break


                if section_title:

                    blocks.append(
                        f"**{section_title}**"
                    )


                for sec_child in list(
                    child
                ):

                    if (
                        local_tag(sec_child.tag)
                        ==
                        "title"
                    ):
                        continue


                    rendered = render_content_element(
                        sec_child,
                        section_depth=1,
                    )


                    if rendered:

                        blocks.append(
                            rendered
                        )


            else:

                rendered = render_content_element(
                    child,
                    section_depth=1,
                )


                if rendered:

                    blocks.append(
                        rendered
                    )


    return normalize_markdown(
        "\n\n".join(blocks)
    )


# ============================================================
# 17. BODY
# ============================================================

def extract_body_markdown(
    root,
):

    body = find_body(
        root
    )


    if body is None:
        return ""


    blocks = []


    for child in list(
        body
    ):

        tag = local_tag(
            child.tag
        )


        if tag == "sec":

            rendered = render_section(
                child,
                depth=1,
            )


        else:

            rendered = render_content_element(
                child,
                section_depth=1,
            )


        if rendered:

            blocks.append(
                rendered
            )


    return normalize_markdown(
        "\n\n".join(blocks)
    )


# ============================================================
# 18. REFERENCES
# ============================================================

def render_reference(
    ref_element,
):

    label = None


    for child in list(
        ref_element
    ):

        if local_tag(child.tag) == "label":

            label = element_text(
                child
            )

            break


    # Exclui label da extração principal para evitar:
    # "1 1 Author..."
    #
    # Tenta primeiro mixed-citation / element-citation.

    citation = None


    for element in ref_element.iter():

        if local_tag(element.tag) in {
            "mixed-citation",
            "element-citation",
            "nlm-citation",
        }:

            citation = element_text(
                element
            )

            if citation:
                break


    if not citation:

        parts = []


        for child in list(
            ref_element
        ):

            if local_tag(child.tag) == "label":
                continue


            text = element_text(
                child
            )


            if text:
                parts.append(
                    text
                )


        citation = clean_inline_text(
            " ".join(parts)
        )


    if not citation:
        return ""


    if label:

        return (
            f"{label}. {citation}"
        )


    return citation


def extract_references_markdown(
    root,
):

    back = find_back(
        root
    )


    if back is None:

        return "", 0


    ref_list = None


    for element in back.iter():

        if local_tag(element.tag) == "ref-list":

            ref_list = element
            break


    if ref_list is None:

        return "", 0


    references = []


    for child in list(
        ref_list
    ):

        if local_tag(child.tag) != "ref":
            continue


        rendered = render_reference(
            child
        )


        if rendered:

            references.append(
                rendered
            )


    if not references:

        return "", 0


    lines = [
        "##### References",
        "",
    ]


    for index, reference in enumerate(
        references,
        start=1,
    ):

        # Se a referência já possui label explícito,
        # não adiciona novo número.

        if re.match(
            r"^\s*\d+\s*[\.\)]",
            reference,
        ):

            lines.append(
                reference
            )

        else:

            lines.append(
                f"{index}. {reference}"
            )


    return (
        normalize_markdown(
            "\n".join(lines)
        ),
        len(references),
    )


# ============================================================
# 19. CONTAGENS ESTRUTURAIS
# ============================================================

def count_sections(
    root,
):

    body = find_body(
        root
    )


    if body is None:
        return 0


    return sum(
        1
        for element in body.iter()
        if local_tag(element.tag) == "sec"
    )


def count_tables(
    root,
):

    body = find_body(
        root
    )


    if body is None:
        return 0


    return sum(
        1
        for element in body.iter()
        if local_tag(element.tag) == "table-wrap"
    )


def count_figures(
    root,
):

    body = find_body(
        root
    )


    if body is None:
        return 0


    return sum(
        1
        for element in body.iter()
        if local_tag(element.tag) == "fig"
    )


# ============================================================
# 20. PARSER DE UMA ARTICLEVERSION
# ============================================================

def parse_article_version(
    xml_bytes,
    expected_doi,
):

    root = ET.fromstring(
        xml_bytes
    )


    observed_doi = extract_main_article_doi(
        root
    )


    observed_doi = normalize_doi(
        observed_doi
    )


    expected_doi = normalize_doi(
        expected_doi
    )


    if observed_doi != expected_doi:

        raise RuntimeError(
            "DOI mismatch during parsing: "
            f"expected={expected_doi}; "
            f"observed={observed_doi}"
        )


    title = extract_article_title(
        root
    )


    article_date = extract_article_date(
        root
    )


    abstract_text = extract_abstract_markdown(
        root
    )


    body_text = extract_body_markdown(
        root
    )


    references_text, reference_count = (
        extract_references_markdown(
            root
        )
    )


    # Referências fazem parte do conteúdo textual canônico
    # da ArticleVersion, mas permanecem separadas do body
    # científico até esta etapa.

    if references_text:

        if body_text:

            canonical_body = (
                body_text
                +
                "\n\n"
                +
                references_text
            )

        else:

            canonical_body = (
                references_text
            )

    else:

        canonical_body = (
            body_text
        )


    canonical_body = normalize_markdown(
        canonical_body
    )


    article_payload_parts = [
        value
        for value in [
            title,
            abstract_text,
            canonical_body,
        ]
        if value
    ]


    article_payload = normalize_markdown(
        "\n\n".join(
            article_payload_parts
        )
    )


    return {

        "article_date":
            article_date,

        "title":
            title,

        "abstract_text":
            abstract_text,

        "body_text":
            canonical_body,

        "abstract_word_count":
            markdown_word_count(
                abstract_text
            ),

        "body_word_count":
            markdown_word_count(
                canonical_body
            ),

        "article_word_count":
            markdown_word_count(
                article_payload
            ),

        "article_character_count":
            len(
                article_payload
            ),

        "section_count":
            count_sections(
                root
            ),

        "table_count":
            count_tables(
                root
            ),

        "figure_count":
            count_figures(
                root
            ),

        "reference_count":
            reference_count,
    }


# ============================================================
# 21. SQLITE / PRÉ-CONDIÇÕES
# ============================================================

conn = open_builder_db()


article_rows = conn.execute(
    """
    SELECT
        article_doi,
        base_doi,
        article_version,
        source_kind,
        xml_path,
        xml_sha256,
        xml_bytes,
        parse_status
    FROM article_versions
    ORDER BY
        base_doi,
        article_version,
        article_doi
    """
).fetchall()


review_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM reviews
    WHERE status = 'READY'
    """
).fetchone()[0]


review_relation_missing = conn.execute(
    """
    SELECT COUNT(*)
    FROM reviews AS r

    LEFT JOIN article_versions AS a
        ON a.article_doi = r.reviewed_doi

    WHERE
        r.status = 'READY'
        AND a.article_doi IS NULL
    """
).fetchone()[0]


if not article_rows:

    conn.close()

    raise RuntimeError(
        "Nenhuma ArticleVersion registrada. "
        "Execute C6 antes da C7."
    )


if review_relation_missing != 0:

    conn.close()

    raise RuntimeError(
        "Há Reviews sem ArticleVersion antes do parsing: "
        f"{review_relation_missing}"
    )


# ============================================================
# 22. RELATÓRIO INICIAL
# ============================================================

print()
print("=" * 72)
print(
    "PARSING CANÔNICO DAS ARTICLEVERSIONS"
)
print("=" * 72)

print(
    f"ArticleVersions........: "
    f"{len(article_rows)}"
)

print(
    f"Canonical Reviews......: "
    f"{review_count}"
)

print(
    f"Parser version.........: "
    f"{ARTICLE_PARSER_VERSION}"
)

print()


# ============================================================
# 23. CONTADORES
# ============================================================

parsed_count = 0
error_count = 0

missing_title = []
missing_abstract = []
missing_body = []
missing_date = []

zero_reference_articles = []
zero_section_articles = []

parse_errors = []

source_counter = Counter()


# ============================================================
# 24. PARSING
# ============================================================

for index, (
    article_doi,
    base_doi,
    article_version,
    source_kind,
    xml_path_text,
    expected_hash,
    expected_bytes,
    old_parse_status,
) in enumerate(
    article_rows,
    start=1,
):

    doi = normalize_doi(
        article_doi
    )


    source_counter[
        source_kind
    ] += 1


    xml_path = Path(
        xml_path_text
    )


    try:

        # ----------------------------------------------------
        # A. Integridade física
        # ----------------------------------------------------

        if not xml_path.exists():

            raise RuntimeError(
                "XML file missing."
            )


        xml_bytes = xml_path.read_bytes()


        if (
            expected_bytes is not None
            and
            len(xml_bytes)
            !=
            expected_bytes
        ):

            raise RuntimeError(
                "XML byte count mismatch."
            )


        actual_hash = sha256_bytes(
            xml_bytes
        )


        if (
            expected_hash
            and
            actual_hash
            !=
            expected_hash
        ):

            raise RuntimeError(
                "XML SHA256 mismatch."
            )


        # ----------------------------------------------------
        # B. Parsing
        # ----------------------------------------------------

        parsed = parse_article_version(
            xml_bytes,
            expected_doi=doi,
        )


        # ----------------------------------------------------
        # C. Diagnósticos não destrutivos
        # ----------------------------------------------------

        if not parsed["title"]:

            missing_title.append(
                doi
            )


        if not parsed["abstract_text"]:

            missing_abstract.append(
                doi
            )


        if not parsed["body_text"]:

            missing_body.append(
                doi
            )


        if not parsed["article_date"]:

            missing_date.append(
                doi
            )


        if (
            parsed["reference_count"]
            ==
            0
        ):

            zero_reference_articles.append(
                doi
            )


        if (
            parsed["section_count"]
            ==
            0
        ):

            zero_section_articles.append(
                doi
            )


        # ----------------------------------------------------
        # D. Persistir campos derivados
        # ----------------------------------------------------

        now = utc_now()


        conn.execute(
            """
            UPDATE article_versions

            SET
                article_date = ?,
                title = ?,
                abstract_text = ?,
                body_text = ?,

                abstract_word_count = ?,
                body_word_count = ?,
                article_word_count = ?,
                article_character_count = ?,

                section_count = ?,
                table_count = ?,
                figure_count = ?,
                reference_count = ?,

                parse_status = 'PARSED',
                error_message = NULL,
                parsed_at = ?,
                updated_at = ?

            WHERE article_doi = ?
            """,
            (
                parsed[
                    "article_date"
                ],
                parsed[
                    "title"
                ],
                parsed[
                    "abstract_text"
                ],
                parsed[
                    "body_text"
                ],
                parsed[
                    "abstract_word_count"
                ],
                parsed[
                    "body_word_count"
                ],
                parsed[
                    "article_word_count"
                ],
                parsed[
                    "article_character_count"
                ],
                parsed[
                    "section_count"
                ],
                parsed[
                    "table_count"
                ],
                parsed[
                    "figure_count"
                ],
                parsed[
                    "reference_count"
                ],
                now,
                now,
                doi,
            ),
        )


        parsed_count += 1


    except Exception as exc:

        error_count += 1


        error_message = (
            f"{type(exc).__name__}: "
            f"{exc}"
        )


        parse_errors.append(
            {
                "article_doi":
                    doi,

                "source_kind":
                    source_kind,

                "error":
                    error_message,
            }
        )


        conn.execute(
            """
            UPDATE article_versions

            SET
                parse_status = 'ERROR',
                error_message = ?,
                parsed_at = ?,
                updated_at = ?

            WHERE article_doi = ?
            """,
            (
                error_message,
                utc_now(),
                utc_now(),
                doi,
            ),
        )


    conn.commit()


    # --------------------------------------------------------
    # E. Progresso
    # --------------------------------------------------------

    if (
        index == 1
        or
        index % 25 == 0
        or
        index == len(article_rows)
    ):

        print(
            f"[{index:>3}/{len(article_rows)}] "
            f"parsed={parsed_count} | "
            f"errors={error_count} | "
            f"missing_abstract="
            f"{len(missing_abstract)} | "
            f"missing_body="
            f"{len(missing_body)}"
        )


# ============================================================
# 25. ESTADO FINAL NO SQLITE
# ============================================================

parse_distribution = conn.execute(
    """
    SELECT
        parse_status,
        COUNT(*)
    FROM article_versions
    GROUP BY parse_status
    ORDER BY parse_status
    """
).fetchall()


parsed_db_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM article_versions
    WHERE parse_status = 'PARSED'
    """
).fetchone()[0]


error_db_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM article_versions
    WHERE parse_status = 'ERROR'
    """
).fetchone()[0]


# ============================================================
# 26. ESTATÍSTICAS
# ============================================================

stats = conn.execute(
    """
    SELECT

        MIN(article_word_count),
        AVG(article_word_count),
        MAX(article_word_count),

        MIN(body_word_count),
        AVG(body_word_count),
        MAX(body_word_count),

        SUM(section_count),
        SUM(table_count),
        SUM(figure_count),
        SUM(reference_count)

    FROM article_versions
    WHERE parse_status = 'PARSED'
    """
).fetchone()


(
    min_article_words,
    avg_article_words,
    max_article_words,
    min_body_words,
    avg_body_words,
    max_body_words,
    total_sections,
    total_tables,
    total_figures,
    total_references,
) = stats


# ============================================================
# 27. SANITY CHECKS DE MARKDOWN
# ============================================================

markdown_h1_h4_violations = []


parsed_content_rows = conn.execute(
    """
    SELECT
        article_doi,
        abstract_text,
        body_text
    FROM article_versions
    WHERE parse_status = 'PARSED'
    """
).fetchall()


for (
    article_doi,
    abstract_text,
    body_text,
) in parsed_content_rows:

    combined = (
        (abstract_text or "")
        +
        "\n"
        +
        (body_text or "")
    )


    # H1-H4 são reservados à Knowledge Unit.

    if re.search(
        r"(?m)^#{1,4}\s+",
        combined,
    ):

        markdown_h1_h4_violations.append(
            article_doi
        )


# ============================================================
# 28. RELAÇÃO REVIEW → PARSED ARTICLE
# ============================================================

reviews_without_parsed_article = (
    conn.execute(
        """
        SELECT COUNT(*)

        FROM reviews AS r

        LEFT JOIN article_versions AS a
            ON a.article_doi = r.reviewed_doi

        WHERE
            r.status = 'READY'

            AND (
                a.article_doi IS NULL
                OR
                a.parse_status != 'PARSED'
            )
        """
    ).fetchone()[0]
)


# ============================================================
# 29. SQLITE INTEGRITY
# ============================================================

integrity = [
    row[0]
    for row in conn.execute(
        "PRAGMA integrity_check;"
    ).fetchall()
]


foreign_key_errors = conn.execute(
    "PRAGMA foreign_key_check;"
).fetchall()


# ============================================================
# 30. PASS / FAIL
# ============================================================

parsing_pass = (

    parsed_db_count
    ==
    len(article_rows)

    and

    error_db_count
    ==
    0

    and

    reviews_without_parsed_article
    ==
    0

    and

    len(markdown_h1_h4_violations)
    ==
    0

    and

    integrity
    ==
    ["ok"]

    and

    len(foreign_key_errors)
    ==
    0
)


# ============================================================
# 31. RUN INFO
# ============================================================

now = utc_now()


run_info_updates = {

    "article_parser_version":
        ARTICLE_PARSER_VERSION,

    "article_parser_status":
        (
            "PASS"
            if parsing_pass
            else "FAIL"
        ),

    "article_versions_parsed":
        str(
            parsed_db_count
        ),

    "article_versions_parse_errors":
        str(
            error_db_count
        ),

    "article_versions_missing_title":
        str(
            len(missing_title)
        ),

    "article_versions_missing_abstract":
        str(
            len(missing_abstract)
        ),

    "article_versions_missing_body":
        str(
            len(missing_body)
        ),

    "article_versions_missing_date":
        str(
            len(missing_date)
        ),

    "article_markdown_heading_policy":
        (
            "H1-H4 reserved for KnowledgeUnit; "
            "article section level 1=H5; "
            "level 2=H6; "
            "level 3+=bold"
        ),

    "article_parser_completed_at":
        now,
}


for key, value in run_info_updates.items():

    conn.execute(
        """
        INSERT INTO run_info (
            key,
            value,
            updated_at
        )
        VALUES (?, ?, ?)

        ON CONFLICT(key)
        DO UPDATE SET
            value = excluded.value,
            updated_at = excluded.updated_at
        """,
        (
            key,
            value,
            now,
        ),
    )


conn.commit()


# ============================================================
# 32. FECHAR SQLITE
# ============================================================

conn.close()


# ============================================================
# 33. LOG
# ============================================================

log(
    "ArticleVersion canonical parsing: "
    f"total={len(article_rows)}, "
    f"parsed={parsed_db_count}, "
    f"errors={error_db_count}, "
    f"missing_title={len(missing_title)}, "
    f"missing_abstract={len(missing_abstract)}, "
    f"missing_body={len(missing_body)}, "
    f"status="
    f"{'PASS' if parsing_pass else 'FAIL'}."
)


# ============================================================
# 34. RELATÓRIO FINAL
# ============================================================

print()
print("=" * 72)
print(
    "F1000 CORPUS BUILDER 2.0 — "
    "ARTICLE PARSING"
)
print("=" * 72)


print(
    f"ArticleVersions........: "
    f"{len(article_rows)}"
)

print(
    f"Parsed.................: "
    f"{parsed_db_count}"
)

print(
    f"Errors.................: "
    f"{error_db_count}"
)


print()
print("Source distribution:")

for source_kind in sorted(
    source_counter
):

    print(
        f"  {source_kind:<20}: "
        f"{source_counter[source_kind]}"
    )


print()
print("Content diagnostics:")

print(
    f"  Missing title........: "
    f"{len(missing_title)}"
)

print(
    f"  Missing abstract.....: "
    f"{len(missing_abstract)}"
)

print(
    f"  Missing body.........: "
    f"{len(missing_body)}"
)

print(
    f"  Missing article date.: "
    f"{len(missing_date)}"
)

print(
    f"  Zero sections........: "
    f"{len(zero_section_articles)}"
)

print(
    f"  Zero references......: "
    f"{len(zero_reference_articles)}"
)


print()
print("Corpus structure totals:")

print(
    f"  Sections.............: "
    f"{total_sections or 0}"
)

print(
    f"  Tables...............: "
    f"{total_tables or 0}"
)

print(
    f"  Figures..............: "
    f"{total_figures or 0}"
)

print(
    f"  References...........: "
    f"{total_references or 0}"
)


print()
print("Article word counts:")

print(
    f"  Minimum..............: "
    f"{min_article_words or 0}"
)

print(
    f"  Mean.................: "
    f"{(avg_article_words or 0):.1f}"
)

print(
    f"  Maximum..............: "
    f"{max_article_words or 0}"
)


print()
print("Body word counts:")

print(
    f"  Minimum..............: "
    f"{min_body_words or 0}"
)

print(
    f"  Mean.................: "
    f"{(avg_body_words or 0):.1f}"
)

print(
    f"  Maximum..............: "
    f"{max_body_words or 0}"
)


print()
print("Relational validation:")

print(
    f"  Reviews without parsed article: "
    f"{reviews_without_parsed_article}"
)


print()
print("Markdown validation:")

print(
    f"  H1-H4 violations.....: "
    f"{len(markdown_h1_h4_violations)}"
)


print()
print("SQLite validation:")

print(
    f"  Integrity............: "
    f"{integrity}"
)

print(
    f"  Foreign-key errors...: "
    f"{len(foreign_key_errors)}"
)


# ============================================================
# 35. DIAGNÓSTICOS SOMENTE SE NECESSÁRIOS
# ============================================================

if parse_errors:

    print()
    print(
        "First parsing errors:"
    )

    for item in parse_errors[:20]:

        print(
            f"  {item}"
        )


if missing_title:

    print()
    print(
        "First ArticleVersions without title:"
    )

    for doi in missing_title[:20]:

        print(
            f"  {doi}"
        )


if missing_abstract:

    print()
    print(
        "First ArticleVersions without abstract:"
    )

    for doi in missing_abstract[:20]:

        print(
            f"  {doi}"
        )


if missing_body:

    print()
    print(
        "First ArticleVersions without body:"
    )

    for doi in missing_body[:20]:

        print(
            f"  {doi}"
        )


if markdown_h1_h4_violations:

    print()
    print(
        "First H1-H4 Markdown violations:"
    )

    for doi in markdown_h1_h4_violations[:20]:

        print(
            f"  {doi}"
        )


print()
print(
    "HTTP requests performed: 0"
)

print()
print(
    "STATUS.................: "
    f"{'PASS' if parsing_pass else 'FAIL'}"
)

print("=" * 72)
print(
    "CÉLULA 7 CONCLUÍDA"
)
print("=" * 72)


PARSING CANÔNICO DAS ARTICLEVERSIONS
ArticleVersions........: 639
Canonical Reviews......: 1131
Parser version.........: 2.0.0

[  1/639] parsed=1 | errors=0 | missing_abstract=0 | missing_body=0
[ 25/639] parsed=25 | errors=0 | missing_abstract=0 | missing_body=0
[ 50/639] parsed=50 | errors=0 | missing_abstract=0 | missing_body=0
[ 75/639] parsed=75 | errors=0 | missing_abstract=0 | missing_body=0
[100/639] parsed=100 | errors=0 | missing_abstract=0 | missing_body=0
[125/639] parsed=125 | errors=0 | missing_abstract=0 | missing_body=0
[150/639] parsed=150 | errors=0 | missing_abstract=0 | missing_body=0
[175/639] parsed=175 | errors=0 | missing_abstract=0 | missing_body=0
[200/639] parsed=200 | errors=0 | missing_abstract=0 | missing_body=0
[225/639] parsed=225 | errors=0 | missing_abstract=0 | missing_body=0
[250/639] parsed=250 | errors=0 | missing_abstract=0 | missing_body=0
[275/639] parsed=275 | errors=0 | missing_abstract=0 | missing_body=0
[300/639] parsed=300 | errors=0 | mi

In [ ]:
# ============================================================
# CÉLULA 8 — CONSTRUÇÃO DOS CANONICALPAIRS
# F1000 Corpus Builder 2.0
# ============================================================
#
# RESPONSABILIDADE
#
# Materializar a relação atômica:
#
#     ArticleVersion exata ↔ Review
#
# Cada Review READY deve produzir exatamente UM CanonicalPair.
#
# O CanonicalPair:
#
#   - NÃO duplica o texto do artigo;
#   - NÃO duplica o texto da review;
#   - preserva apenas a relação e metadados relacionais;
#   - aponta para ArticleVersion via article_doi;
#   - aponta para Review via review_doi;
#   - mantém base_doi;
#   - mantém recommendation;
#   - mantém CURRENT/HISTORICAL;
#   - mantém source_container_doi.
#
# INVARIANTES:
#
#   Reviews READY = CanonicalPairs READY
#
#   review.reviewed_doi
#       =
#   canonical_pair.article_doi
#
#   review.base_doi
#       =
#   article_version.base_doi
#       =
#   canonical_pair.base_doi
#
#   review.reviewed_version
#       =
#   article_version.article_version
#       =
#   canonical_pair.article_version
#
# NÃO FAZ:
#
#   - HTTP;
#   - parsing XML;
#   - KnowledgeUnits;
#   - Markdown;
#   - curadoria.
#
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import sqlite3
from collections import Counter


CANONICAL_PAIR_BUILDER_VERSION = "2.0.0"


# ============================================================
# 2. SQLITE
# ============================================================

conn = open_builder_db()


# ============================================================
# 3. PRÉ-CONDIÇÕES
# ============================================================

review_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM reviews
    WHERE status = 'READY'
    """
).fetchone()[0]


article_version_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM article_versions
    WHERE parse_status = 'PARSED'
    """
).fetchone()[0]


review_errors = conn.execute(
    """
    SELECT COUNT(*)
    FROM reviews
    WHERE status != 'READY'
    """
).fetchone()[0]


article_parse_errors = conn.execute(
    """
    SELECT COUNT(*)
    FROM article_versions
    WHERE parse_status != 'PARSED'
    """
).fetchone()[0]


if review_count == 0:

    conn.close()

    raise RuntimeError(
        "Nenhuma Review READY encontrada."
    )


if article_version_count == 0:

    conn.close()

    raise RuntimeError(
        "Nenhuma ArticleVersion PARSED encontrada."
    )


if review_errors != 0:

    conn.close()

    raise RuntimeError(
        "Existem Reviews não READY: "
        f"{review_errors}"
    )


if article_parse_errors != 0:

    conn.close()

    raise RuntimeError(
        "Existem ArticleVersions não PARSED: "
        f"{article_parse_errors}"
    )


# ============================================================
# 4. CARREGAR RELAÇÕES
# ============================================================

rows = conn.execute(
    """
    SELECT

        r.review_doi,
        r.base_doi,
        r.source_container_doi,
        r.reviewed_doi,
        r.reviewed_version,
        r.recommendation_label,
        r.relation_to_container,

        a.article_doi,
        a.base_doi,
        a.article_version,
        a.parse_status

    FROM reviews AS r

    LEFT JOIN article_versions AS a
        ON a.article_doi = r.reviewed_doi

    WHERE r.status = 'READY'

    ORDER BY
        r.base_doi,
        r.reviewed_version,
        r.reviewed_doi,
        r.review_doi
    """
).fetchall()


if len(rows) != review_count:

    conn.close()

    raise RuntimeError(
        "JOIN Review→ArticleVersion alterou "
        "cardinalidade: "
        f"reviews={review_count}, "
        f"joined={len(rows)}"
    )


# ============================================================
# 5. VALIDAÇÃO PRÉVIA COMPLETA
# ============================================================

missing_articles = []
base_mismatches = []
version_mismatches = []
doi_mismatches = []
missing_recommendations = []
invalid_recommendations = []
invalid_relations = []


valid_recommendations = {
    "Approved",
    "Approved with Reservations",
    "Not Approved",
}


valid_relations = {
    "CURRENT",
    "HISTORICAL",
}


for row in rows:

    (
        review_doi,
        review_base_doi,
        source_container_doi,
        reviewed_doi,
        reviewed_version,
        recommendation_label,
        relation_to_container,

        article_doi,
        article_base_doi,
        article_version,
        parse_status,
    ) = row


    # --------------------------------------------------------
    # A. ArticleVersion existe
    # --------------------------------------------------------

    if article_doi is None:

        missing_articles.append(
            {
                "review_doi":
                    review_doi,

                "reviewed_doi":
                    reviewed_doi,
            }
        )

        continue


    # --------------------------------------------------------
    # B. DOI exato
    # --------------------------------------------------------

    if (
        normalize_doi(reviewed_doi)
        !=
        normalize_doi(article_doi)
    ):

        doi_mismatches.append(
            {
                "review_doi":
                    review_doi,

                "reviewed_doi":
                    reviewed_doi,

                "article_doi":
                    article_doi,
            }
        )


    # --------------------------------------------------------
    # C. base_doi
    # --------------------------------------------------------

    if (
        review_base_doi
        !=
        article_base_doi
    ):

        base_mismatches.append(
            {
                "review_doi":
                    review_doi,

                "review_base_doi":
                    review_base_doi,

                "article_base_doi":
                    article_base_doi,
            }
        )


    # --------------------------------------------------------
    # D. version
    # --------------------------------------------------------

    if (
        reviewed_version
        !=
        article_version
    ):

        version_mismatches.append(
            {
                "review_doi":
                    review_doi,

                "reviewed_version":
                    reviewed_version,

                "article_version":
                    article_version,
            }
        )


    # --------------------------------------------------------
    # E. recommendation
    # --------------------------------------------------------

    if recommendation_label is None:

        missing_recommendations.append(
            review_doi
        )


    elif (
        recommendation_label
        not in
        valid_recommendations
    ):

        invalid_recommendations.append(
            {
                "review_doi":
                    review_doi,

                "recommendation":
                    recommendation_label,
            }
        )


    # --------------------------------------------------------
    # F. CURRENT / HISTORICAL
    # --------------------------------------------------------

    if (
        relation_to_container
        not in
        valid_relations
    ):

        invalid_relations.append(
            {
                "review_doi":
                    review_doi,

                "relation":
                    relation_to_container,
            }
        )


precondition_errors = (

    len(missing_articles)
    +
    len(base_mismatches)
    +
    len(version_mismatches)
    +
    len(doi_mismatches)
    +
    len(missing_recommendations)
    +
    len(invalid_recommendations)
    +
    len(invalid_relations)
)


if precondition_errors != 0:

    print()
    print("=" * 72)
    print(
        "CANONICALPAIR PRECONDITION FAILURE"
    )
    print("=" * 72)

    print(
        f"Missing ArticleVersions.: "
        f"{len(missing_articles)}"
    )

    print(
        f"Base DOI mismatches.....: "
        f"{len(base_mismatches)}"
    )

    print(
        f"Version mismatches......: "
        f"{len(version_mismatches)}"
    )

    print(
        f"DOI mismatches..........: "
        f"{len(doi_mismatches)}"
    )

    print(
        f"Missing recommendations.: "
        f"{len(missing_recommendations)}"
    )

    print(
        f"Invalid recommendations.: "
        f"{len(invalid_recommendations)}"
    )

    print(
        f"Invalid relations.......: "
        f"{len(invalid_relations)}"
    )

    conn.close()

    raise RuntimeError(
        "CanonicalPairs não serão construídos "
        "porque as relações de origem não "
        "passaram na validação."
    )


# ============================================================
# 6. LIMPEZA CONTROLADA
# ============================================================

try:

    conn.execute(
        "BEGIN IMMEDIATE;"
    )


    conn.execute(
        """
        DELETE FROM canonical_pairs
        """
    )


    conn.commit()


except Exception:

    conn.rollback()
    conn.close()
    raise


# ============================================================
# 7. CONSTRUÇÃO
# ============================================================

print()
print("=" * 72)
print(
    "CONSTRUÇÃO DOS CANONICALPAIRS"
)
print("=" * 72)

print(
    f"Reviews READY..........: "
    f"{review_count}"
)

print(
    f"ArticleVersions PARSED.: "
    f"{article_version_count}"
)

print(
    f"Builder version........: "
    f"{CANONICAL_PAIR_BUILDER_VERSION}"
)

print()


inserted = 0

recommendation_counter = Counter()
relation_counter = Counter()
version_counter = Counter()


for index, row in enumerate(
    rows,
    start=1,
):

    (
        review_doi,
        base_doi,
        source_container_doi,
        reviewed_doi,
        reviewed_version,
        recommendation_label,
        relation_to_container,

        article_doi,
        article_base_doi,
        article_version,
        parse_status,
    ) = row


    pair_id = make_pair_id(
        review_doi
    )


    now = utc_now()


    conn.execute(
        """
        INSERT INTO canonical_pairs (

            pair_id,
            base_doi,
            article_doi,
            article_version,
            review_doi,
            recommendation_label,
            relation_to_container,
            source_container_doi,
            status,
            error_message,
            created_at,
            updated_at

        )
        VALUES (
            ?, ?, ?, ?, ?, ?, ?, ?,
            'READY',
            NULL,
            ?,
            ?
        )
        """,
        (
            pair_id,
            base_doi,
            article_doi,
            article_version,
            review_doi,
            recommendation_label,
            relation_to_container,
            source_container_doi,
            now,
            now,
        ),
    )


    inserted += 1


    recommendation_counter[
        recommendation_label
    ] += 1


    relation_counter[
        relation_to_container
    ] += 1


    version_counter[
        article_version
    ] += 1


    if (
        index == 1
        or
        index % 100 == 0
        or
        index == len(rows)
    ):

        print(
            f"[{index:>4}/{len(rows)}] "
            f"pairs={inserted}"
        )


conn.commit()


# ============================================================
# 8. CARDINALIDADE FINAL
# ============================================================

pair_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM canonical_pairs
    """
).fetchone()[0]


ready_pair_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM canonical_pairs
    WHERE status = 'READY'
    """
).fetchone()[0]


pair_error_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM canonical_pairs
    WHERE status != 'READY'
    """
).fetchone()[0]


# ============================================================
# 9. UNICIDADE REVIEW → PAIR
# ============================================================

duplicate_review_pairs = conn.execute(
    """
    SELECT COUNT(*)

    FROM (

        SELECT
            review_doi,
            COUNT(*) AS n

        FROM canonical_pairs

        GROUP BY review_doi

        HAVING COUNT(*) > 1
    )
    """
).fetchone()[0]


reviews_without_pair = conn.execute(
    """
    SELECT COUNT(*)

    FROM reviews AS r

    LEFT JOIN canonical_pairs AS p
        ON p.review_doi = r.review_doi

    WHERE
        r.status = 'READY'
        AND p.pair_id IS NULL
    """
).fetchone()[0]


pairs_without_review = conn.execute(
    """
    SELECT COUNT(*)

    FROM canonical_pairs AS p

    LEFT JOIN reviews AS r
        ON r.review_doi = p.review_doi

    WHERE r.review_doi IS NULL
    """
).fetchone()[0]


# ============================================================
# 10. PAIR → ARTICLEVERSION
# ============================================================

pairs_without_article = conn.execute(
    """
    SELECT COUNT(*)

    FROM canonical_pairs AS p

    LEFT JOIN article_versions AS a
        ON a.article_doi = p.article_doi

    WHERE a.article_doi IS NULL
    """
).fetchone()[0]


pairs_with_unparsed_article = conn.execute(
    """
    SELECT COUNT(*)

    FROM canonical_pairs AS p

    JOIN article_versions AS a
        ON a.article_doi = p.article_doi

    WHERE a.parse_status != 'PARSED'
    """
).fetchone()[0]


# ============================================================
# 11. SEMANTIC CONSISTENCY
# ============================================================

semantic_mismatches = conn.execute(
    """
    SELECT COUNT(*)

    FROM canonical_pairs AS p

    JOIN reviews AS r
        ON r.review_doi = p.review_doi

    JOIN article_versions AS a
        ON a.article_doi = p.article_doi

    WHERE

        p.base_doi != r.base_doi

        OR

        p.base_doi != a.base_doi

        OR

        p.article_doi != r.reviewed_doi

        OR

        p.article_version != r.reviewed_version

        OR

        p.article_version != a.article_version

        OR

        p.recommendation_label
            !=
        r.recommendation_label

        OR

        p.relation_to_container
            !=
        r.relation_to_container

        OR

        p.source_container_doi
            !=
        r.source_container_doi
    """
).fetchone()[0]


# ============================================================
# 12. CROSS-BASE RELATIONS
# ============================================================

cross_base_pairs = conn.execute(
    """
    SELECT COUNT(*)

    FROM canonical_pairs AS p

    JOIN reviews AS r
        ON r.review_doi = p.review_doi

    JOIN article_versions AS a
        ON a.article_doi = p.article_doi

    WHERE

        p.base_doi != r.base_doi

        OR

        p.base_doi != a.base_doi
    """
).fetchone()[0]


# ============================================================
# 13. DETERMINISTIC ID CHECK
# ============================================================

pair_id_mismatches = []


pair_id_rows = conn.execute(
    """
    SELECT
        pair_id,
        review_doi

    FROM canonical_pairs
    """
).fetchall()


for (
    stored_pair_id,
    review_doi,
) in pair_id_rows:

    expected_pair_id = make_pair_id(
        review_doi
    )


    if (
        stored_pair_id
        !=
        expected_pair_id
    ):

        pair_id_mismatches.append(
            {
                "review_doi":
                    review_doi,

                "stored":
                    stored_pair_id,

                "expected":
                    expected_pair_id,
            }
        )


# ============================================================
# 14. BASE ARTICLE COVERAGE
# ============================================================

base_articles_with_pairs = (
    conn.execute(
        """
        SELECT COUNT(
            DISTINCT base_doi
        )
        FROM canonical_pairs
        WHERE status = 'READY'
        """
    ).fetchone()[0]
)


structured_base_articles = (
    conn.execute(
        """
        SELECT COUNT(*)
        FROM base_articles
        WHERE review_structure_status = 'STRUCTURED'
        """
    ).fetchone()[0]
)


structured_without_pairs = (
    conn.execute(
        """
        SELECT COUNT(*)

        FROM base_articles AS b

        LEFT JOIN canonical_pairs AS p
            ON p.base_doi = b.base_doi

        WHERE
            b.review_structure_status = 'STRUCTURED'
            AND p.pair_id IS NULL
        """
    ).fetchone()[0]
)


nonstructured_with_pairs = (
    conn.execute(
        """
        SELECT COUNT(
            DISTINCT b.base_doi
        )

        FROM base_articles AS b

        JOIN canonical_pairs AS p
            ON p.base_doi = b.base_doi

        WHERE
            b.review_structure_status
            !=
            'STRUCTURED'
        """
    ).fetchone()[0]
)


# ============================================================
# 15. DISTRIBUIÇÕES DIRETAS DO SQLITE
# ============================================================

db_recommendation_distribution = (
    conn.execute(
        """
        SELECT
            recommendation_label,
            COUNT(*)

        FROM canonical_pairs

        GROUP BY recommendation_label

        ORDER BY recommendation_label
        """
    ).fetchall()
)


db_relation_distribution = (
    conn.execute(
        """
        SELECT
            relation_to_container,
            COUNT(*)

        FROM canonical_pairs

        GROUP BY relation_to_container

        ORDER BY relation_to_container
        """
    ).fetchall()
)


db_version_distribution = (
    conn.execute(
        """
        SELECT
            article_version,
            COUNT(*)

        FROM canonical_pairs

        GROUP BY article_version

        ORDER BY article_version
        """
    ).fetchall()
)


# ============================================================
# 16. SQLITE INTEGRITY
# ============================================================

integrity = [
    row[0]
    for row in conn.execute(
        "PRAGMA integrity_check;"
    ).fetchall()
]


foreign_key_errors = conn.execute(
    "PRAGMA foreign_key_check;"
).fetchall()


# ============================================================
# 17. PASS / FAIL
# ============================================================

canonical_pair_pass = (

    pair_count
    ==
    review_count

    and

    ready_pair_count
    ==
    review_count

    and

    pair_error_count
    ==
    0

    and

    duplicate_review_pairs
    ==
    0

    and

    reviews_without_pair
    ==
    0

    and

    pairs_without_review
    ==
    0

    and

    pairs_without_article
    ==
    0

    and

    pairs_with_unparsed_article
    ==
    0

    and

    semantic_mismatches
    ==
    0

    and

    cross_base_pairs
    ==
    0

    and

    len(pair_id_mismatches)
    ==
    0

    and

    structured_without_pairs
    ==
    0

    and

    nonstructured_with_pairs
    ==
    0

    and

    integrity
    ==
    ["ok"]

    and

    len(foreign_key_errors)
    ==
    0
)


# ============================================================
# 18. RUN INFO
# ============================================================

now = utc_now()


run_info_updates = {

    "canonical_pair_builder_version":
        CANONICAL_PAIR_BUILDER_VERSION,

    "canonical_pair_builder_status":
        (
            "PASS"
            if canonical_pair_pass
            else "FAIL"
        ),

    "canonical_pairs_ready":
        str(
            ready_pair_count
        ),

    "canonical_pair_base_articles":
        str(
            base_articles_with_pairs
        ),

    "canonical_pair_semantic_mismatches":
        str(
            semantic_mismatches
        ),

    "canonical_pair_id_rule":
        (
            "pair_ + first20hex("
            "sha256(normalized review_doi))"
        ),

    "canonical_pair_builder_completed_at":
        now,
}


for key, value in run_info_updates.items():

    conn.execute(
        """
        INSERT INTO run_info (
            key,
            value,
            updated_at
        )

        VALUES (?, ?, ?)

        ON CONFLICT(key)

        DO UPDATE SET
            value = excluded.value,
            updated_at = excluded.updated_at
        """,
        (
            key,
            value,
            now,
        ),
    )


conn.commit()


# ============================================================
# 19. FECHAR SQLITE
# ============================================================

conn.close()


# ============================================================
# 20. LOG
# ============================================================

log(
    "CanonicalPair construction: "
    f"reviews={review_count}, "
    f"pairs={pair_count}, "
    f"base_articles={base_articles_with_pairs}, "
    f"semantic_mismatches={semantic_mismatches}, "
    f"status="
    f"{'PASS' if canonical_pair_pass else 'FAIL'}."
)


# ============================================================
# 21. RELATÓRIO FINAL
# ============================================================

print()
print("=" * 72)
print(
    "F1000 CORPUS BUILDER 2.0 — "
    "CANONICAL PAIRS"
)
print("=" * 72)


print(
    f"Reviews READY..........: "
    f"{review_count}"
)

print(
    f"CanonicalPairs.........: "
    f"{pair_count}"
)

print(
    f"CanonicalPairs READY...: "
    f"{ready_pair_count}"
)

print(
    f"Pair errors............: "
    f"{pair_error_count}"
)


print()
print("Relational integrity:")

print(
    f"  Reviews without pair.: "
    f"{reviews_without_pair}"
)

print(
    f"  Pairs without review.: "
    f"{pairs_without_review}"
)

print(
    f"  Pairs without article: "
    f"{pairs_without_article}"
)

print(
    f"  Unparsed articles....: "
    f"{pairs_with_unparsed_article}"
)

print(
    f"  Duplicate review pair: "
    f"{duplicate_review_pairs}"
)

print(
    f"  Semantic mismatches..: "
    f"{semantic_mismatches}"
)

print(
    f"  Cross-base pairs.....: "
    f"{cross_base_pairs}"
)

print(
    f"  Pair ID mismatches...: "
    f"{len(pair_id_mismatches)}"
)


print()
print("BaseArticle coverage:")

print(
    f"  Structured BaseArticles: "
    f"{structured_base_articles}"
)

print(
    f"  BaseArticles with pairs: "
    f"{base_articles_with_pairs}"
)

print(
    f"  Structured without pair: "
    f"{structured_without_pairs}"
)

print(
    f"  Nonstructured with pair: "
    f"{nonstructured_with_pairs}"
)


print()
print("Recommendation distribution:")

for recommendation, count in (
    db_recommendation_distribution
):

    print(
        f"  {recommendation:<30}: "
        f"{count}"
    )


print()
print("Relation distribution:")

for relation, count in (
    db_relation_distribution
):

    print(
        f"  {relation:<20}: "
        f"{count}"
    )


print()
print("Reviewed article version distribution:")

for version, count in (
    db_version_distribution
):

    print(
        f"  V{version:<19}: "
        f"{count}"
    )


print()
print("SQLite validation:")

print(
    f"  Integrity............: "
    f"{integrity}"
)

print(
    f"  Foreign-key errors...: "
    f"{len(foreign_key_errors)}"
)


if pair_id_mismatches:

    print()
    print(
        "First Pair ID mismatches:"
    )

    for item in pair_id_mismatches[:20]:

        print(
            f"  {item}"
        )


print()
print(
    "HTTP requests performed: 0"
)

print()
print(
    "STATUS.................: "
    f"{'PASS' if canonical_pair_pass else 'FAIL'}"
)

print("=" * 72)
print(
    "CÉLULA 8 CONCLUÍDA"
)
print("=" * 72)


CONSTRUÇÃO DOS CANONICALPAIRS
Reviews READY..........: 1131
ArticleVersions PARSED.: 639
Builder version........: 2.0.0

[   1/1131] pairs=1
[ 100/1131] pairs=100
[ 200/1131] pairs=200
[ 300/1131] pairs=300
[ 400/1131] pairs=400
[ 500/1131] pairs=500
[ 600/1131] pairs=600
[ 700/1131] pairs=700
[ 800/1131] pairs=800
[ 900/1131] pairs=900
[1000/1131] pairs=1000
[1100/1131] pairs=1100
[1131/1131] pairs=1131
[2026-09-17T23:09:26.172368+00:00] CanonicalPair construction: reviews=1131, pairs=1131, base_articles=396, semantic_mismatches=0, status=PASS.

F1000 CORPUS BUILDER 2.0 — CANONICAL PAIRS
Reviews READY..........: 1131
CanonicalPairs.........: 1131
CanonicalPairs READY...: 1131
Pair errors............: 0

Relational integrity:
  Reviews without pair.: 0
  Pairs without review.: 0
  Pairs without article: 0
  Unparsed articles....: 0
  Duplicate review pair: 0
  Semantic mismatches..: 0
  Cross-base pairs.....: 0
  Pair ID mismatches...: 0

BaseArticle coverage:
  Structured BaseArticle

In [ ]:
# ============================================================
# CÉLULA 9 — KNOWLEDGE UNITS + MATERIALIZAÇÃO MARKDOWN
# F1000 Corpus Builder 2.0
# ============================================================
#
# RESPONSABILIDADE
#
# Agregar as entidades canônicas por base_doi e produzir:
#
#   corpus/<knowledge_unit_id>.md
#   curation/<knowledge_unit_id>.md
#
# KNOWLEDGE UNIT
#
# É a unidade mínima de análise e curadoria.
#
# Cada UK contém:
#
#   BaseArticle
#   ├── ArticleVersion(s) efetivamente revisadas
#   │   ├── artigo
#   │   ├── Review(s)
#   │   └── AuthorResponse(s)
#   └── latest observed peer-review state
#
# PRINCÍPIOS
#
# - CanonicalPair = relação atômica interna.
# - KnowledgeUnit = unidade analítica/curatorial.
# - Recommendation permanece associada à Review.
# - Não existe recommendation agregada da UK.
# - PeerReviewState é evidência observada no container.
# - Não inferimos editorial decision.
# - Não inferimos datas ausentes.
# - Não reparseamos XML.
# - Não fazemos HTTP.
# - Não fazemos avaliação automática de qualidade.
#
# MATERIALIZAÇÃO
#
# Apenas BaseArticles STRUCTURED com CanonicalPairs READY
# tornam-se Knowledge Units.
#
# Uma ArticleVersion entra na UK quando possui pelo menos
# um CanonicalPair naquele base_doi.
#
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import re

from pathlib import Path
from collections import Counter, defaultdict


KNOWLEDGE_UNIT_BUILDER_VERSION = "2.0.1"
MARKDOWN_MATERIALIZER_VERSION = "2.0.1"


# ============================================================
# 2. HELPERS
# ============================================================

def md_scalar(value):
    """
    Valor seguro para linha simples de Markdown.
    """

    if value is None:
        return "Not available"

    value = re.sub(
        r"\s+",
        " ",
        str(value),
    ).strip()

    return (
        value
        if value
        else "Not available"
    )


def md_table_cell(value):
    """
    Valor seguro para célula Markdown.
    """

    value = md_scalar(value)

    return (
        value
        .replace("|", "\\|")
        .replace("\n", " ")
    )


def normalize_output_markdown(text):
    """
    Remove espaços finais e excesso de linhas vazias.
    """

    lines = [
        line.rstrip()
        for line in str(text).splitlines()
    ]

    output = []
    previous_blank = False

    for line in lines:

        if line.strip():

            output.append(line)
            previous_blank = False

        else:

            if not previous_blank:
                output.append("")

            previous_blank = True

    while output and not output[0].strip():
        output.pop(0)

    while output and not output[-1].strip():
        output.pop()

    return "\n".join(output) + "\n"


def format_optional_date(value):

    if value is None:
        return "Not available"

    value = str(value).strip()

    return (
        value
        if value
        else "Not available"
    )


def render_state_counts(
    approved_count,
    reservations_count,
    not_approved_count,
):
    """
    Semântica preservada:

      NULL = não observado / não determinável
      0    = zero explicitamente observado
    """

    def render_count(value):

        if value is None:
            return "Not determinable"

        return str(value)

    return {
        "approved":
            render_count(approved_count),

        "reservations":
            render_count(reservations_count),

        "not_approved":
            render_count(not_approved_count),
    }


# ============================================================
# 3. SQLITE
# ============================================================

conn = open_builder_db()


# ============================================================
# 4. CARDINALIDADES DE ENTRADA
# ============================================================

base_article_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM base_articles
    """
).fetchone()[0]


ready_base_article_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM base_articles
    WHERE container_status = 'READY'
    """
).fetchone()[0]


structured_base_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM base_articles
    WHERE review_structure_status = 'STRUCTURED'
    """
).fetchone()[0]


parsed_article_version_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM article_versions
    WHERE parse_status = 'PARSED'
    """
).fetchone()[0]


review_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM reviews
    WHERE status = 'READY'
    """
).fetchone()[0]


response_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM author_responses
    WHERE status = 'READY'
    """
).fetchone()[0]


state_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM peer_review_states
    WHERE status = 'READY'
    """
).fetchone()[0]


pair_count = conn.execute(
    """
    SELECT COUNT(*)
    FROM canonical_pairs
    WHERE status = 'READY'
    """
).fetchone()[0]


paired_article_version_count = conn.execute(
    """
    SELECT COUNT(DISTINCT article_doi)
    FROM canonical_pairs
    WHERE status = 'READY'
    """
).fetchone()[0]


# ============================================================
# 5. PRÉ-CONDIÇÕES GERAIS
# ============================================================

general_errors = []


if base_article_count != TARGET_BASE_ARTICLES:

    general_errors.append(
        "BaseArticle cardinality differs from target: "
        f"target={TARGET_BASE_ARTICLES}, "
        f"actual={base_article_count}"
    )


if ready_base_article_count != base_article_count:

    general_errors.append(
        "Not all BaseArticles are READY: "
        f"total={base_article_count}, "
        f"ready={ready_base_article_count}"
    )


if parsed_article_version_count == 0:

    general_errors.append(
        "No parsed ArticleVersions."
    )


if review_count == 0:

    general_errors.append(
        "No READY Reviews."
    )


if pair_count != review_count:

    general_errors.append(
        "CanonicalPair cardinality differs from Reviews: "
        f"reviews={review_count}, "
        f"pairs={pair_count}"
    )


if state_count != base_article_count:

    general_errors.append(
        "PeerReviewState cardinality differs from BaseArticles: "
        f"states={state_count}, "
        f"base_articles={base_article_count}"
    )


if general_errors:

    print()
    print("=" * 72)
    print("GENERAL PRECONDITION FAILURE")
    print("=" * 72)

    for item in general_errors:
        print(f"- {item}")

    conn.close()

    raise RuntimeError(
        "Knowledge Unit materialization aborted."
    )


# ============================================================
# 6. VALIDAR AUTHORRESPONSES
# ============================================================

orphan_responses = conn.execute(
    """
    SELECT COUNT(*)

    FROM author_responses AS ar

    LEFT JOIN reviews AS r
        ON r.review_doi = ar.review_doi

    WHERE
        ar.status = 'READY'
        AND r.review_doi IS NULL
    """
).fetchone()[0]


response_cross_base = conn.execute(
    """
    SELECT COUNT(*)

    FROM author_responses AS ar

    JOIN reviews AS r
        ON r.review_doi = ar.review_doi

    WHERE
        ar.status = 'READY'
        AND ar.base_doi != r.base_doi
    """
).fetchone()[0]


duplicate_response_order = conn.execute(
    """
    SELECT COUNT(*)

    FROM (
        SELECT
            review_doi,
            response_order,
            COUNT(*) AS n

        FROM author_responses

        WHERE status = 'READY'

        GROUP BY
            review_doi,
            response_order

        HAVING COUNT(*) > 1
    )
    """
).fetchone()[0]


invalid_response_order = conn.execute(
    """
    SELECT COUNT(*)

    FROM author_responses

    WHERE
        status = 'READY'
        AND (
            response_order IS NULL
            OR response_order < 1
        )
    """
).fetchone()[0]


# ============================================================
# 7. VALIDAR CANONICALPAIRS
# ============================================================

pair_semantic_errors = conn.execute(
    """
    SELECT COUNT(*)

    FROM canonical_pairs AS p

    JOIN reviews AS r
        ON r.review_doi = p.review_doi

    JOIN article_versions AS a
        ON a.article_doi = p.article_doi

    WHERE
        p.status != 'READY'

        OR r.status != 'READY'

        OR a.parse_status != 'PARSED'

        OR p.base_doi != r.base_doi

        OR p.base_doi != a.base_doi

        OR p.article_doi != r.reviewed_doi

        OR p.article_version != r.reviewed_version

        OR p.article_version != a.article_version

        OR p.recommendation_label != r.recommendation_label

        OR p.relation_to_container != r.relation_to_container

        OR p.source_container_doi != r.source_container_doi
    """
).fetchone()[0]


structured_without_pair = conn.execute(
    """
    SELECT COUNT(*)

    FROM base_articles AS b

    LEFT JOIN canonical_pairs AS p
        ON
            p.base_doi = b.base_doi
            AND p.status = 'READY'

    WHERE
        b.review_structure_status = 'STRUCTURED'
        AND p.pair_id IS NULL
    """
).fetchone()[0]


nonstructured_with_pair = conn.execute(
    """
    SELECT COUNT(DISTINCT b.base_doi)

    FROM base_articles AS b

    JOIN canonical_pairs AS p
        ON
            p.base_doi = b.base_doi
            AND p.status = 'READY'

    WHERE
        b.review_structure_status != 'STRUCTURED'
    """
).fetchone()[0]


cross_base_article_pairs = conn.execute(
    """
    SELECT COUNT(*)

    FROM canonical_pairs AS p

    JOIN article_versions AS a
        ON a.article_doi = p.article_doi

    WHERE
        p.status = 'READY'
        AND p.base_doi != a.base_doi
    """
).fetchone()[0]


# ============================================================
# 8. ARTICLEVERSIONS PARSEADAS MAS NÃO PAREADAS
# ============================================================

unpaired_parsed_article_versions = conn.execute(
    """
    SELECT COUNT(*)

    FROM article_versions AS a

    LEFT JOIN canonical_pairs AS p
        ON
            p.article_doi = a.article_doi
            AND p.status = 'READY'

    WHERE
        a.parse_status = 'PARSED'
        AND p.pair_id IS NULL
    """
).fetchone()[0]


# ============================================================
# 9. VALIDAR PEER REVIEW STATES
# ============================================================

structured_without_state = conn.execute(
    """
    SELECT COUNT(*)

    FROM base_articles AS b

    LEFT JOIN peer_review_states AS s
        ON s.base_doi = b.base_doi

    WHERE
        b.review_structure_status = 'STRUCTURED'
        AND (
            s.state_id IS NULL
            OR s.status != 'READY'
        )
    """
).fetchone()[0]


duplicate_states_per_base = conn.execute(
    """
    SELECT COUNT(*)

    FROM (
        SELECT
            base_doi,
            COUNT(*) AS n

        FROM peer_review_states

        WHERE status = 'READY'

        GROUP BY base_doi

        HAVING COUNT(*) > 1
    )
    """
).fetchone()[0]


# ============================================================
# 10. BLOQUEIO RELACIONAL
# ============================================================

fatal_preconditions = {

    "orphan_responses":
        orphan_responses,

    "response_cross_base":
        response_cross_base,

    "duplicate_response_order":
        duplicate_response_order,

    "invalid_response_order":
        invalid_response_order,

    "pair_semantic_errors":
        pair_semantic_errors,

    "structured_without_pair":
        structured_without_pair,

    "nonstructured_with_pair":
        nonstructured_with_pair,

    "cross_base_article_pairs":
        cross_base_article_pairs,

    "structured_without_state":
        structured_without_state,

    "duplicate_states_per_base":
        duplicate_states_per_base,
}


fatal_error_total = sum(
    fatal_preconditions.values()
)


if fatal_error_total != 0:

    print()
    print("=" * 72)
    print("KNOWLEDGE UNIT PRECONDITION FAILURE")
    print("=" * 72)

    for key, value in fatal_preconditions.items():
        print(
            f"{key:<32}: {value}"
        )

    conn.close()

    raise RuntimeError(
        "Knowledge Units não serão materializadas "
        "porque existem inconsistências relacionais."
    )


# ============================================================
# 11. IDENTIFICAR KNOWLEDGE UNITS
# ============================================================

uk_base_rows = conn.execute(
    """
    SELECT
        b.base_doi,
        b.container_doi,
        b.container_version,
        b.discovery_order,
        b.review_structure_status

    FROM base_articles AS b

    WHERE
        b.review_structure_status = 'STRUCTURED'

        AND EXISTS (
            SELECT 1
            FROM canonical_pairs AS p
            WHERE
                p.base_doi = b.base_doi
                AND p.status = 'READY'
        )

    ORDER BY
        b.discovery_order,
        b.base_doi
    """
).fetchall()


expected_uk_count = len(
    uk_base_rows
)


if expected_uk_count != structured_base_count:

    conn.close()

    raise RuntimeError(
        "Structured BaseArticle cardinality differs "
        "from KnowledgeUnit candidate cardinality: "
        f"structured={structured_base_count}, "
        f"uk_candidates={expected_uk_count}."
    )


# ============================================================
# 12. CARREGAR ARTICLEVERSIONS PAREADAS
# ============================================================

article_rows = conn.execute(
    """
    SELECT DISTINCT

        a.article_doi,
        a.base_doi,
        a.article_version,
        a.source_kind,
        a.article_date,
        a.title,
        a.abstract_text,
        a.body_text,
        a.abstract_word_count,
        a.body_word_count,
        a.article_word_count,
        a.article_character_count,
        a.section_count,
        a.table_count,
        a.figure_count,
        a.reference_count

    FROM article_versions AS a

    JOIN canonical_pairs AS p
        ON
            p.article_doi = a.article_doi
            AND p.status = 'READY'

    WHERE
        a.parse_status = 'PARSED'

    ORDER BY
        a.base_doi,
        a.article_version,
        a.article_doi
    """
).fetchall()


articles_by_base = defaultdict(list)


for row in article_rows:

    articles_by_base[
        row[1]
    ].append(row)


# ============================================================
# 13. CARREGAR REVIEWS + CANONICALPAIRS
#
# SCHEMA FÍSICO CONFIRMADO NA C9A:
#
#   review_character_count
#   review_word_count
#
# ============================================================

review_rows = conn.execute(
    """
    SELECT

        p.pair_id,
        p.base_doi,
        p.article_doi,
        p.article_version,
        p.review_doi,
        p.recommendation_label,
        p.relation_to_container,
        p.source_container_doi,

        r.report_id,
        r.review_date,
        r.review_text,
        r.review_word_count,
        r.review_character_count

    FROM canonical_pairs AS p

    JOIN reviews AS r
        ON r.review_doi = p.review_doi

    WHERE
        p.status = 'READY'
        AND r.status = 'READY'

    ORDER BY
        p.base_doi,
        p.article_version,
        p.article_doi,
        CASE
            WHEN r.review_date IS NULL THEN 1
            ELSE 0
        END,
        r.review_date,
        p.review_doi
    """
).fetchall()


reviews_by_article = defaultdict(list)


for row in review_rows:

    reviews_by_article[
        row[2]
    ].append(row)


# ============================================================
# 14. CARREGAR AUTHORRESPONSES
#
# SCHEMA FÍSICO CONFIRMADO NA C9A:
#
#   response_character_count
#   response_word_count
#
# ============================================================

response_rows = conn.execute(
    """
    SELECT

        response_id,
        review_doi,
        base_doi,
        source_container_doi,
        response_order,
        response_date,
        response_text,
        response_word_count,
        response_character_count

    FROM author_responses

    WHERE status = 'READY'

    ORDER BY
        review_doi,
        response_order,
        response_id
    """
).fetchall()


responses_by_review = defaultdict(list)


for row in response_rows:

    responses_by_review[
        row[1]
    ].append(row)


# ============================================================
# 15. CARREGAR PEER REVIEW STATES
# ============================================================

state_rows = conn.execute(
    """
    SELECT

        state_id,
        base_doi,
        container_doi,
        article_version,
        state_raw,
        approved_count,
        reservations_count,
        not_approved_count,
        not_peer_reviewed,
        status

    FROM peer_review_states

    WHERE status = 'READY'
    """
).fetchall()


states_by_base = {}


for row in state_rows:

    base_doi = row[1]

    if base_doi in states_by_base:

        conn.close()

        raise RuntimeError(
            "Multiple READY PeerReviewStates for "
            f"{base_doi}."
        )

    states_by_base[
        base_doi
    ] = row


# ============================================================
# 16. VALIDAÇÃO DA MONTAGEM EM MEMÓRIA
# ============================================================

memory_validation_errors = []


for base_row in uk_base_rows:

    (
        base_doi,
        container_doi,
        container_version,
        discovery_order,
        review_structure_status,
    ) = base_row


    article_list = articles_by_base.get(
        base_doi,
        [],
    )


    if not article_list:

        memory_validation_errors.append(
            (
                base_doi,
                "NO_ARTICLE_VERSION",
            )
        )

        continue


    if base_doi not in states_by_base:

        memory_validation_errors.append(
            (
                base_doi,
                "NO_PEER_REVIEW_STATE",
            )
        )


    for article in article_list:

        article_doi = article[0]


        review_list = reviews_by_article.get(
            article_doi,
            [],
        )


        if not review_list:

            memory_validation_errors.append(
                (
                    base_doi,
                    "ARTICLE_WITHOUT_REVIEW",
                    article_doi,
                )
            )


if memory_validation_errors:

    print()
    print("First in-memory validation errors:")

    for item in memory_validation_errors[:20]:
        print(f"  {item}")

    conn.close()

    raise RuntimeError(
        "Knowledge Unit in-memory assembly "
        "validation failed."
    )


# ============================================================
# 17. FUNÇÃO — CORPUS MARKDOWN
# ============================================================

def build_corpus_markdown(
    base_row,
    article_list,
    state_row,
):

    (
        base_doi,
        container_doi,
        container_version,
        discovery_order,
        review_structure_status,
    ) = base_row


    knowledge_unit_id = make_knowledge_unit_id(
        base_doi
    )


    total_reviews = sum(
        len(
            reviews_by_article.get(
                article[0],
                [],
            )
        )
        for article in article_list
    )


    total_responses = sum(
        len(
            responses_by_review.get(
                review[4],
                [],
            )
        )
        for article in article_list
        for review in reviews_by_article.get(
            article[0],
            [],
        )
    )


    lines = []


    # --------------------------------------------------------
    # H1
    # --------------------------------------------------------

    lines.append(
        f"# Knowledge Unit — {base_doi}"
    )

    lines.append("")


    # --------------------------------------------------------
    # IDENTITY
    # --------------------------------------------------------

    lines.append(
        "## Identity"
    )

    lines.append("")

    lines.append(
        f"- Knowledge Unit ID: `{knowledge_unit_id}`"
    )

    lines.append(
        f"- Base DOI: `{base_doi}`"
    )

    lines.append(
        f"- Observed container DOI: `{container_doi}`"
    )

    lines.append(
        f"- Observed container version: {container_version}"
    )

    lines.append(
        f"- Discovery order: {discovery_order}"
    )

    lines.append(
        f"- Review structure: {review_structure_status}"
    )

    lines.append(
        f"- Article versions represented: {len(article_list)}"
    )

    lines.append(
        f"- Reviews represented: {total_reviews}"
    )

    lines.append(
        f"- Author responses represented: {total_responses}"
    )

    lines.append("")


    # --------------------------------------------------------
    # TRAJECTORY
    # --------------------------------------------------------

    lines.append(
        "## Peer-review trajectory"
    )

    lines.append("")


    # ========================================================
    # ARTICLE VERSIONS
    # ========================================================

    for article in article_list:

        (
            article_doi,
            article_base_doi,
            article_version,
            source_kind,
            article_date,
            title,
            abstract_text,
            body_text,
            abstract_word_count,
            body_word_count,
            article_word_count,
            article_character_count,
            section_count,
            table_count,
            figure_count,
            reference_count,
        ) = article


        review_list = reviews_by_article.get(
            article_doi,
            [],
        )


        # ----------------------------------------------------
        # H3 — VERSION
        # ----------------------------------------------------

        lines.append(
            f"### Article Version {article_version}"
        )

        lines.append("")


        # ----------------------------------------------------
        # H4 — ARTICLE
        # ----------------------------------------------------

        lines.append(
            "#### Article"
        )

        lines.append("")

        lines.append(
            f"- DOI: `{article_doi}`"
        )

        lines.append(
            f"- Date: {format_optional_date(article_date)}"
        )

        lines.append(
            f"- Source: {source_kind}"
        )

        lines.append(
            f"- Word count: {article_word_count}"
        )

        lines.append(
            f"- Sections: {section_count}"
        )

        lines.append(
            f"- Tables: {table_count}"
        )

        lines.append(
            f"- Figures: {figure_count}"
        )

        lines.append(
            f"- References: {reference_count}"
        )

        lines.append("")


        # ----------------------------------------------------
        # H5 — TITLE
        # ----------------------------------------------------

        lines.append(
            "##### Title"
        )

        lines.append("")

        lines.append(
            title or "Not available"
        )

        lines.append("")


        # ----------------------------------------------------
        # H5 — ABSTRACT
        # ----------------------------------------------------

        lines.append(
            "##### Abstract"
        )

        lines.append("")

        lines.append(
            abstract_text or "Not available"
        )

        lines.append("")


        # ----------------------------------------------------
        # H5 — ARTICLE BODY
        #
        # body_text já foi produzido pela C7.
        # Não há novo parsing XML aqui.
        # ----------------------------------------------------

        lines.append(
            "##### Article body"
        )

        lines.append("")

        lines.append(
            body_text or "Not available"
        )

        lines.append("")


        # ----------------------------------------------------
        # H4 — REVIEWS
        # ----------------------------------------------------

        lines.append(
            "#### Reviews"
        )

        lines.append("")


        for review_index, review in enumerate(
            review_list,
            start=1,
        ):

            (
                pair_id,
                review_base_doi,
                reviewed_article_doi,
                reviewed_version,
                review_doi,
                recommendation_label,
                relation_to_container,
                source_container_doi,
                report_id,
                review_date,
                review_text,
                review_word_count,
                review_character_count,
            ) = review


            response_list = responses_by_review.get(
                review_doi,
                [],
            )


            # ------------------------------------------------
            # H5 — REVIEW
            # ------------------------------------------------

            lines.append(
                f"##### Review {review_index}"
            )

            lines.append("")

            lines.append(
                f"- Review ID: `{review_doi}`"
            )

            lines.append(
                f"- Canonical Pair ID: `{pair_id}`"
            )

            lines.append(
                f"- Reviewed article DOI: "
                f"`{reviewed_article_doi}`"
            )

            lines.append(
                f"- Reviewed version: {reviewed_version}"
            )

            lines.append(
                f"- Review date: "
                f"{format_optional_date(review_date)}"
            )

            lines.append(
                f"- Recommendation: "
                f"{recommendation_label}"
            )

            lines.append(
                f"- Relation to observed container: "
                f"{relation_to_container}"
            )

            lines.append(
                f"- Source container DOI: "
                f"`{source_container_doi}`"
            )

            lines.append(
                f"- Review word count: "
                f"{review_word_count}"
            )

            lines.append("")

            lines.append(
                review_text
            )

            lines.append("")


            # ------------------------------------------------
            # H6 — AUTHOR RESPONSES
            # ------------------------------------------------

            for response in response_list:

                (
                    response_id,
                    response_review_doi,
                    response_base_doi,
                    response_source_container,
                    response_order,
                    response_date,
                    response_text,
                    response_word_count,
                    response_character_count,
                ) = response


                if len(response_list) == 1:

                    response_heading = (
                        "###### Author Response"
                    )

                else:

                    response_heading = (
                        "###### Author Response "
                        f"{response_order}"
                    )


                lines.append(
                    response_heading
                )

                lines.append("")

                lines.append(
                    f"- Response ID: `{response_id}`"
                )

                lines.append(
                    f"- Response date: "
                    f"{format_optional_date(response_date)}"
                )

                lines.append(
                    f"- Response word count: "
                    f"{response_word_count}"
                )

                lines.append("")

                lines.append(
                    response_text
                )

                lines.append("")


    # ========================================================
    # LATEST OBSERVED PEER-REVIEW STATE
    # ========================================================

    (
        state_id,
        state_base_doi,
        state_container_doi,
        state_article_version,
        state_raw,
        approved_count,
        reservations_count,
        not_approved_count,
        not_peer_reviewed,
        state_status,
    ) = state_row


    state_counts = render_state_counts(
        approved_count,
        reservations_count,
        not_approved_count,
    )


    lines.append(
        "## Latest observed peer-review state"
    )

    lines.append("")

    lines.append(
        f"- State ID: `{state_id}`"
    )

    lines.append(
        f"- Observed at container DOI: "
        f"`{state_container_doi}`"
    )

    lines.append(
        f"- Observed article version: "
        f"{state_article_version}"
    )

    lines.append(
        f"- Approved count: "
        f"{state_counts['approved']}"
    )

    lines.append(
        f"- Approved with Reservations count: "
        f"{state_counts['reservations']}"
    )

    lines.append(
        f"- Not Approved count: "
        f"{state_counts['not_approved']}"
    )

    lines.append(
        f"- Explicitly not peer reviewed: "
        f"{'Yes' if not_peer_reviewed else 'No'}"
    )

    lines.append("")


    # --------------------------------------------------------
    # SOURCE STATE
    # --------------------------------------------------------

    lines.append(
        "### Source state"
    )

    lines.append("")

    lines.append(
        state_raw
        if state_raw
        else "Not available"
    )

    lines.append("")


    # --------------------------------------------------------
    # PROVENANCE
    # --------------------------------------------------------

    lines.append(
        "## Provenance note"
    )

    lines.append("")

    lines.append(
        "This Knowledge Unit was assembled "
        "deterministically from exact article versions, "
        "structured peer-review reports, author responses, "
        "and the observed peer-review state preserved by "
        "F1000 Corpus Builder 2.0. Recommendations remain "
        "attributes of individual reviews. The observed "
        "peer-review state is preserved as source evidence "
        "and must not be interpreted as an inferred "
        "editorial decision."
    )

    lines.append("")


    return normalize_output_markdown(
        "\n".join(lines)
    )


# ============================================================
# 18. FUNÇÃO — CURATION MARKDOWN
# ============================================================

def build_curation_markdown(
    base_row,
    article_list,
    state_row,
):

    (
        base_doi,
        container_doi,
        container_version,
        discovery_order,
        review_structure_status,
    ) = base_row


    knowledge_unit_id = make_knowledge_unit_id(
        base_doi
    )


    corpus_relative_path = (
        f"../corpus/{knowledge_unit_id}.md"
    )


    all_reviews = []
    total_responses = 0


    for article in article_list:

        article_doi = article[0]
        article_version = article[2]


        for review in reviews_by_article.get(
            article_doi,
            [],
        ):

            response_list = responses_by_review.get(
                review[4],
                [],
            )


            total_responses += len(
                response_list
            )


            all_reviews.append(
                (
                    article_version,
                    article_doi,
                    review,
                    response_list,
                )
            )


    (
        state_id,
        state_base_doi,
        state_container_doi,
        state_article_version,
        state_raw,
        approved_count,
        reservations_count,
        not_approved_count,
        not_peer_reviewed,
        state_status,
    ) = state_row


    lines = []


    # --------------------------------------------------------
    # H1
    # --------------------------------------------------------

    lines.append(
        f"# Curation — {base_doi}"
    )

    lines.append("")


    # --------------------------------------------------------
    # IDENTITY
    # --------------------------------------------------------

    lines.append(
        "## Identity"
    )

    lines.append("")

    lines.append(
        f"- Knowledge Unit ID: `{knowledge_unit_id}`"
    )

    lines.append(
        f"- Base DOI: `{base_doi}`"
    )

    lines.append(
        f"- Corpus document: `{corpus_relative_path}`"
    )

    lines.append(
        f"- Discovery order: {discovery_order}"
    )

    lines.append("")


    # --------------------------------------------------------
    # TRAJECTORY SUMMARY
    # --------------------------------------------------------

    lines.append(
        "## Trajectory summary"
    )

    lines.append("")

    lines.append(
        f"- Article versions represented: "
        f"{len(article_list)}"
    )

    lines.append(
        f"- Reviews: {len(all_reviews)}"
    )

    lines.append(
        f"- Author responses: {total_responses}"
    )

    lines.append(
        f"- Latest observed container: "
        f"`{state_container_doi}`"
    )

    lines.append(
        f"- Latest observed version: "
        f"{state_article_version}"
    )

    lines.append(
        f"- Review structure: "
        f"{review_structure_status}"
    )

    lines.append("")


    # --------------------------------------------------------
    # REVIEW MAP
    # --------------------------------------------------------

    lines.append(
        "## Review map"
    )

    lines.append("")

    lines.append(
        "| Version | Article DOI | Review ID | "
        "Recommendation | Relation | "
        "Review date | Response(s) |"
    )

    lines.append(
        "| --- | --- | --- | --- | --- | --- | ---: |"
    )


    for (
        article_version,
        article_doi,
        review,
        response_list,
    ) in all_reviews:

        (
            pair_id,
            review_base_doi,
            reviewed_article_doi,
            reviewed_version,
            review_doi,
            recommendation_label,
            relation_to_container,
            source_container_doi,
            report_id,
            review_date,
            review_text,
            review_word_count,
            review_character_count,
        ) = review


        lines.append(
            "| "
            f"V{article_version} | "
            f"{md_table_cell(article_doi)} | "
            f"{md_table_cell(review_doi)} | "
            f"{md_table_cell(recommendation_label)} | "
            f"{md_table_cell(relation_to_container)} | "
            f"{md_table_cell(format_optional_date(review_date))} | "
            f"{len(response_list)} |"
        )


    lines.append("")


    # --------------------------------------------------------
    # OBSERVED STATE
    # --------------------------------------------------------

    state_counts = render_state_counts(
        approved_count,
        reservations_count,
        not_approved_count,
    )


    lines.append(
        "## Observed peer-review state"
    )

    lines.append("")

    lines.append(
        f"- Container DOI: `{state_container_doi}`"
    )

    lines.append(
        f"- Version: {state_article_version}"
    )

    lines.append(
        f"- Approved: "
        f"{state_counts['approved']}"
    )

    lines.append(
        f"- Approved with Reservations: "
        f"{state_counts['reservations']}"
    )

    lines.append(
        f"- Not Approved: "
        f"{state_counts['not_approved']}"
    )

    lines.append(
        f"- Explicitly not peer reviewed: "
        f"{'Yes' if not_peer_reviewed else 'No'}"
    )

    lines.append("")

    lines.append(
        "> Source state: "
        f"{md_scalar(state_raw)}"
    )

    lines.append("")


    # --------------------------------------------------------
    # CURATION DECISION
    # --------------------------------------------------------

    lines.append(
        "## Curation decision"
    )

    lines.append("")

    lines.append(
        "- [ ] INCLUDE"
    )

    lines.append(
        "- [ ] EXCLUDE"
    )

    lines.append("")


    # --------------------------------------------------------
    # EXCLUSION REASON
    # --------------------------------------------------------

    lines.append(
        "## Exclusion reason"
    )

    lines.append("")

    lines.append(
        "_Complete only if the Knowledge Unit is excluded._"
    )

    lines.append("")


    # --------------------------------------------------------
    # ELEMENT FLAGS
    # --------------------------------------------------------

    lines.append(
        "## Element flags"
    )

    lines.append("")

    lines.append(
        "_Record any individual article version, review, "
        "or author response that requires attention or "
        "possible exclusion without changing the primary "
        "curation unit._"
    )

    lines.append("")

    lines.append(
        "| Element type | Element ID | Flag | Notes |"
    )

    lines.append(
        "| --- | --- | --- | --- |"
    )

    lines.append(
        "|  |  |  |  |"
    )

    lines.append("")


    # --------------------------------------------------------
    # CURATOR NOTES
    # --------------------------------------------------------

    lines.append(
        "## Curator notes"
    )

    lines.append("")

    lines.append("")


    # --------------------------------------------------------
    # CURATOR
    # --------------------------------------------------------

    lines.append(
        "## Curator"
    )

    lines.append("")

    lines.append("")


    # --------------------------------------------------------
    # CURATION DATE
    # --------------------------------------------------------

    lines.append(
        "## Curation date"
    )

    lines.append("")

    lines.append("")


    return normalize_output_markdown(
        "\n".join(lines)
    )


# ============================================================
# 19. RELATÓRIO INICIAL
# ============================================================

print()
print("=" * 72)
print("MONTAGEM DAS KNOWLEDGE UNITS")
print("=" * 72)

print(
    f"BaseArticles............: "
    f"{base_article_count}"
)

print(
    f"Structured BaseArticles.: "
    f"{structured_base_count}"
)

print(
    f"KnowledgeUnit candidates: "
    f"{expected_uk_count}"
)

print(
    f"Parsed ArticleVersions..: "
    f"{parsed_article_version_count}"
)

print(
    f"Paired ArticleVersions..: "
    f"{paired_article_version_count}"
)

print(
    f"CanonicalPairs..........: "
    f"{pair_count}"
)

print(
    f"AuthorResponses.........: "
    f"{response_count}"
)

print(
    f"Materializer version....: "
    f"{MARKDOWN_MATERIALIZER_VERSION}"
)

print()


# ============================================================
# 20. LIMPEZA CONTROLADA DOS MARKDOWN DERIVADOS
# ============================================================

CORPUS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


CURATION_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


existing_corpus_md = list(
    CORPUS_DIR.glob("*.md")
)


existing_curation_md = list(
    CURATION_DIR.glob("*.md")
)


for path in existing_corpus_md:
    path.unlink()


for path in existing_curation_md:
    path.unlink()


print(
    f"Previous corpus MD removed..: "
    f"{len(existing_corpus_md)}"
)

print(
    f"Previous curation MD removed: "
    f"{len(existing_curation_md)}"
)

print()


# ============================================================
# 21. MATERIALIZAÇÃO
# ============================================================

materialized_uks = 0
corpus_files_written = 0
curation_files_written = 0

materialized_article_versions = set()
materialized_review_ids = set()
materialized_response_ids = set()

knowledge_unit_ids = set()
knowledge_unit_base_dois = set()

file_errors = []

uk_article_version_distribution = Counter()
uk_review_distribution = Counter()
uk_response_distribution = Counter()


for index, base_row in enumerate(
    uk_base_rows,
    start=1,
):

    (
        base_doi,
        container_doi,
        container_version,
        discovery_order,
        review_structure_status,
    ) = base_row


    knowledge_unit_id = make_knowledge_unit_id(
        base_doi
    )


    if knowledge_unit_id in knowledge_unit_ids:

        conn.close()

        raise RuntimeError(
            "Duplicate deterministic KnowledgeUnit ID: "
            f"{knowledge_unit_id}"
        )


    if base_doi in knowledge_unit_base_dois:

        conn.close()

        raise RuntimeError(
            "Duplicate KnowledgeUnit base DOI: "
            f"{base_doi}"
        )


    knowledge_unit_ids.add(
        knowledge_unit_id
    )

    knowledge_unit_base_dois.add(
        base_doi
    )


    article_list = articles_by_base[
        base_doi
    ]


    state_row = states_by_base[
        base_doi
    ]


    corpus_markdown = build_corpus_markdown(
        base_row,
        article_list,
        state_row,
    )


    curation_markdown = build_curation_markdown(
        base_row,
        article_list,
        state_row,
    )


    corpus_path = (
        CORPUS_DIR
        /
        f"{knowledge_unit_id}.md"
    )


    curation_path = (
        CURATION_DIR
        /
        f"{knowledge_unit_id}.md"
    )


    try:

        corpus_path.write_text(
            corpus_markdown,
            encoding="utf-8",
        )


        curation_path.write_text(
            curation_markdown,
            encoding="utf-8",
        )


    except Exception as exc:

        file_errors.append(
            {
                "base_doi":
                    base_doi,

                "knowledge_unit_id":
                    knowledge_unit_id,

                "error":
                    f"{type(exc).__name__}: {exc}",
            }
        )

        raise


    materialized_uks += 1
    corpus_files_written += 1
    curation_files_written += 1


    article_count_for_uk = len(
        article_list
    )

    review_count_for_uk = 0
    response_count_for_uk = 0


    for article in article_list:

        article_doi = article[0]

        materialized_article_versions.add(
            article_doi
        )


        review_list = reviews_by_article.get(
            article_doi,
            [],
        )


        review_count_for_uk += len(
            review_list
        )


        for review in review_list:

            review_doi = review[4]

            materialized_review_ids.add(
                review_doi
            )


            response_list = responses_by_review.get(
                review_doi,
                [],
            )


            response_count_for_uk += len(
                response_list
            )


            for response in response_list:

                materialized_response_ids.add(
                    response[0]
                )


    uk_article_version_distribution[
        article_count_for_uk
    ] += 1


    uk_review_distribution[
        review_count_for_uk
    ] += 1


    uk_response_distribution[
        response_count_for_uk
    ] += 1


    if (
        index == 1
        or index % 25 == 0
        or index == expected_uk_count
    ):

        print(
            f"[{index:>3}/{expected_uk_count}] "
            f"UKs={materialized_uks} | "
            f"corpus={corpus_files_written} | "
            f"curation={curation_files_written}"
        )


# ============================================================
# 22. INSPEÇÃO FÍSICA DOS ARQUIVOS
# ============================================================

corpus_files = sorted(
    CORPUS_DIR.glob("*.md")
)


curation_files = sorted(
    CURATION_DIR.glob("*.md")
)


corpus_names = {
    path.name
    for path in corpus_files
}


curation_names = {
    path.name
    for path in curation_files
}


missing_curation_twins = sorted(
    corpus_names - curation_names
)


missing_corpus_twins = sorted(
    curation_names - corpus_names
)


empty_corpus_files = [
    path.name
    for path in corpus_files
    if path.stat().st_size == 0
]


empty_curation_files = [
    path.name
    for path in curation_files
    if path.stat().st_size == 0
]


expected_file_names = {
    make_knowledge_unit_id(
        base_doi
    ) + ".md"

    for base_doi in knowledge_unit_base_dois
}


unexpected_corpus_files = sorted(
    corpus_names - expected_file_names
)


unexpected_curation_files = sorted(
    curation_names - expected_file_names
)


missing_expected_corpus_files = sorted(
    expected_file_names - corpus_names
)


missing_expected_curation_files = sorted(
    expected_file_names - curation_names
)


# ============================================================
# 23. VALIDAR CORPUS MARKDOWN
# ============================================================

corpus_identity_errors = []
corpus_required_section_errors = []
corpus_h1_errors = []


required_corpus_sections = [
    "## Identity",
    "## Peer-review trajectory",
    "## Latest observed peer-review state",
    "## Provenance note",
]


for base_row in uk_base_rows:

    base_doi = base_row[0]

    ku_id = make_knowledge_unit_id(
        base_doi
    )


    path = (
        CORPUS_DIR
        /
        f"{ku_id}.md"
    )


    text = path.read_text(
        encoding="utf-8"
    )


    expected_h1 = (
        f"# Knowledge Unit — {base_doi}"
    )


    if not text.startswith(expected_h1):

        corpus_identity_errors.append(
            path.name
        )


    for required_section in required_corpus_sections:

        if required_section not in text:

            corpus_required_section_errors.append(
                (
                    path.name,
                    required_section,
                )
            )


    h1_lines = re.findall(
        r"(?m)^#\s+.+$",
        text,
    )


    if len(h1_lines) != 1:

        corpus_h1_errors.append(
            (
                path.name,
                len(h1_lines),
            )
        )


# ============================================================
# 24. VALIDAR CURATION MARKDOWN
# ============================================================

curation_identity_errors = []
curation_required_section_errors = []
curation_decision_prefilled = []


required_curation_sections = [
    "## Identity",
    "## Trajectory summary",
    "## Review map",
    "## Observed peer-review state",
    "## Curation decision",
    "## Exclusion reason",
    "## Element flags",
    "## Curator notes",
    "## Curator",
    "## Curation date",
]


for base_row in uk_base_rows:

    base_doi = base_row[0]

    ku_id = make_knowledge_unit_id(
        base_doi
    )


    path = (
        CURATION_DIR
        /
        f"{ku_id}.md"
    )


    text = path.read_text(
        encoding="utf-8"
    )


    expected_h1 = (
        f"# Curation — {base_doi}"
    )


    if not text.startswith(expected_h1):

        curation_identity_errors.append(
            path.name
        )


    for required_section in required_curation_sections:

        if required_section not in text:

            curation_required_section_errors.append(
                (
                    path.name,
                    required_section,
                )
            )


    lower_text = text.lower()


    if (
        "- [x] include" in lower_text
        or "- [x] exclude" in lower_text
    ):

        curation_decision_prefilled.append(
            path.name
        )


# ============================================================
# 25. COVERAGE DOS ELEMENTOS
# ============================================================

all_ready_review_ids = {
    row[0]
    for row in conn.execute(
        """
        SELECT review_doi
        FROM reviews
        WHERE status = 'READY'
        """
    ).fetchall()
}


all_ready_response_ids = {
    row[0]
    for row in conn.execute(
        """
        SELECT response_id
        FROM author_responses
        WHERE status = 'READY'
        """
    ).fetchall()
}


all_paired_article_dois = {
    row[0]
    for row in conn.execute(
        """
        SELECT DISTINCT article_doi
        FROM canonical_pairs
        WHERE status = 'READY'
        """
    ).fetchall()
}


reviews_not_materialized = (
    all_ready_review_ids
    -
    materialized_review_ids
)


responses_not_materialized = (
    all_ready_response_ids
    -
    materialized_response_ids
)


paired_articles_not_materialized = (
    all_paired_article_dois
    -
    materialized_article_versions
)


# ============================================================
# 26. FILE SIZE + HASH
# ============================================================

corpus_total_bytes = sum(
    path.stat().st_size
    for path in corpus_files
)


curation_total_bytes = sum(
    path.stat().st_size
    for path in curation_files
)


corpus_hashes = {
    path.name:
        sha256_file(path)

    for path in corpus_files
}


curation_hashes = {
    path.name:
        sha256_file(path)

    for path in curation_files
}


duplicate_corpus_hashes = (
    len(corpus_hashes)
    -
    len(set(corpus_hashes.values()))
)


duplicate_curation_hashes = (
    len(curation_hashes)
    -
    len(set(curation_hashes.values()))
)


# ============================================================
# 27. SQLITE INTEGRITY
# ============================================================

integrity = [
    row[0]
    for row in conn.execute(
        "PRAGMA integrity_check;"
    ).fetchall()
]


foreign_key_errors = conn.execute(
    "PRAGMA foreign_key_check;"
).fetchall()


# ============================================================
# 28. PASS / FAIL
# ============================================================

materialization_pass = (

    materialized_uks
    ==
    expected_uk_count

    and

    len(corpus_files)
    ==
    expected_uk_count

    and

    len(curation_files)
    ==
    expected_uk_count

    and

    corpus_names
    ==
    curation_names

    and

    len(empty_corpus_files)
    ==
    0

    and

    len(empty_curation_files)
    ==
    0

    and

    len(unexpected_corpus_files)
    ==
    0

    and

    len(unexpected_curation_files)
    ==
    0

    and

    len(missing_expected_corpus_files)
    ==
    0

    and

    len(missing_expected_curation_files)
    ==
    0

    and

    len(materialized_review_ids)
    ==
    review_count

    and

    len(materialized_response_ids)
    ==
    response_count

    and

    len(materialized_article_versions)
    ==
    paired_article_version_count

    and

    len(reviews_not_materialized)
    ==
    0

    and

    len(responses_not_materialized)
    ==
    0

    and

    len(paired_articles_not_materialized)
    ==
    0

    and

    len(corpus_identity_errors)
    ==
    0

    and

    len(corpus_required_section_errors)
    ==
    0

    and

    len(corpus_h1_errors)
    ==
    0

    and

    len(curation_identity_errors)
    ==
    0

    and

    len(curation_required_section_errors)
    ==
    0

    and

    len(curation_decision_prefilled)
    ==
    0

    and

    integrity
    ==
    ["ok"]

    and

    len(foreign_key_errors)
    ==
    0
)


# ============================================================
# 29. RUN INFO
# ============================================================

now = utc_now()


run_info_updates = {

    "knowledge_unit_builder_version":
        KNOWLEDGE_UNIT_BUILDER_VERSION,

    "markdown_materializer_version":
        MARKDOWN_MATERIALIZER_VERSION,

    "knowledge_unit_materialization_status":
        (
            "PASS"
            if materialization_pass
            else "FAIL"
        ),

    "knowledge_units_materialized":
        str(materialized_uks),

    "knowledge_unit_aggregation_key":
        "base_doi",

    "knowledge_unit_curation_unit":
        "KnowledgeUnit",

    "knowledge_unit_article_versions_materialized":
        str(
            len(materialized_article_versions)
        ),

    "knowledge_unit_reviews_materialized":
        str(
            len(materialized_review_ids)
        ),

    "knowledge_unit_author_responses_materialized":
        str(
            len(materialized_response_ids)
        ),

    "corpus_markdown_files":
        str(len(corpus_files)),

    "curation_markdown_files":
        str(len(curation_files)),

    "corpus_markdown_total_bytes":
        str(corpus_total_bytes),

    "curation_markdown_total_bytes":
        str(curation_total_bytes),

    "curation_decisions_prefilled":
        str(
            len(curation_decision_prefilled)
        ),

    "knowledge_unit_materialization_completed_at":
        now,
}


for key, value in run_info_updates.items():

    conn.execute(
        """
        INSERT INTO run_info (
            key,
            value,
            updated_at
        )

        VALUES (?, ?, ?)

        ON CONFLICT(key)

        DO UPDATE SET
            value = excluded.value,
            updated_at = excluded.updated_at
        """,
        (
            key,
            value,
            now,
        ),
    )


conn.commit()


# ============================================================
# 30. FECHAR SQLITE
# ============================================================

conn.close()


# ============================================================
# 31. LOG
# ============================================================

log(
    "KnowledgeUnit materialization: "
    f"uks={materialized_uks}, "
    f"article_versions={len(materialized_article_versions)}, "
    f"reviews={len(materialized_review_ids)}, "
    f"responses={len(materialized_response_ids)}, "
    f"corpus_md={len(corpus_files)}, "
    f"curation_md={len(curation_files)}, "
    f"status="
    f"{'PASS' if materialization_pass else 'FAIL'}."
)


# ============================================================
# 32. RELATÓRIO FINAL
# ============================================================

print()
print("=" * 72)
print(
    "F1000 CORPUS BUILDER 2.0 — "
    "KNOWLEDGE UNIT MATERIALIZATION"
)
print("=" * 72)


print(
    f"BaseArticles total......: "
    f"{base_article_count}"
)

print(
    f"Structured BaseArticles.: "
    f"{structured_base_count}"
)

print(
    f"Knowledge Units.........: "
    f"{materialized_uks}"
)


print()
print("Materialized entities:")

print(
    f"  ArticleVersions.......: "
    f"{len(materialized_article_versions)}"
)

print(
    f"  Reviews...............: "
    f"{len(materialized_review_ids)}"
)

print(
    f"  AuthorResponses.......: "
    f"{len(materialized_response_ids)}"
)


print()
print("Markdown files:")

print(
    f"  corpus/*.md...........: "
    f"{len(corpus_files)}"
)

print(
    f"  curation/*.md.........: "
    f"{len(curation_files)}"
)

print(
    f"  Missing curation twin.: "
    f"{len(missing_curation_twins)}"
)

print(
    f"  Missing corpus twin...: "
    f"{len(missing_corpus_twins)}"
)

print(
    f"  Empty corpus files....: "
    f"{len(empty_corpus_files)}"
)

print(
    f"  Empty curation files..: "
    f"{len(empty_curation_files)}"
)


print()
print("Materialization coverage:")

print(
    f"  Reviews missing.......: "
    f"{len(reviews_not_materialized)}"
)

print(
    f"  Responses missing.....: "
    f"{len(responses_not_materialized)}"
)

print(
    f"  Paired articles missing: "
    f"{len(paired_articles_not_materialized)}"
)


print()
print("Document validation:")

print(
    f"  Corpus identity errors.: "
    f"{len(corpus_identity_errors)}"
)

print(
    f"  Corpus section errors..: "
    f"{len(corpus_required_section_errors)}"
)

print(
    f"  Corpus H1 errors.......: "
    f"{len(corpus_h1_errors)}"
)

print(
    f"  Curation identity errors: "
    f"{len(curation_identity_errors)}"
)

print(
    f"  Curation section errors.: "
    f"{len(curation_required_section_errors)}"
)

print(
    f"  Prefilled decisions.....: "
    f"{len(curation_decision_prefilled)}"
)


print()
print("KnowledgeUnit distributions:")


print(
    "  ArticleVersions per UK:"
)

for count, frequency in sorted(
    uk_article_version_distribution.items()
):

    print(
        f"    {count:<3} -> "
        f"{frequency} UKs"
    )


print(
    "  Reviews per UK:"
)

for count, frequency in sorted(
    uk_review_distribution.items()
):

    print(
        f"    {count:<3} -> "
        f"{frequency} UKs"
    )


print(
    "  AuthorResponses per UK:"
)

for count, frequency in sorted(
    uk_response_distribution.items()
):

    print(
        f"    {count:<3} -> "
        f"{frequency} UKs"
    )


print()
print("File sizes:")

print(
    f"  Corpus total bytes....: "
    f"{corpus_total_bytes:,}"
)

print(
    f"  Curation total bytes..: "
    f"{curation_total_bytes:,}"
)


print()
print("Duplicate full-file hashes:")

print(
    f"  Corpus................: "
    f"{duplicate_corpus_hashes}"
)

print(
    f"  Curation..............: "
    f"{duplicate_curation_hashes}"
)


print()
print("Pipeline diagnostics:")

print(
    f"  Parsed ArticleVersions: "
    f"{parsed_article_version_count}"
)

print(
    f"  Paired ArticleVersions: "
    f"{paired_article_version_count}"
)

print(
    f"  Parsed but unpaired...: "
    f"{unpaired_parsed_article_versions}"
)

print(
    "  Parsed-but-unpaired versions are not "
    "materialized unless linked by a CanonicalPair."
)


print()
print("SQLite validation:")

print(
    f"  Integrity.............: "
    f"{integrity}"
)

print(
    f"  Foreign-key errors....: "
    f"{len(foreign_key_errors)}"
)


# ============================================================
# 33. DIAGNÓSTICOS EM CASO DE FALHA
# ============================================================

if file_errors:

    print()
    print("First file errors:")

    for item in file_errors[:20]:
        print(f"  {item}")


if reviews_not_materialized:

    print()
    print(
        "First Reviews not materialized:"
    )

    for value in sorted(
        reviews_not_materialized
    )[:20]:

        print(f"  {value}")


if responses_not_materialized:

    print()
    print(
        "First AuthorResponses not materialized:"
    )

    for value in sorted(
        responses_not_materialized
    )[:20]:

        print(f"  {value}")


if paired_articles_not_materialized:

    print()
    print(
        "First paired ArticleVersions not materialized:"
    )

    for value in sorted(
        paired_articles_not_materialized
    )[:20]:

        print(f"  {value}")


if corpus_h1_errors:

    print()
    print(
        "First corpus H1 errors:"
    )

    for value in corpus_h1_errors[:20]:
        print(f"  {value}")


print()
print(
    "HTTP requests performed: 0"
)

print()
print(
    "STATUS.................: "
    f"{'PASS' if materialization_pass else 'FAIL'}"
)

print("=" * 72)
print("CÉLULA 9 CONCLUÍDA")
print("=" * 72)


MONTAGEM DAS KNOWLEDGE UNITS
BaseArticles............: 500
Structured BaseArticles.: 396
KnowledgeUnit candidates: 396
Parsed ArticleVersions..: 639
Paired ArticleVersions..: 639
CanonicalPairs..........: 1131
AuthorResponses.........: 459
Materializer version....: 2.0.1

Previous corpus MD removed..: 0
Previous curation MD removed: 0

[  1/396] UKs=1 | corpus=1 | curation=1
[ 25/396] UKs=25 | corpus=25 | curation=25
[ 50/396] UKs=50 | corpus=50 | curation=50
[ 75/396] UKs=75 | corpus=75 | curation=75
[100/396] UKs=100 | corpus=100 | curation=100
[125/396] UKs=125 | corpus=125 | curation=125
[150/396] UKs=150 | corpus=150 | curation=150
[175/396] UKs=175 | corpus=175 | curation=175
[200/396] UKs=200 | corpus=200 | curation=200
[225/396] UKs=225 | corpus=225 | curation=225
[250/396] UKs=250 | corpus=250 | curation=250
[275/396] UKs=275 | corpus=275 | curation=275
[300/396] UKs=300 | corpus=300 | curation=300
[325/396] UKs=325 | corpus=325 | curation=325
[350/396] UKs=350 | corpus=350 |

In [ ]:
# ============================================================
# CÉLULA 10.1 — MANIFEST JSON-SAFE + FINAL FREEZE
# F1000 Corpus Builder 2.0
# ============================================================
#
# CORREÇÃO DA C10
#
# A auditoria global da C10 foi concluída:
#
#   52 checks
#   52 PASS
#   0 FAIL
#
# A execução falhou somente em json.dumps() porque um
# dos valores de audit_checks era um Python set.
#
# Esta célula:
#
#   1. confirma que a auditoria anterior passou;
#   2. converte recursivamente estruturas Python para
#      tipos JSON-safe;
#   3. gera manifest.json;
#   4. registra o freeze em run_info;
#   5. revalida SQLite;
#   6. confirma que corpus/curation não mudaram.
#
# NÃO:
#
#   - faz HTTP;
#   - altera corpus/*.md;
#   - altera curation/*.md;
#   - reparseia XML;
#   - reconstrói entidades;
#   - repete aquisição.
#
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import json

from pathlib import Path


# ============================================================
# 2. CONSTANTES
# ============================================================

FINAL_AUDIT_VERSION = "2.0.1"
MANIFEST_SCHEMA_VERSION = "2.0"
FREEZE_LABEL = "F1000_CORPUS_BUILDER_2.0_FINAL"


# ============================================================
# 3. JSON-SAFE CONVERTER
# ============================================================

def to_json_safe(value):
    """
    Converte recursivamente estruturas Python para
    tipos serializáveis por JSON.

    Regras:

      dict  -> dict
      list  -> list
      tuple -> list
      set   -> lista ordenada
      Path  -> string

    Tipos escalares JSON permanecem inalterados.
    """

    if value is None:
        return None


    if isinstance(
        value,
        (
            str,
            int,
            float,
            bool,
        ),
    ):
        return value


    if isinstance(
        value,
        Path,
    ):
        return str(value)


    if isinstance(
        value,
        dict,
    ):
        return {
            str(key):
                to_json_safe(item)

            for key, item
            in value.items()
        }


    if isinstance(
        value,
        set,
    ):
        converted = [
            to_json_safe(item)
            for item in value
        ]

        return sorted(
            converted,
            key=lambda item:
                json.dumps(
                    item,
                    ensure_ascii=False,
                    sort_keys=True,
                ),
        )


    if isinstance(
        value,
        (
            list,
            tuple,
        ),
    ):
        return [
            to_json_safe(item)
            for item in value
        ]


    # Fallback conservador para tipos inesperados.
    return str(value)


# ============================================================
# 4. VERIFICAR SE O ESTADO DA C10 ESTÁ DISPONÍVEL
# ============================================================

required_variables = [

    "audit_checks",
    "failed_checks",
    "final_audit_pass",

    "base_article_count",
    "exclusion_count",

    "structured_base_count",
    "review_unavailable_count",
    "not_peer_reviewed_count",
    "no_review_evidence_count",
    "structure_error_count",

    "article_version_count",
    "review_count",
    "pair_count",
    "response_count",
    "state_count",
    "knowledge_unit_count",

    "article_source_distribution",
    "recommendation_distribution",
    "relation_distribution",
    "pair_version_distribution",

    "article_word_statistics",
    "review_word_statistics",
    "response_word_statistics",

    "article_versions_per_uk",
    "reviews_per_uk",
    "responses_per_uk",

    "multi_version_uk_count",
    "multi_version_uk_percentage",

    "corpus_files",
    "curation_files",

    "corpus_total_bytes",
    "curation_total_bytes",

    "corpus_directory_sha256",
    "curation_directory_sha256",

    "corpus_file_records",
    "curation_file_records",

    "container_rows",
    "reviewed_version_rows",

    "container_xml_missing",
    "reviewed_xml_missing",

    "container_xml_hash_errors",
    "reviewed_xml_hash_errors",

    "corpus_article_coverage_errors",
    "corpus_review_coverage_errors",
    "corpus_response_coverage_errors",

    "curation_prefilled_decisions",

    "sqlite_integrity",
    "foreign_key_errors",

    "paired_article_version_count",
    "unpaired_article_versions",

    "review_structure_distribution",

    "run_info",
]


missing_variables = [
    name

    for name in required_variables

    if name not in globals()
]


if missing_variables:

    raise RuntimeError(
        "O estado da C10 não está mais integralmente "
        "disponível nesta sessão. Variáveis ausentes: "
        + ", ".join(missing_variables)
        + ". Nesse caso, execute novamente a C10 completa "
        "com a correção JSON-safe incorporada."
    )


# ============================================================
# 5. CONFIRMAR RESULTADO DA AUDITORIA
# ============================================================

if not final_audit_pass:

    raise RuntimeError(
        "A auditoria anterior não terminou em PASS. "
        "O corpus não será congelado."
    )


if len(audit_checks) != 52:

    raise RuntimeError(
        "Cardinalidade inesperada de audit checks: "
        f"{len(audit_checks)}."
    )


if len(failed_checks) != 0:

    raise RuntimeError(
        "Existem audit checks com FAIL. "
        "O corpus não será congelado."
    )


if sqlite_integrity != ["ok"]:

    raise RuntimeError(
        "SQLite não estava íntegro ao final da auditoria."
    )


if len(foreign_key_errors) != 0:

    raise RuntimeError(
        "Foram encontrados erros de foreign key "
        "na auditoria."
    )


# ============================================================
# 6. VALIDAR NOVAMENTE OS MARKDOWN ANTES DO FREEZE
# ============================================================

current_corpus_files = sorted(
    CORPUS_DIR.glob("*.md")
)


current_curation_files = sorted(
    CURATION_DIR.glob("*.md")
)


current_corpus_records = [
    file_record(path)
    for path in current_corpus_files
]


current_curation_records = [
    file_record(path)
    for path in current_curation_files
]


current_corpus_digest = directory_digest(
    current_corpus_records
)


current_curation_digest = directory_digest(
    current_curation_records
)


if (
    current_corpus_digest
    !=
    corpus_directory_sha256
):

    raise RuntimeError(
        "O diretório corpus mudou desde a auditoria C10. "
        "Freeze cancelado."
    )


if (
    current_curation_digest
    !=
    curation_directory_sha256
):

    raise RuntimeError(
        "O diretório curation mudou desde a auditoria C10. "
        "Freeze cancelado."
    )


# ============================================================
# 7. TIMESTAMP DO FREEZE
# ============================================================

freeze_timestamp = utc_now()


# ============================================================
# 8. MANIFEST
# ============================================================

manifest = {

    "manifest_schema_version":
        MANIFEST_SCHEMA_VERSION,


    "builder": {

        "name":
            BUILDER_NAME,

        "version":
            BUILDER_VERSION,

        "final_audit_version":
            FINAL_AUDIT_VERSION,

        "knowledge_unit_builder_version":
            run_info.get(
                "knowledge_unit_builder_version"
            ),

        "markdown_materializer_version":
            run_info.get(
                "markdown_materializer_version"
            ),

        "article_parser_version":
            run_info.get(
                "article_parser_version"
            ),

        "canonical_pair_builder_version":
            run_info.get(
                "canonical_pair_builder_version"
            ),
    },


    "freeze": {

        "status":
            "FROZEN",

        "label":
            FREEZE_LABEL,

        "timestamp_utc":
            freeze_timestamp,

        "final_audit_status":
            "PASS",

        "audit_checks":
            len(audit_checks),

        "audit_failures":
            len(failed_checks),
    },


    "scope": {

        "source_platform":
            "F1000Research",

        "search_endpoint":
            SEARCH_ENDPOINT,

        "article_xml_endpoint":
            ARTICLE_XML_ENDPOINT,

        "target_eligible_base_articles":
            TARGET_BASE_ARTICLES,

        "eligible_base_articles":
            base_article_count,

        "discovery_exclusions":
            exclusion_count,

        "sampling_method":
            (
                "Sequential search-order sampling. "
                "Candidates were evaluated in observed "
                "F1000Research search order. A candidate "
                "entered the final sample only when its "
                "observed container XML was retrievable "
                "and structurally valid. Rejected "
                "candidates were recorded in "
                "discovery_exclusions and search "
                "continued until the target number of "
                "eligible unique base_doi values was "
                "reached."
            ),

        "sampling_type":
            "non-random",

        "representativeness_claim":
            (
                "No population representativeness "
                "is claimed."
            ),
    },


    "analytical_model": {

        "document_level_entities": [
            "ArticleVersion",
            "Review",
            "AuthorResponse",
            "PeerReviewState",
        ],

        "atomic_relational_unit":
            "CanonicalPair",

        "primary_analytical_unit":
            "KnowledgeUnit",

        "primary_curation_unit":
            "KnowledgeUnit",

        "aggregation_key":
            "base_doi",

        "future_split_isolation_key":
            "base_doi",

        "canonical_pair_definition":
            (
                "Exact reviewed ArticleVersion "
                "linked to one individual Review."
            ),

        "knowledge_unit_definition":
            (
                "Longitudinal aggregation by base_doi "
                "of exact reviewed article versions, "
                "their reviews, associated author "
                "responses, and the latest observed "
                "peer-review state available from the "
                "sampled container."
            ),
    },


    "semantic_rules": {

        "review_version_linkage":
            (
                "Reviews are linked to the exact "
                "reviewed DOI declared by the structured "
                "review report. Container DOI is never "
                "used as a substitute when the review "
                "targets a historical version."
            ),

        "recommendation_scope":
            "individual Review",

        "recommendation_mapping": {

            "approve":
                "Approved",

            "approve-with-reservations":
                "Approved with Reservations",

            "reject":
                "Not Approved",
        },

        "peer_review_state":
            (
                "Observed source state from the sampled "
                "container; not an inferred editorial "
                "decision."
            ),

        "peer_review_state_count_semantics":
            (
                "NULL=not_observed_or_not_determinable;"
                "0=explicit_zero"
            ),

        "author_response_date":
            (
                "Missing dates remain missing; "
                "no date is inferred."
            ),

        "reviewer_identity":
            (
                "Reviewer identities are not required "
                "in the training payload."
            ),

        "quality_scoring":
            (
                "No automated review-quality score "
                "is assigned."
            ),

        "curation":
            (
                "Human curation decisions are left "
                "unfilled in curation Markdown."
            ),
    },


    "classification": {

        "review_structure_status":
            review_structure_distribution,
    },


    "cardinalities": {

        "base_articles":
            base_article_count,

        "article_versions":
            article_version_count,

        "reviews":
            review_count,

        "canonical_pairs":
            pair_count,

        "author_responses":
            response_count,

        "peer_review_states":
            state_count,

        "knowledge_units":
            knowledge_unit_count,

        "corpus_markdown_files":
            len(current_corpus_files),

        "curation_markdown_files":
            len(current_curation_files),

        "additional_reviewed_version_xml":
            len(reviewed_version_rows),
    },


    "article_versions": {

        "source_distribution":
            article_source_distribution,

        "paired_article_versions":
            paired_article_version_count,

        "unpaired_parsed_article_versions":
            unpaired_article_versions,

        "word_count_statistics":
            article_word_statistics,
    },


    "reviews": {

        "recommendation_distribution":
            recommendation_distribution,

        "relation_to_observed_container":
            relation_distribution,

        "reviewed_version_distribution":
            pair_version_distribution,

        "word_count_statistics":
            review_word_statistics,
    },


    "author_responses": {

        "count":
            response_count,

        "word_count_statistics":
            response_word_statistics,
    },


    "knowledge_units": {

        "article_versions_per_uk":
            sorted_counter_dict(
                article_versions_per_uk
            ),

        "reviews_per_uk":
            sorted_counter_dict(
                reviews_per_uk
            ),

        "author_responses_per_uk":
            sorted_counter_dict(
                responses_per_uk
            ),

        "single_version_uk_count":
            article_versions_per_uk.get(
                1,
                0,
            ),

        "multi_version_uk_count":
            multi_version_uk_count,

        "multi_version_uk_percentage":
            multi_version_uk_percentage,
    },


    "materialization": {

        "corpus_directory":
            str(CORPUS_DIR),

        "curation_directory":
            str(CURATION_DIR),

        "corpus_files":
            len(current_corpus_files),

        "curation_files":
            len(current_curation_files),

        "corpus_total_bytes":
            corpus_total_bytes,

        "curation_total_bytes":
            curation_total_bytes,

        "corpus_directory_sha256":
            corpus_directory_sha256,

        "curation_directory_sha256":
            curation_directory_sha256,

        "corpus_files_sha256":
            current_corpus_records,

        "curation_files_sha256":
            current_curation_records,
    },


    "raw_preservation": {

        "container_xml_files":
            len(container_rows),

        "additional_reviewed_version_xml_files":
            len(reviewed_version_rows),

        "container_xml_missing":
            len(container_xml_missing),

        "reviewed_version_xml_missing":
            len(reviewed_xml_missing),

        "container_xml_hash_errors":
            len(container_xml_hash_errors),

        "reviewed_version_xml_hash_errors":
            len(reviewed_xml_hash_errors),
    },


    "validation": {

        "status":
            "PASS",

        "checks":
            audit_checks,

        "sqlite_integrity":
            sqlite_integrity,

        "foreign_key_errors":
            len(foreign_key_errors),

        "corpus_article_coverage_errors":
            len(
                corpus_article_coverage_errors
            ),

        "corpus_review_coverage_errors":
            len(
                corpus_review_coverage_errors
            ),

        "corpus_response_coverage_errors":
            len(
                corpus_response_coverage_errors
            ),

        "curation_prefilled_decisions":
            len(
                curation_prefilled_decisions
            ),
    },
}


# ============================================================
# 9. CONVERSÃO RECURSIVA PARA JSON-SAFE
# ============================================================

manifest_safe = to_json_safe(
    manifest
)


# ============================================================
# 10. TESTE DE SERIALIZAÇÃO ANTES DE QUALQUER WRITE
# ============================================================

manifest_json = json.dumps(
    manifest_safe,
    ensure_ascii=False,
    indent=2,
    sort_keys=True,
)


# ============================================================
# 11. ESCREVER MANIFEST ATOMICAMENTE
# ============================================================

manifest_temp_path = (
    MANIFEST_PATH.with_suffix(
        ".json.tmp"
    )
)


manifest_temp_path.write_text(
    manifest_json + "\n",
    encoding="utf-8",
)


manifest_temp_path.replace(
    MANIFEST_PATH
)


manifest_sha256 = sha256_file(
    MANIFEST_PATH
)


manifest_bytes = (
    MANIFEST_PATH.stat().st_size
)


# ============================================================
# 12. VALIDAR MANIFEST GRAVADO
# ============================================================

manifest_reloaded = json.loads(
    MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)


if (
    manifest_reloaded["freeze"]["status"]
    !=
    "FROZEN"
):

    raise RuntimeError(
        "Manifest reload validation failed: "
        "freeze status is not FROZEN."
    )


if (
    manifest_reloaded["validation"]["status"]
    !=
    "PASS"
):

    raise RuntimeError(
        "Manifest reload validation failed: "
        "validation status is not PASS."
    )


if (
    manifest_reloaded["freeze"]["audit_failures"]
    !=
    0
):

    raise RuntimeError(
        "Manifest reload validation failed: "
        "audit failures != 0."
    )


# ============================================================
# 13. REGISTRAR FREEZE NO SQLITE
# ============================================================

conn = open_builder_db()


freeze_run_info = {

    "final_audit_version":
        FINAL_AUDIT_VERSION,

    "final_audit_status":
        "PASS",

    "final_audit_checks":
        str(
            len(audit_checks)
        ),

    "final_audit_failures":
        "0",

    "manifest_schema_version":
        MANIFEST_SCHEMA_VERSION,

    "manifest_path":
        str(MANIFEST_PATH),

    "manifest_sha256":
        manifest_sha256,

    "manifest_bytes":
        str(manifest_bytes),

    "corpus_directory_sha256":
        corpus_directory_sha256,

    "curation_directory_sha256":
        curation_directory_sha256,

    "corpus_freeze_status":
        "FROZEN",

    "corpus_freeze_label":
        FREEZE_LABEL,

    "corpus_freeze_timestamp":
        freeze_timestamp,
}


for key, value in freeze_run_info.items():

    conn.execute(
        """
        INSERT INTO run_info (
            key,
            value,
            updated_at
        )

        VALUES (?, ?, ?)

        ON CONFLICT(key)

        DO UPDATE SET
            value = excluded.value,
            updated_at = excluded.updated_at
        """,
        (
            key,
            value,
            freeze_timestamp,
        ),
    )


conn.commit()


# ============================================================
# 14. SQLITE PÓS-FREEZE
# ============================================================

post_freeze_integrity = [
    row[0]

    for row in conn.execute(
        "PRAGMA integrity_check;"
    ).fetchall()
]


post_freeze_fk_errors = conn.execute(
    "PRAGMA foreign_key_check;"
).fetchall()


stored_freeze_status = conn.execute(
    """
    SELECT value
    FROM run_info
    WHERE key = 'corpus_freeze_status'
    """
).fetchone()


stored_manifest_hash = conn.execute(
    """
    SELECT value
    FROM run_info
    WHERE key = 'manifest_sha256'
    """
).fetchone()


conn.close()


# ============================================================
# 15. REVALIDAR MARKDOWN APÓS FREEZE
# ============================================================

post_freeze_corpus_records = [
    file_record(path)

    for path in sorted(
        CORPUS_DIR.glob("*.md")
    )
]


post_freeze_curation_records = [
    file_record(path)

    for path in sorted(
        CURATION_DIR.glob("*.md")
    )
]


post_freeze_corpus_digest = directory_digest(
    post_freeze_corpus_records
)


post_freeze_curation_digest = directory_digest(
    post_freeze_curation_records
)


markdown_unchanged_after_freeze = (

    post_freeze_corpus_digest
    ==
    corpus_directory_sha256

    and

    post_freeze_curation_digest
    ==
    curation_directory_sha256
)


# ============================================================
# 16. VALIDAÇÃO FINAL DO FREEZE
# ============================================================

freeze_validation_errors = []


if post_freeze_integrity != ["ok"]:

    freeze_validation_errors.append(
        "SQLite integrity failed after freeze."
    )


if len(post_freeze_fk_errors) != 0:

    freeze_validation_errors.append(
        "Foreign-key errors after freeze."
    )


if not markdown_unchanged_after_freeze:

    freeze_validation_errors.append(
        "Markdown changed during freeze."
    )


if (
    stored_freeze_status is None
    or
    stored_freeze_status[0] != "FROZEN"
):

    freeze_validation_errors.append(
        "Freeze status was not persisted in run_info."
    )


if (
    stored_manifest_hash is None
    or
    stored_manifest_hash[0] != manifest_sha256
):

    freeze_validation_errors.append(
        "Manifest SHA256 was not persisted correctly."
    )


if sha256_file(MANIFEST_PATH) != manifest_sha256:

    freeze_validation_errors.append(
        "Manifest SHA256 changed after write."
    )


if freeze_validation_errors:

    print()
    print("=" * 72)
    print("FINAL FREEZE VALIDATION FAILURE")
    print("=" * 72)

    for error in freeze_validation_errors:
        print(f"- {error}")

    raise RuntimeError(
        "Freeze validation failed."
    )


# ============================================================
# 17. LOG
# ============================================================

log(
    "FINAL FREEZE: "
    f"base_articles={base_article_count}, "
    f"knowledge_units={knowledge_unit_count}, "
    f"article_versions={article_version_count}, "
    f"reviews={review_count}, "
    f"pairs={pair_count}, "
    f"responses={response_count}, "
    f"corpus_md={len(current_corpus_files)}, "
    f"curation_md={len(current_curation_files)}, "
    f"manifest_sha256={manifest_sha256}, "
    f"status=PASS."
)


# ============================================================
# 18. RELATÓRIO FINAL
# ============================================================

print()
print("=" * 72)
print(
    "F1000 CORPUS BUILDER 2.0 — FINAL FREEZE"
)
print("=" * 72)


print()
print("Audit inherited from C10:")

print(
    f"  Audit checks.................: "
    f"{len(audit_checks)}"
)

print(
    f"  Passed.......................: "
    f"{len(audit_checks) - len(failed_checks)}"
)

print(
    f"  Failed.......................: "
    f"{len(failed_checks)}"
)


print()
print("Final cardinalities:")

print(
    f"  Eligible BaseArticles........: "
    f"{base_article_count}"
)

print(
    f"  KnowledgeUnits...............: "
    f"{knowledge_unit_count}"
)

print(
    f"  ArticleVersions..............: "
    f"{article_version_count}"
)

print(
    f"  Reviews......................: "
    f"{review_count}"
)

print(
    f"  CanonicalPairs...............: "
    f"{pair_count}"
)

print(
    f"  AuthorResponses..............: "
    f"{response_count}"
)

print(
    f"  PeerReviewStates.............: "
    f"{state_count}"
)


print()
print("Final materialization:")

print(
    f"  Corpus Markdown..............: "
    f"{len(current_corpus_files)}"
)

print(
    f"  Curation Markdown............: "
    f"{len(current_curation_files)}"
)

print(
    f"  Corpus directory SHA256......: "
    f"{corpus_directory_sha256}"
)

print(
    f"  Curation directory SHA256....: "
    f"{curation_directory_sha256}"
)


print()
print("Manifest:")

print(
    f"  Path.........................: "
    f"{MANIFEST_PATH}"
)

print(
    f"  Bytes........................: "
    f"{manifest_bytes:,}"
)

print(
    f"  SHA256.......................: "
    f"{manifest_sha256}"
)

print(
    f"  JSON reload..................: PASS"
)


print()
print("Final validation:")

print(
    f"  SQLite integrity.............: "
    f"{post_freeze_integrity}"
)

print(
    f"  Foreign-key errors...........: "
    f"{len(post_freeze_fk_errors)}"
)

print(
    f"  Markdown unchanged by freeze.: "
    f"{markdown_unchanged_after_freeze}"
)

print(
    f"  Freeze persisted in SQLite...: "
    f"{stored_freeze_status[0] == 'FROZEN'}"
)

print(
    f"  Manifest hash persisted......: "
    f"{stored_manifest_hash[0] == manifest_sha256}"
)


print()
print(
    "HTTP requests performed.......: 0"
)

print(
    "Corpus Markdown modified......: 0"
)

print(
    "Curation Markdown modified....: 0"
)


print()
print(
    "MANIFEST WRITTEN..............: YES"
)

print(
    "CORPUS FROZEN.................: YES"
)

print(
    f"FREEZE LABEL...................: "
    f"{FREEZE_LABEL}"
)

print(
    f"FREEZE TIMESTAMP UTC...........: "
    f"{freeze_timestamp}"
)


print()
print(
    "STATUS........................: PASS"
)

print("=" * 72)
print(
    "CÉLULA 10.1 CONCLUÍDA — "
    "BUILDER 2.0 FINALIZADO"
)
print("=" * 72)

[2026-09-17T23:21:23.767301+00:00] FINAL FREEZE: base_articles=500, knowledge_units=396, article_versions=639, reviews=1131, pairs=1131, responses=459, corpus_md=396, curation_md=396, manifest_sha256=4afa3e1b2d820f1607f3c11640d9ea3bd978a070e033bf824d13e50451c67a05, status=PASS.

F1000 CORPUS BUILDER 2.0 — FINAL FREEZE

Audit inherited from C10:
  Audit checks.................: 52
  Passed.......................: 52
  Failed.......................: 0

Final cardinalities:
  Eligible BaseArticles........: 500
  KnowledgeUnits...............: 396
  ArticleVersions..............: 639
  Reviews......................: 1131
  CanonicalPairs...............: 1131
  AuthorResponses..............: 459
  PeerReviewStates.............: 500

Final materialization:
  Corpus Markdown..............: 396
  Curation Markdown............: 396
  Corpus directory SHA256......: 68f291deb39372f126c9b7251f439973946dbabadc946dc2d1a1571b0b24d60f
  Curation directory SHA256....: 981c5e846c2598522dd574c19b3e93dcb4